In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors opencv-python controlnet-aux gradio scikit-image


In [2]:
# ============================================================
# Kaggle dependency setup
# If you run this in a Kaggle notebook, you can also run this once
# in a separate cell:
# !pip -q install -U diffusers transformers accelerate safetensors opencv-python controlnet-aux gradio scikit-image
# ============================================================

#NEW ENGLISH
# ============================================================
# Camouflage Latent Blending + Dual ControlNet + Gradio UI
# Clean version with JSON config load/save support
# Compatible with legacy ui_config_*.json (params as positional list)
# ============================================================
import os
import re
import math
import json
import time
import base64
import io
import html as html_lib
import contextlib
from dataclasses import dataclass
from datetime import datetime
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)


# ============================================================
# 0) Small utils
# ============================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

# You can paste image from clipboard 
KAGGLE_ANIMALS_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/animals" 
KAGGLE_BG_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/bg"
DEFAULT_OUT_DIR = "/kaggle/working/outputs_camouflage_softgradmask" if os.path.exists("/kaggle/working") else "./outputs_camouflage_softgradmask"

def list_image_files(path: str) -> List[str]:
    path = str(path or "").strip()
    if not path or not os.path.exists(path):
        return []
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    if os.path.isfile(path):
        return [path] if os.path.splitext(path)[1].lower() in exts else []
    files = []
    for name in sorted(os.listdir(path)):
        full = os.path.join(path, name)
        if os.path.isfile(full) and os.path.splitext(name)[1].lower() in exts:
            files.append(full)
    return files


def resolve_image_path(path_or_dir: Optional[str], fallback_dir: Optional[str] = None) -> Optional[str]:
    candidate = str(path_or_dir or "").strip()
    if candidate:
        if os.path.isfile(candidate):
            return candidate
        if os.path.isdir(candidate):
            files = list_image_files(candidate)
            if files:
                return files[0]
    fallback = str(fallback_dir or "").strip()
    if fallback:
        if os.path.isfile(fallback):
            return fallback
        if os.path.isdir(fallback):
            files = list_image_files(fallback)
            if files:
                return files[0]
    return None


def _odd(k: int) -> int:
    k = int(k)
    return k if k % 2 == 1 else k + 1


def _ensure_odd(k: int, minimum: int = 1) -> int:
    k = max(int(k), int(minimum))
    return k if k % 2 == 1 else k + 1


def _get_kernel(shape: str, k: int) -> np.ndarray:
    k = _ensure_odd(k, 1)
    shape = str(shape).lower().strip()

    if shape == "rect":
        return cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    if shape == "cross":
        return cv2.getStructuringElement(cv2.MORPH_CROSS, (k, k))
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def seed_everything(seed: int) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def slugify(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"[^a-z0-9\-_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "item"


def format_prompt(template: str, animal: str) -> str:
    return (template or "").format(animals=animal, animal=animal)


def save_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def fresh_seed() -> int:
    seed = int.from_bytes(os.urandom(8), "big") % 2147483647
    return seed if seed > 0 else 1


def normalize_0_255(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-6:
        return np.zeros_like(x, dtype=np.uint8)
    x = (x - mn) / (mx - mn)
    return (x * 255.0).clip(0, 255).astype(np.uint8)


def gray01_to_u8(x: np.ndarray) -> np.ndarray:
    return (np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)


def pil_to_data_uri(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    encoded = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/{fmt.lower()};base64,{encoded}"


def make_checkerboard(size: Tuple[int, int], tile: int = 32) -> Image.Image:
    w, h = size
    tile = max(int(tile), 4)
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    c0 = np.array([38, 38, 42], dtype=np.uint8)
    c1 = np.array([58, 58, 64], dtype=np.uint8)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            use_c0 = ((x // tile) + (y // tile)) % 2 == 0
            arr[y : y + tile, x : x + tile] = c0 if use_c0 else c1
    return Image.fromarray(arr, mode="RGB")


def load_rgb_preview_image(path: Optional[str], size: Tuple[int, int]) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved and os.path.exists(resolved):
        return Image.open(resolved).convert("RGB").resize(size, Image.LANCZOS)
    return make_checkerboard(size)


def mask_preview_to_rgba(mask_pil: Image.Image, alpha_scale: float = 0.82) -> Image.Image:
    gray = np.array(mask_pil.convert("L"), dtype=np.uint8)
    edges = cv2.Canny(gray, 32, 128)

    rgba = np.zeros((gray.shape[0], gray.shape[1], 4), dtype=np.uint8)
    rgba[..., 0] = 255
    rgba[..., 1] = 255
    rgba[..., 2] = 255
    rgba[..., 3] = np.clip(gray.astype(np.float32) * float(alpha_scale), 0, 255).astype(np.uint8)

    outline = edges > 0
    rgba[outline, 0] = 255
    rgba[outline, 1] = 96
    rgba[outline, 2] = 96
    rgba[outline, 3] = 255
    return Image.fromarray(rgba, mode="RGBA")


def build_interactive_layout_html(
    bg_img: Image.Image,
    overlay_img: Image.Image,
    width: int,
    height: int,
    init_scale: float,
    init_offset_x: float,
    init_offset_y: float,
    init_rotation_deg: float,
) -> str:
    bg_uri = pil_to_data_uri(bg_img.convert("RGB"), fmt="PNG")
    overlay_uri = pil_to_data_uri(overlay_img.convert("RGBA"), fmt="PNG")
    canvas_w = int(width)
    canvas_h = int(height)
    display_w = min(max(canvas_w, 320), 640)
    display_h = max(int(display_w * canvas_h / max(canvas_w, 1)), 220)

    iframe_id = f"camo-placement-frame-{int(time.time() * 1000)}"

    inner_html = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <style>
    html, body { margin: 0; padding: 0; background: transparent; font-family: ui-sans-serif, system-ui, sans-serif; color: #e5e7eb; overflow: hidden; }
    .wrap { border: 1px solid rgba(255,255,255,0.08); border-radius: 14px; padding: 12px; background: #111827; color: #e5e7eb; box-sizing: border-box; width: 100%; min-height: 100vh; }
    .hint { font-size: 13px; line-height: 1.5; color: #cbd5e1; margin-bottom: 10px; }
    .canvas-wrap { position: relative; width: __DISPLAY_W__px; max-width: 100%; }
    canvas { width: 100%; height: auto; display: block; border-radius: 12px; cursor: grab; background: #0f172a; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.08); }
    canvas.dragging, canvas.resizing, canvas.rotating { cursor: grabbing; }
    .toolbar { margin-top: 10px; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }
    button { border: 0; border-radius: 10px; padding: 8px 12px; background: #1f2937; color: #f9fafb; cursor: pointer; }
    button:hover { background: #374151; }
    .stats { font-size: 12px; color: #cbd5e1; }
    .badge { display: inline-flex; align-items: center; gap: 6px; padding: 6px 10px; border-radius: 999px; background: rgba(255,255,255,0.06); }
    .status { font-size: 12px; color: #93c5fd; margin-top: 8px; }
  </style>
</head>
<body>
  <div class="wrap">
    <div class="hint">Drag the box to move the subject. Drag the blue handle at the bottom-right corner to resize, drag the yellow handle above the box to rotate, and use the mouse wheel to zoom. The Layout sliders on the left panel are synchronized <b>in real time</b>.</div>
    <div class="canvas-wrap">
      <canvas id="canvas" width="__CANVAS_W__" height="__CANVAS_H__"></canvas>
    </div>
    <div class="toolbar">
      <button type="button" data-action="center">Center</button>
      <button type="button" data-action="fit">Fit 0.7</button>
      <button type="button" data-action="reset">Reset</button>
      <span class="badge stats">scale=<span data-key="scale"></span></span>
      <span class="badge stats">offsetX=<span data-key="offsetX"></span></span>
      <span class="badge stats">offsetY=<span data-key="offsetY"></span></span>
      <span class="badge stats">rotation=<span data-key="rotationDeg"></span>°</span>
    </div>
    <div class="status" data-role="status">Canvas state is ready.</div>
  </div>
  <script>
    (() => {
      const canvas = document.getElementById('canvas');
      const ctx = canvas.getContext('2d');
      const statusEl = document.querySelector('[data-role="status"]');
      const bg = new Image();
      const overlay = new Image();
      bg.src = '__BG_URI__';
      overlay.src = '__OVERLAY_URI__';

      const clamp = (v, lo, hi) => Math.min(Math.max(v, lo), hi);
      const wrapDeg = (deg) => {
        let out = Number(deg) || 0;
        while (out > 180) out -= 360;
        while (out <= -180) out += 360;
        return out;
      };
      const wrapRad = (rad) => {
        let out = Number(rad) || 0;
        while (out > Math.PI) out -= Math.PI * 2;
        while (out <= -Math.PI) out += Math.PI * 2;
        return out;
      };
      const nearlyEqual = (a, b, eps=1e-4) => Math.abs((Number(a)||0) - (Number(b)||0)) <= eps;

      const state = {
        scale: clamp(Number('__INIT_SCALE__') || 1, 0.10, 2.50),
        offsetX: clamp(Number('__INIT_OFFSET_X__') || 0, -1.00, 1.00),
        offsetY: clamp(Number('__INIT_OFFSET_Y__') || 0, -1.00, 1.00),
        rotationDeg: wrapDeg(Number('__INIT_ROTATION_DEG__') || 0),
      };

      let parentDoc = null;
      try { parentDoc = window.parent && window.parent.document ? window.parent.document : null; } catch (_) { parentDoc = null; }

      function setStatus(msg) {
        if (statusEl) statusEl.textContent = msg;
      }

      function syncOneSlider(elemId, value) {
        if (!parentDoc) return false;
        const host = parentDoc.getElementById(elemId);
        if (!host) return false;
        const inputs = host.querySelectorAll('input');
        if (!inputs.length) return false;
        const fixed = elemId === 'blend-mask-rotation-deg' ? String(Number(value).toFixed(1)) : String(Number(value).toFixed(4));
        inputs.forEach((input) => {
          if (input.value !== fixed) {
            input.value = fixed;
            input.dispatchEvent(new Event('input', { bubbles: true }));
            input.dispatchEvent(new Event('change', { bubbles: true }));
          }
        });
        return true;
      }

      function readOneSlider(elemId, fallback) {
        if (!parentDoc) return fallback;
        const host = parentDoc.getElementById(elemId);
        if (!host) return fallback;
        const input = host.querySelector('input');
        if (!input) return fallback;
        const val = Number(input.value);
        return Number.isFinite(val) ? val : fallback;
      }

      function pushStateToParent() {
        if (window.parent) {
          window.parent.camoPlacementState = { ...state };
        }
        syncOneSlider('blend-mask-scale', state.scale);
        syncOneSlider('blend-mask-offset-x', state.offsetX);
        syncOneSlider('blend-mask-offset-y', state.offsetY);
        syncOneSlider('blend-mask-rotation-deg', state.rotationDeg);
      }

      function pullStateFromParent(force=false) {
        const nextScale = clamp(readOneSlider('blend-mask-scale', state.scale), 0.10, 2.50);
        const nextOffsetX = clamp(readOneSlider('blend-mask-offset-x', state.offsetX), -1.00, 1.00);
        const nextOffsetY = clamp(readOneSlider('blend-mask-offset-y', state.offsetY), -1.00, 1.00);
        const nextRotation = wrapDeg(readOneSlider('blend-mask-rotation-deg', state.rotationDeg));
        const changed = force || !nearlyEqual(nextScale, state.scale) || !nearlyEqual(nextOffsetX, state.offsetX) || !nearlyEqual(nextOffsetY, state.offsetY) || !nearlyEqual(nextRotation, state.rotationDeg, 0.11);
        if (changed) {
          state.scale = nextScale;
          state.offsetX = nextOffsetX;
          state.offsetY = nextOffsetY;
          state.rotationDeg = nextRotation;
          syncBadges();
          draw();
        }
      }

      let mode = null;
      let dragStart = null;
      let rotateStart = null;
      const handleR = 11;
      const rotateHandleGap = 42;

      function syncBadges() {
        document.querySelector('[data-key="scale"]').textContent = state.scale.toFixed(3);
        document.querySelector('[data-key="offsetX"]').textContent = (state.offsetX).toFixed(3);
        document.querySelector('[data-key="offsetY"]').textContent = state.offsetY.toFixed(3);
        document.querySelector('[data-key="rotationDeg"]').textContent = state.rotationDeg.toFixed(1);
      }

      function getGeometry() {
        const cw = canvas.width;
        const ch = canvas.height;
        const dw = cw * state.scale;
        const dh = ch * state.scale;
        const cx = (cw / 2) + state.offsetX * cw;
        const cy = (ch / 2) + state.offsetY * ch;
        const rad = state.rotationDeg * Math.PI / 180.0;
        const cos = Math.cos(rad);
        const sin = Math.sin(rad);

        const localToWorld = (lx, ly) => ({
          x: cx + lx * cos - ly * sin,
          y: cy + lx * sin + ly * cos,
        });
        const worldToLocal = (px, py) => {
          const dx = px - cx;
          const dy = py - cy;
          return {
            x: dx * cos + dy * sin,
            y: -dx * sin + dy * cos,
          };
        };

        const halfW = dw / 2;
        const halfH = dh / 2;
        const corners = [
          localToWorld(-halfW, -halfH),
          localToWorld( halfW, -halfH),
          localToWorld( halfW,  halfH),
          localToWorld(-halfW,  halfH),
        ];
        const resizeHandle = localToWorld(halfW, halfH);
        const rotateHandle = localToWorld(0, -halfH - rotateHandleGap);

        return {
          cw, ch, dw, dh, cx, cy, rad, halfW, halfH,
          corners, resizeHandle, rotateHandle, localToWorld, worldToLocal,
        };
      }

      function drawGuides(geom) {
        ctx.save();
        ctx.strokeStyle = 'rgba(255,255,255,0.15)';
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(geom.cw / 2, 0);
        ctx.lineTo(geom.cw / 2, geom.ch);
        ctx.moveTo(0, geom.ch / 2);
        ctx.lineTo(geom.cw, geom.ch / 2);
        ctx.stroke();

        ctx.setLineDash([8, 8]);
        ctx.strokeStyle = 'rgba(255,255,255,0.70)';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(geom.corners[0].x, geom.corners[0].y);
        for (let i = 1; i < geom.corners.length; i++) ctx.lineTo(geom.corners[i].x, geom.corners[i].y);
        ctx.closePath();
        ctx.stroke();
        ctx.setLineDash([]);

        const topMid = geom.localToWorld(0, -geom.halfH);
        ctx.beginPath();
        ctx.moveTo(topMid.x, topMid.y);
        ctx.lineTo(geom.rotateHandle.x, geom.rotateHandle.y);
        ctx.strokeStyle = 'rgba(251,191,36,0.9)';
        ctx.lineWidth = 2;
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#60a5fa';
        ctx.arc(geom.resizeHandle.x, geom.resizeHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#fbbf24';
        ctx.arc(geom.rotateHandle.x, geom.rotateHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();
        ctx.restore();
      }

      function draw() {
        const geom = getGeometry();
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        if (bg.complete) ctx.drawImage(bg, 0, 0, canvas.width, canvas.height);
        if (overlay.complete) {
          ctx.save();
          ctx.translate(geom.cx, geom.cy);
          ctx.rotate(geom.rad);
          ctx.drawImage(overlay, -geom.dw / 2, -geom.dh / 2, geom.dw, geom.dh);
          ctx.restore();
        }
        drawGuides(geom);
      }

      function localPoint(evt) {
        const rect = canvas.getBoundingClientRect();
        return {
          x: (evt.clientX - rect.left) * (canvas.width / rect.width),
          y: (evt.clientY - rect.top) * (canvas.height / rect.height),
        };
      }

      function startAction(evt) {
        evt.preventDefault();
        const p = localPoint(evt);
        const geom = getGeometry();
        const distResize = Math.hypot(p.x - geom.resizeHandle.x, p.y - geom.resizeHandle.y);
        const distRotate = Math.hypot(p.x - geom.rotateHandle.x, p.y - geom.rotateHandle.y);
        const local = geom.worldToLocal(p.x, p.y);
        const inside = Math.abs(local.x) <= geom.halfW && Math.abs(local.y) <= geom.halfH;

        if (distRotate <= handleR * 1.6) {
          mode = 'rotate';
          rotateStart = {
            baseRotation: state.rotationDeg,
            baseAngle: Math.atan2(p.y - geom.cy, p.x - geom.cx),
          };
          canvas.classList.add('rotating');
        } else if (distResize <= handleR * 1.6) {
          mode = 'resize';
          canvas.classList.add('resizing');
        } else if (inside) {
          mode = 'drag';
          dragStart = {
            startX: p.x,
            startY: p.y,
            baseOffsetX: state.offsetX,
            baseOffsetY: state.offsetY,
          };
          canvas.classList.add('dragging');
        }
      }

      function afterCanvasEdit(message) {
        syncBadges();
        pushStateToParent();
        draw();
        setStatus(message || 'Layout has been synchronized with the left panel.');
      }

      function moveAction(evt) {
        if (!mode) return;
        const p = localPoint(evt);
        const geom = getGeometry();

        if (mode === 'drag') {
          const dx = p.x - dragStart.startX;
          const dy = p.y - dragStart.startY;
          state.offsetX = clamp(dragStart.baseOffsetX + dx / canvas.width, -1.00, 1.00);
          state.offsetY = clamp(dragStart.baseOffsetY + dy / canvas.height, -1.00, 1.00);
        } else if (mode === 'resize') {
          const local = geom.worldToLocal(p.x, p.y);
          const sx = Math.abs(local.x) / (canvas.width / 2);
          const sy = Math.abs(local.y) / (canvas.height / 2);
          state.scale = clamp(Math.max(sx, sy, 0.10), 0.10, 2.50);
        } else if (mode === 'rotate') {
          const angle = Math.atan2(p.y - geom.cy, p.x - geom.cx);
          const delta = wrapRad(angle - rotateStart.baseAngle);
          state.rotationDeg = wrapDeg(rotateStart.baseRotation - delta * 180.0 / Math.PI);
        }

        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }

      function endAction() {
        mode = null;
        dragStart = null;
        rotateStart = null;
        canvas.classList.remove('dragging');
        canvas.classList.remove('resizing');
        canvas.classList.remove('rotating');
      }

      canvas.addEventListener('mousedown', startAction);
      window.addEventListener('mousemove', moveAction);
      window.addEventListener('mouseup', endAction);
      canvas.addEventListener('mouseleave', endAction);
      canvas.addEventListener('wheel', (evt) => {
        evt.preventDefault();
        const factor = evt.deltaY < 0 ? 1.05 : 0.95;
        state.scale = clamp(state.scale * factor, 0.10, 2.50);
        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }, { passive: false });

      document.querySelector('[data-action="center"]').addEventListener('click', () => {
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        afterCanvasEdit('Centered and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="fit"]').addEventListener('click', () => {
        state.scale = 0.70;
        afterCanvasEdit('Set scale to 0.7 and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="reset"]').addEventListener('click', () => {
        state.scale = 1.0;
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        state.rotationDeg = 0.0;
        afterCanvasEdit('Reset and synchronized with the layout sliders.');
      });

      bg.onload = draw;
      overlay.onload = draw;
      syncBadges();
      pushStateToParent();
      draw();
      setStatus(parentDoc ? 'Canvas is ready and synchronized in real time with the left panel.' : 'Canvas is ready. The parent panel is not accessible, so only internal synchronization is available.');
      setInterval(() => {
        if (!mode) pullStateFromParent(false);
      }, 120);
    })();
  </script>
</body>
</html>
"""

    inner_html = (
        inner_html.replace("__DISPLAY_W__", str(display_w))
        .replace("__CANVAS_W__", str(canvas_w))
        .replace("__CANVAS_H__", str(canvas_h))
        .replace("__BG_URI__", bg_uri)
        .replace("__OVERLAY_URI__", overlay_uri)
        .replace("__INIT_SCALE__", f"{float(init_scale):.6f}")
        .replace("__INIT_OFFSET_X__", f"{float(init_offset_x):.6f}")
        .replace("__INIT_OFFSET_Y__", f"{float(init_offset_y):.6f}")
        .replace("__INIT_ROTATION_DEG__", f"{float(init_rotation_deg):.6f}")
    )
    srcdoc = html_lib.escape(inner_html, quote=True)
    return f"""<div style='width:100%'>
  <iframe id="{iframe_id}" srcdoc="{srcdoc}" style="width:100%; height:{display_h + 150}px; border:0; background:transparent; border-radius:14px;"></iframe>
</div>"""


# ============================================================
# 1) Core config
# ============================================================
CFG: Dict[str, Any] = dict(
    out_dir=DEFAULT_OUT_DIR,
    base_model_id="runwayml/stable-diffusion-v1-5",
    canny_cn_id="lllyasviel/sd-controlnet-canny",
    soft_cn_id="lllyasviel/control_v11p_sd15_softedge",
    hed_annotator_id="lllyasviel/Annotators",
    height=512,
    width=512,
    steps=57,
    batch=1,
    seed=10,
    guidance_scale=2.0,
    guidance_bg=1.2,
    guidance_sub=2.3,
    guess_mode=True,
    canny_low=60,
    canny_high=160,
    canny_blur_ks=5,
    use_broken_edges=True,
    broken_drop_prob=0.30,
    mask_dilate=1,
    mask_close_ks=1,
    mask_close_iter=1,
    mask_blur=1,
    mask_mode="softedge",  # silhouette / outline / composite / softedge
    mask_ring_k=1,
    mask_ring_blur=1,
    mask_gamma=0.1,
    mask_auto_invert=True,
    outer_weight=1.00,
    feature_weight=0.45,
    fill_weight=0.10,
    feature_thresh=32,
    feature_open_ks=3,
    feature_blur=5,
    fill_blur=15,
    use_soft_grad_mask=False,
    soft_grad_dilation_k=3,
    soft_grad_blur_k=3,
    soft_grad_kernel_shape="ellipse",
    soft_grad_iterations=1,
    soft_grad_use_skeleton=False,
    soft_grad_skeleton_stage="post",
    soft_grad_skeleton_thresh="otsu",
    soft_grad_post_skel_dilate_k=0,
    soft_mask_blur=11,
    soft_mask_gamma=0.5,
    soft_mask_invert=False,
    soft_mask_floor=0.05,
    soft_mask_ceiling=0.90,
    debug_mask=False,
    save_mask_images=True,
    control_scales=(0.02, 1.5),
    use_weights="A",
    start_frac_control=0.0,
    end_frac_control=0.84,
    control_gate_kind="cosine",
    blend_profile="decay",
    blend_start_frac=0.05,
    blend_end_frac=0.56,
    blend_alpha_end=0.54,
    auto_attention_compensation=False,
    attention_scale_ref=0.32,
    attention_gamma_ref=0.5,
    attention_ceiling_ref=0.90,
    attention_blur_ref=11,
    blend_mask_scale=1.0,
    blend_mask_offset_x=0.0,
    blend_mask_offset_y=0.0,
    blend_mask_rotation_deg=0.0,
    randomize_seed_each_generate=False,
    bg_image_path=KAGGLE_BG_DIR,
    bg_strength=0.52,
    sub_init_noise=0.0,
    couple_mode="bg_only",
    prompt_bg=(
        ""
    ),
    prompt_subject_template=(
        "a hidden {animal} silhouette seamlessly integrated into the background, subtle facial cues, optical illusion, camouflage"
    ),
    negative=(
        "sticker, pasted object, sharp outline, cartoon, text, watermark, logo, "
        "extra limbs, deformed, low quality, high contrast foreground object"
    ),
    seed_stride=1000,
    log_every=10,
    show_preview=True,
)


def build_cfg(raw_cfg: Dict[str, Any]) -> SimpleNamespace:
    cfg = SimpleNamespace(**raw_cfg)

    if not getattr(cfg, "device", None):
        cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    if not getattr(cfg, "dtype", None):
        cfg.dtype = "float16" if cfg.device.startswith("cuda") and torch.cuda.is_available() else "float32"

    cfg.torch_dtype = torch.float16 if cfg.dtype == "float16" else torch.float32
    ensure_dir(cfg.out_dir)

    odd_fields = [
        "canny_blur_ks",
        "mask_dilate",
        "mask_close_ks",
        "mask_blur",
        "mask_ring_k",
        "mask_ring_blur",
        "feature_open_ks",
        "feature_blur",
        "fill_blur",
        "soft_mask_blur",
        "attention_blur_ref",
        "soft_grad_dilation_k",
        "soft_grad_blur_k",
        "soft_grad_post_skel_dilate_k",
    ]
    for name in odd_fields:
        value = getattr(cfg, name, 0)
        setattr(cfg, name, _odd(value) if value and value > 1 else value)

    cfg.control_scales = tuple(getattr(cfg, "control_scales", (1.0, 1.0)))
    cfg.seed_stride = int(getattr(cfg, "seed_stride", 1000))
    cfg.log_every = int(getattr(cfg, "log_every", 10))
    cfg.show_preview = bool(getattr(cfg, "show_preview", False))
    cfg.debug_mask = bool(getattr(cfg, "debug_mask", False))
    cfg.save_mask_images = bool(getattr(cfg, "save_mask_images", True))
    cfg.use_broken_edges = bool(getattr(cfg, "use_broken_edges", False))
    cfg.broken_drop_prob = float(getattr(cfg, "broken_drop_prob", 0.2))

    cfg.mask_mode = str(getattr(cfg, "mask_mode", "softedge")).lower().strip()
    cfg.mask_gamma = float(getattr(cfg, "mask_gamma", 2.0))
    cfg.mask_auto_invert = bool(getattr(cfg, "mask_auto_invert", True))

    cfg.feature_weight = float(getattr(cfg, "feature_weight", 0.45))
    cfg.fill_weight = float(getattr(cfg, "fill_weight", 0.10))
    cfg.outer_weight = float(getattr(cfg, "outer_weight", 1.00))
    cfg.feature_thresh = int(getattr(cfg, "feature_thresh", 32))
    cfg.feature_open_ks = int(getattr(cfg, "feature_open_ks", 3))
    cfg.feature_blur = int(getattr(cfg, "feature_blur", 5))
    cfg.fill_blur = int(getattr(cfg, "fill_blur", 15))

    cfg.soft_mask_blur = int(getattr(cfg, "soft_mask_blur", 5))
    cfg.soft_mask_gamma = float(getattr(cfg, "soft_mask_gamma", 1.4))
    cfg.soft_mask_invert = bool(getattr(cfg, "soft_mask_invert", False))
    cfg.soft_mask_floor = float(getattr(cfg, "soft_mask_floor", 0.0))
    cfg.soft_mask_ceiling = float(getattr(cfg, "soft_mask_ceiling", 1.0))

    cfg.auto_attention_compensation = bool(getattr(cfg, "auto_attention_compensation", False))
    cfg.attention_scale_ref = float(getattr(cfg, "attention_scale_ref", 0.32))
    cfg.attention_gamma_ref = float(getattr(cfg, "attention_gamma_ref", cfg.soft_mask_gamma))
    cfg.attention_ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", cfg.soft_mask_ceiling))
    cfg.attention_blur_ref = int(getattr(cfg, "attention_blur_ref", cfg.soft_mask_blur))

    cfg.use_soft_grad_mask = bool(getattr(cfg, "use_soft_grad_mask", True))
    cfg.soft_grad_dilation_k = int(getattr(cfg, "soft_grad_dilation_k", 5))
    cfg.soft_grad_blur_k = int(getattr(cfg, "soft_grad_blur_k", 5))
    cfg.soft_grad_kernel_shape = str(getattr(cfg, "soft_grad_kernel_shape", "ellipse")).lower().strip()
    cfg.soft_grad_iterations = int(getattr(cfg, "soft_grad_iterations", 1))
    cfg.soft_grad_use_skeleton = bool(getattr(cfg, "soft_grad_use_skeleton", False))
    cfg.soft_grad_skeleton_stage = str(getattr(cfg, "soft_grad_skeleton_stage", "post")).lower().strip()
    cfg.soft_grad_skeleton_thresh = getattr(cfg, "soft_grad_skeleton_thresh", "otsu")
    cfg.soft_grad_post_skel_dilate_k = int(getattr(cfg, "soft_grad_post_skel_dilate_k", 0))

    cfg.start_frac_control = float(getattr(cfg, "start_frac_control", 0.0))
    cfg.end_frac_control = float(getattr(cfg, "end_frac_control", 1.0))
    cfg.control_gate_kind = str(getattr(cfg, "control_gate_kind", "cosine")).lower().strip()
    cfg.blend_profile = str(getattr(cfg, "blend_profile", "decay")).lower().strip()
    cfg.blend_start_frac = float(getattr(cfg, "blend_start_frac", 0.05))
    cfg.blend_end_frac = float(getattr(cfg, "blend_end_frac", 0.70))
    cfg.blend_alpha_end = float(getattr(cfg, "blend_alpha_end", 0.45))
    cfg.blend_mask_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
    cfg.blend_mask_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
    cfg.blend_mask_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
    cfg.blend_mask_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
    cfg.randomize_seed_each_generate = bool(getattr(cfg, "randomize_seed_each_generate", False))

    cfg.bg_image_path = getattr(cfg, "bg_image_path", None)
    cfg.bg_strength = float(getattr(cfg, "bg_strength", 0.85))
    cfg.sub_init_noise = float(getattr(cfg, "sub_init_noise", 0.0))
    cfg.couple_mode = str(getattr(cfg, "couple_mode", "none")).lower().strip()
    cfg.use_weights = str(getattr(cfg, "use_weights", "A")).upper().strip()

    cfg.guidance_scale = float(getattr(cfg, "guidance_scale", 1.0))
    cfg.guidance_bg = float(getattr(cfg, "guidance_bg", cfg.guidance_scale))
    cfg.guidance_sub = float(getattr(cfg, "guidance_sub", cfg.guidance_scale))
    cfg.guess_mode = bool(getattr(cfg, "guess_mode", True))
    return cfg


# ============================================================
# 2) JSON UI config compatibility layer
# ============================================================
UI_FIELD_NAMES: List[str] = [
    "path_animal",
    "path_bg",
    "animal_name",
    "prompt_bg",
    "prompt_sub",
    "negative_prompt",
    "steps",
    "seed",
    "randomize_seed_each_generate",
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "couple_mode",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "use_broken_edges",
    "broken_drop_prob",
    "mask_mode",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "mask_gamma",
    "mask_auto_invert",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "use_soft_grad_mask",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_kernel_shape",
    "soft_grad_iterations",
    "soft_grad_use_skeleton",
    "soft_grad_skeleton_stage",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "soft_mask_gamma",
    "soft_mask_invert",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "gate_kind",
    "blend_profile",
    "blend_start",
    "blend_end",
    "alpha_end",
    "auto_attention_compensation",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "attention_blur_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
]


UI_DEFAULTS: Dict[str, Any] = {
    "path_animal": KAGGLE_ANIMALS_DIR,
    "path_bg": KAGGLE_BG_DIR,
    "animal_name": "cat",
    "prompt_bg": CFG["prompt_bg"],
    "prompt_sub": CFG["prompt_subject_template"],
    "negative_prompt": CFG["negative"],
    "steps": 57,
    "seed": 10,
    "randomize_seed_each_generate": False,
    "bg_strength": 0.52,
    "guidance_bg": 1.2,
    "guidance_sub": 2.3,
    "sub_init_noise": 0.0,
    "couple_mode": "bg_only",
    "canny_low": 60,
    "canny_high": 160,
    "canny_blur_ks": 5,
    "use_broken_edges": True,
    "broken_drop_prob": 0.3,
    "mask_mode": "softedge",
    "mask_dilate": 1,
    "mask_close_ks": 1,
    "mask_blur": 1,
    "mask_ring_k": 1,
    "mask_ring_blur": 1,
    "mask_gamma": 0.1,
    "mask_auto_invert": True,
    "outer_weight": 1.0,
    "feature_weight": 0.45,
    "fill_weight": 0.10,
    "feature_thresh": 32,
    "feature_open_ks": 3,
    "feature_blur": 5,
    "fill_blur": 15,
    "use_soft_grad_mask": False,
    "soft_grad_dilation_k": 3,
    "soft_grad_blur_k": 3,
    "soft_grad_kernel_shape": "ellipse",
    "soft_grad_iterations": 1,
    "soft_grad_use_skeleton": False,
    "soft_grad_skeleton_stage": "post",
    "soft_grad_post_skel_dilate_k": 0,
    "soft_mask_blur": 11,
    "soft_mask_gamma": 0.5,
    "soft_mask_invert": False,
    "soft_mask_floor": 0.05,
    "soft_mask_ceiling": 0.90,
    "canny_scale": 0.02,
    "soft_scale": 1.5,
    "start_frac": 0.0,
    "end_frac": 0.84,
    "gate_kind": "cosine",
    "blend_profile": "decay",
    "blend_start": 0.05,
    "blend_end": 0.56,
    "alpha_end": 0.54,
    "auto_attention_compensation": False,
    "attention_scale_ref": 0.32,
    "attention_gamma_ref": 0.5,
    "attention_ceiling_ref": 0.90,
    "attention_blur_ref": 11,
    "blend_mask_scale": 1.0,
    "blend_mask_offset_x": 0.0,
    "blend_mask_offset_y": 0.0,
    "blend_mask_rotation_deg": 0.0,
}


LAYOUT_ALIAS_TO_UI_FIELD = {
    "layout_scale": "blend_mask_scale",
    "layout_offset_x": "blend_mask_offset_x",
    "layout_offset_y": "blend_mask_offset_y",
    "layout_rotation_deg": "blend_mask_rotation_deg",
}

NUMERIC_INT_FIELDS = {
    "steps",
    "seed",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_iterations",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "attention_blur_ref",
}

NUMERIC_FLOAT_FIELDS = {
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "broken_drop_prob",
    "mask_gamma",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "soft_mask_gamma",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "blend_start",
    "blend_end",
    "alpha_end",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
}

BOOL_FIELDS = {
    "use_broken_edges",
    "mask_auto_invert",
    "use_soft_grad_mask",
    "soft_grad_use_skeleton",
    "soft_mask_invert",
    "auto_attention_compensation",
    "randomize_seed_each_generate",
}


def coerce_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y", "on"}
    return bool(value)


def coerce_ui_value(field: str, value: Any) -> Any:
    if field in BOOL_FIELDS:
        return coerce_bool(value)
    if field in NUMERIC_INT_FIELDS:
        return int(value)
    if field in NUMERIC_FLOAT_FIELDS:
        return float(value)
    return value


def normalize_ui_params(raw_params: Any) -> Dict[str, Any]:
    if isinstance(raw_params, list):
        params = dict(zip(UI_FIELD_NAMES, raw_params))
    elif isinstance(raw_params, dict):
        params = dict(raw_params)
        for alias_name, ui_name in LAYOUT_ALIAS_TO_UI_FIELD.items():
            if alias_name in params and ui_name not in params:
                params[ui_name] = params[alias_name]
    else:
        raise ValueError("Unsupported config format: 'params' must be a list or dict.")

    merged = dict(UI_DEFAULTS)
    merged.update(params)
    return {key: coerce_ui_value(key, merged[key]) for key in UI_FIELD_NAMES}


def load_ui_params_from_json(config_path: str) -> Dict[str, Any]:
    with open(config_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict) and "params" in payload:
        return normalize_ui_params(payload["params"])

    if isinstance(payload, dict):
        return normalize_ui_params(payload)

    raise ValueError("JSON config must be a dict or contain a 'params' field.")


def build_ui_dict_from_args(args: Sequence[Any]) -> Dict[str, Any]:
    if len(args) != len(UI_FIELD_NAMES):
        raise ValueError(f"Expected {len(UI_FIELD_NAMES)} UI values, got {len(args)}")
    raw = dict(zip(UI_FIELD_NAMES, args))
    return {key: coerce_ui_value(key, value) for key, value in raw.items()}


# ============================================================
# 3) Job definition
# ============================================================
@dataclass
class AnimalJob:
    image_path: str
    animals: str
    name: Optional[str] = None
    seed: Optional[int] = None
    subject_template: Optional[str] = None
    prompt_bg: Optional[str] = None
    negative: Optional[str] = None
    bg_image_path: Optional[str] = None
    bg_strength: Optional[float] = None


# ============================================================
# 4) Skeleton + kernel-gradient hint
# ============================================================
def skeletonize_u8(img_u8: np.ndarray, thresh: str | int = "otsu") -> np.ndarray:
    if img_u8.dtype != np.uint8:
        img_u8 = np.clip(img_u8, 0, 255).astype(np.uint8)

    if thresh == "otsu":
        _, bin_u8 = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, bin_u8 = cv2.threshold(img_u8, int(thresh), 255, cv2.THRESH_BINARY)

    mask = bin_u8 > 0

    try:
        from skimage.morphology import skeletonize

        return skeletonize(mask).astype(np.uint8) * 255
    except Exception:
        try:
            return cv2.ximgproc.thinning(mask.astype(np.uint8) * 255).astype(np.uint8)
        except Exception as e:
            raise RuntimeError(
                "Skeletonization failed: scikit-image and/or cv2.ximgproc is missing. "
                "Install: pip install scikit-image or opencv-contrib-python."
            ) from e


def kernel_gradient_hint(
    edges_2d: np.ndarray,
    dilation_k: int = 3,
    blur_k: int = 3,
    kernel_shape: str = "ellipse",
    iterations: int = 1,
    use_skeleton: bool = True,
    skeleton_stage: str = "pre",
    skeleton_thresh: str | int = "otsu",
    post_skel_dilate_k: int = 0,
) -> Image.Image:
    if edges_2d.dtype != np.uint8:
        edges_2d = np.clip(edges_2d, 0, 255).astype(np.uint8)

    x = edges_2d
    if use_skeleton and skeleton_stage.lower() == "pre":
        x = skeletonize_u8(x, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            x = cv2.dilate(x, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad = cv2.morphologyEx(
        x,
        cv2.MORPH_GRADIENT,
        _get_kernel(kernel_shape, dilation_k),
        iterations=iterations,
    )

    if use_skeleton and skeleton_stage.lower() == "post":
        grad = skeletonize_u8(grad, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            grad = cv2.dilate(grad, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad_blur = cv2.GaussianBlur(grad, (_ensure_odd(blur_k, 3), _ensure_odd(blur_k, 3)), 0)
    return Image.fromarray(grad_blur).convert("RGB")


# ============================================================
# 5) Image + edge + mask helpers
# ============================================================
def load_image(path: str, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved is None:
        raise FileNotFoundError(f"No valid image file or image folder found: {path}")
    img = Image.open(resolved).convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    return img


def pil_to_cv_bgr(img_pil: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def gray_to_pil(gray: np.ndarray, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    img = Image.fromarray(normalize_0_255(gray))
    if size is not None:
        img = img.resize(size, Image.NEAREST)
    return img


def rgb_uint8_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8))


def canny_clean_norm(img_bgr: np.ndarray, low: int = 80, high: int = 160, blur_ks: int = 3) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if blur_ks and blur_ks > 1:
        gray = cv2.GaussianBlur(gray, (blur_ks, blur_ks), 0)
    edges = cv2.Canny(gray, low, high)
    hint = np.stack([edges, edges, edges], axis=-1)
    return normalize_0_255(hint)


def canny_broken(edges_3ch: np.ndarray, drop_prob: float = 0.5, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    edges = edges_3ch[..., 0] > 0
    keep = rng.random(edges.shape) > drop_prob
    out = (edges & keep).astype(np.uint8) * 255
    return np.stack([out, out, out], axis=-1)


def edges_to_silhouette_mask(
    edges_3ch: np.ndarray,
    dilate: int = 17,
    close_ks: int = 17,
    close_iter: int = 1,
    blur: int = 41,
) -> np.ndarray:
    edges = (edges_3ch[..., 0] > 0).astype(np.uint8) * 255

    if dilate and dilate > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(dilate), _odd(dilate)))
        edges = cv2.dilate(edges, kernel, iterations=1)

    if close_ks and close_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(close_ks), _odd(close_ks)))
        edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=max(1, int(close_iter)))

    inv = cv2.bitwise_not(edges)
    flood = inv.copy()
    h, w = edges.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)

    fg = (flood > 0).astype(np.uint8) * 255
    if blur and blur > 1:
        fg = cv2.GaussianBlur(fg, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(fg).astype(np.float32) / 255.0


def extract_internal_feature_mask(
    edges_3ch: np.ndarray,
    sil_hw: np.ndarray,
    thresh: int = 32,
    open_ks: int = 3,
    blur: int = 5,
) -> np.ndarray:
    edges = edges_3ch[..., 0].astype(np.uint8)
    sil_bin = (sil_hw > 0.35).astype(np.uint8) * 255

    if sil_bin.max() == 0:
        return np.zeros_like(sil_hw, dtype=np.float32)

    feat = np.where(sil_bin > 0, edges, 0).astype(np.uint8)
    inner = sil_bin.copy()

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        inner = cv2.erode(inner, kernel, iterations=2)

    feat = np.where(inner > 0, feat, 0).astype(np.uint8)
    feat = (feat > thresh).astype(np.uint8) * 255

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        feat = cv2.morphologyEx(feat, cv2.MORPH_OPEN, kernel, iterations=1)

    if blur and blur > 1:
        feat = cv2.GaussianBlur(feat, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(feat).astype(np.float32) / 255.0


def build_composite_mask_hw(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    outer_ring_k: int = 11,
    outer_ring_blur: int = 7,
    outer_weight: float = 1.0,
    feature_weight: float = 0.45,
    fill_weight: float = 0.10,
    fill_blur: int = 15,
) -> np.ndarray:
    sil_u8 = gray01_to_u8(sil_hw)

    if outer_ring_k and outer_ring_k > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(outer_ring_k), _odd(outer_ring_k)))
        dil = cv2.dilate(sil_u8, kernel, iterations=1)
        ero = cv2.erode(sil_u8, kernel, iterations=1)
        ring = cv2.subtract(dil, ero)
    else:
        ring = sil_u8.copy()

    if outer_ring_blur and outer_ring_blur > 1:
        ring = cv2.GaussianBlur(ring, (_odd(outer_ring_blur), _odd(outer_ring_blur)), 0)

    ring = normalize_0_255(ring).astype(np.float32) / 255.0

    fill = sil_u8.copy()
    if fill_blur and fill_blur > 1:
        fill = cv2.GaussianBlur(fill, (_odd(fill_blur), _odd(fill_blur)), 0)
    fill = normalize_0_255(fill).astype(np.float32) / 255.0

    feat = np.clip(feature_hw.astype(np.float32), 0.0, 1.0)
    out = outer_weight * ring + feature_weight * feat + fill_weight * fill
    return np.clip(out, 0.0, 1.0)


def softedge_to_mask_hw(
    soft_hint: np.ndarray,
    blur_ks: int = 0,
    gamma: float = 1.0,
    invert: bool = False,
    floor: float = 0.0,
    ceiling: float = 1.0,
) -> np.ndarray:
    gray = (
        cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
        if soft_hint.ndim == 3
        else soft_hint.astype(np.uint8)
    )
    mask = gray.astype(np.float32) / 255.0

    if blur_ks and blur_ks > 1:
        mask = cv2.GaussianBlur(mask, (_odd(blur_ks), _odd(blur_ks)), 0)

    if invert:
        mask = 1.0 - mask

    mask = np.clip(mask, 0.0, 1.0)
    if gamma != 1.0:
        mask = mask ** gamma

    floor = float(np.clip(floor, 0.0, 1.0))
    ceiling = float(np.clip(ceiling, 0.0, 1.0))
    if ceiling > floor:
        mask = np.clip((mask - floor) / (ceiling - floor), 0.0, 1.0)

    return np.clip(mask, 0.0, 1.0).astype(np.float32)


# ============================================================
# 6) Torch helpers
# ============================================================
def to_torch_image_hint(hint_uint8: np.ndarray, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = torch.from_numpy(hint_uint8).to(device=device).float() / 255.0
    return x.permute(2, 0, 1).unsqueeze(0).contiguous().to(dtype=dtype)


def make_latent_mask(mask_hw: np.ndarray, latent_h: int, latent_w: int, device: str, dtype: torch.dtype) -> torch.Tensor:
    mask = torch.from_numpy(mask_hw).float().unsqueeze(0).unsqueeze(0)
    mask = F.interpolate(mask, size=(latent_h, latent_w), mode="bilinear", align_corners=False)
    return mask.to(device=device, dtype=dtype)


def _dilate_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.max_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def _erode_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return -F.max_pool2d(-x, kernel_size=k, stride=1, padding=k // 2)


def _blur_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.avg_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def save_mask_img(mask_1x1hw: torch.Tensor, out_path: str, width: int, height: int) -> None:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(arr).resize((width, height), Image.NEAREST).save(out_path)


def mask_tensor_to_pil(mask_1x1hw: torch.Tensor, width: int, height: int) -> Image.Image:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr).resize((width, height), Image.NEAREST)


def has_layout_transform(cfg: SimpleNamespace) -> bool:
    return (
        abs(float(getattr(cfg, "blend_mask_scale", 1.0)) - 1.0) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_x", 0.0))) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_y", 0.0))) > 1e-6
    )


def transform_frame(
    arr: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
    border_value: float | Tuple[float, ...] = 0.0,
) -> np.ndarray:
    src = np.asarray(arr)
    h, w = src.shape[:2]
    scale = max(float(scale), 1e-3)
    # UI/canvas convention: +offset_x => move right, +offset_y => move down.
    # Because cv2.warpAffine samples via an inverse map, the translation terms here
    # need the opposite sign so the rendered output moves in the same direction as
    # the interactive canvas preview.
    tx = -float(offset_x_frac) * float(w)
    ty = -float(offset_y_frac) * float(h)
    theta = math.radians(float(rotation_deg))
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    cx = (float(w) - 1.0) * 0.5
    cy = (float(h) - 1.0) * 0.5

    t_neg = np.array([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [0.0, 0.0, 1.0]], dtype=np.float32)
    s_mat = np.array([[scale, 0.0, 0.0], [0.0, scale, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    r_mat = np.array([[cos_t, -sin_t, 0.0], [sin_t, cos_t, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    t_pos = np.array([[1.0, 0.0, cx + tx], [0.0, 1.0, cy + ty], [0.0, 0.0, 1.0]], dtype=np.float32)

    forward = t_pos @ r_mat @ s_mat @ t_neg
    m = np.linalg.inv(forward)[:2].astype(np.float32)

    out = cv2.warpAffine(
        src,
        m,
        (w, h),
        flags=interp,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )
    return out


def transform_mask_hw(

    mask_hw: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(mask_hw, dtype=np.float32),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0.0,
    )
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def transform_hint_uint8(
    hint: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(hint, dtype=np.uint8),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0,
    )
    return np.clip(out, 0, 255).astype(np.uint8)


def compute_layout_scale_eff(mask_hw: np.ndarray, threshold: float = 0.35) -> float:
    mask = np.asarray(mask_hw, dtype=np.float32)
    if mask.ndim != 2 or mask.size == 0:
        return 0.0

    total = float(mask.shape[0] * mask.shape[1])
    area = float((mask > float(threshold)).sum())
    if area <= 0.0:
        area = float(np.clip(mask, 0.0, 1.0).sum())
    if total <= 0.0 or area <= 0.0:
        return 0.0
    return float(math.sqrt(area / total))


def resolve_attention_compensation(cfg: SimpleNamespace, sil_hw: np.ndarray) -> Dict[str, Any]:
    scale_eff = compute_layout_scale_eff(sil_hw, threshold=0.35)
    scale_ref = max(float(getattr(cfg, "attention_scale_ref", 0.32)), 1e-4)
    ratio = max(scale_eff / scale_ref, 1e-4) if scale_eff > 0.0 else 1.0

    gamma_ref = float(getattr(cfg, "attention_gamma_ref", getattr(cfg, "soft_mask_gamma", 1.0)))
    ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", getattr(cfg, "soft_mask_ceiling", 1.0)))
    blur_ref = int(getattr(cfg, "attention_blur_ref", getattr(cfg, "soft_mask_blur", 5)))

    effective_gamma = float(getattr(cfg, "soft_mask_gamma", gamma_ref))
    effective_ceiling = float(getattr(cfg, "soft_mask_ceiling", ceiling_ref))
    effective_blur = int(getattr(cfg, "soft_mask_blur", blur_ref))

    enabled = bool(getattr(cfg, "auto_attention_compensation", False))
    applies = enabled and str(getattr(cfg, "mask_mode", "softedge")).lower() == "softedge"

    if applies:
        effective_gamma = float(np.clip(gamma_ref * (ratio ** 0.70), 0.25, 0.95))
        effective_ceiling = float(np.clip(ceiling_ref * (ratio ** 0.25), 0.70, 0.97))
        blur_val = int(round(blur_ref * (ratio ** 0.50)))
        effective_blur = int(np.clip(_ensure_odd(blur_val, minimum=1), 1, 31))

    return {
        "enabled": enabled,
        "applies": applies,
        "scale_eff": float(scale_eff),
        "scale_ref": float(scale_ref),
        "ratio": float(ratio),
        "gamma_ref": float(gamma_ref),
        "ceiling_ref": float(ceiling_ref),
        "blur_ref": int(blur_ref),
        "effective_gamma": float(effective_gamma),
        "effective_ceiling": float(effective_ceiling),
        "effective_blur": int(effective_blur),
    }


def format_attention_status(info: Dict[str, Any], mask_mode: str) -> str:
    base = (
        f"scale_eff={float(info.get('scale_eff', 0.0)):.3f} | "
        f"scale_ref={float(info.get('scale_ref', 0.0)):.3f} | "
        f"ratio={float(info.get('ratio', 1.0)):.3f}"
    )
    if not bool(info.get("enabled", False)):
        return base + " | Auto attention compensation: OFF"
    if not bool(info.get("applies", False)):
        return base + f" | Auto attention compensation: ON, but mask_mode='{mask_mode}' so it is not applied to the soft mask."
    return (
        base
        + " | effective soft mask: "
        + f"gamma={float(info.get('effective_gamma', 0.0)):.3f}, "
        + f"ceiling={float(info.get('effective_ceiling', 0.0)):.3f}, "
        + f"blur={int(info.get('effective_blur', 0))}"
    )


def build_mask_latent(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    latent_h: int,
    latent_w: int,
    cfg: SimpleNamespace,
    job_out: str,
    soft_hint: Optional[np.ndarray] = None,
    soft_for_mask: Optional[np.ndarray] = None,
    soft_mask_hw: Optional[np.ndarray] = None,
) -> torch.Tensor:
    sil_mask = make_latent_mask(sil_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)

    if cfg.mask_auto_invert and float(sil_mask.mean()) > 0.60:
        sil_mask = 1.0 - sil_mask

    if cfg.save_mask_images:
        save_mask_img(sil_mask, os.path.join(job_out, "mask_silhouette.png"), cfg.width, cfg.height)

    mode = str(cfg.mask_mode).lower()

    if mode == "silhouette":
        mask = sil_mask
    elif mode == "outline":
        ring = (_dilate_t(sil_mask, int(cfg.mask_ring_k)) - _erode_t(sil_mask, int(cfg.mask_ring_k))).clamp(0, 1)
        if int(cfg.mask_ring_blur) > 1:
            ring = _blur_t(ring, int(cfg.mask_ring_blur)).clamp(0, 1)
        mask = ring
    elif mode == "composite":
        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )
        mask = make_latent_mask(comp_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    elif mode == "softedge":
        if soft_mask_hw is None:
            src_for_mask = soft_for_mask if soft_for_mask is not None else soft_hint
            if src_for_mask is None:
                raise ValueError("mask_mode='softedge' but soft source is None")
            soft_mask_hw = softedge_to_mask_hw(
                soft_hint=src_for_mask,
                blur_ks=int(cfg.soft_mask_blur),
                gamma=float(cfg.soft_mask_gamma),
                invert=bool(cfg.soft_mask_invert),
                floor=float(cfg.soft_mask_floor),
                ceiling=float(cfg.soft_mask_ceiling),
            )
        mask = make_latent_mask(soft_mask_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    else:
        raise ValueError(f"Unknown mask_mode: {cfg.mask_mode}")

    if mode != "softedge":
        gamma = float(cfg.mask_gamma)
        if gamma != 1.0:
            mask = (mask.clamp(0, 1) ** gamma).clamp(0, 1)

    mask = mask.clamp(0, 1)

    if cfg.save_mask_images:
        save_mask_img(mask, os.path.join(job_out, "mask_latent.png"), cfg.width, cfg.height)
    return mask


def encode_prompt(pipe, prompt: str, negative_prompt: str, batch_size: int, do_cfg: bool, device: str) -> torch.Tensor:
    cond, uncond = pipe.encode_prompt(
        prompt=[prompt] * batch_size,
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=do_cfg,
        negative_prompt=[negative_prompt] * batch_size,
    )
    return torch.cat([uncond, cond], dim=0) if do_cfg else cond


def prepare_noise_latents(
    pipe,
    batch_size: int,
    height: int,
    width: int,
    generator: torch.Generator,
    device: str,
    dtype: torch.dtype,
) -> torch.Tensor:
    latent_h, latent_w = height // 8, width // 8
    shape = (batch_size, pipe.unet.config.in_channels, latent_h, latent_w)
    latents = torch.randn(shape, generator=generator, device=device, dtype=dtype)
    return latents * pipe.scheduler.init_noise_sigma


def pil_to_vae_tensor(img_pil: Image.Image, device: str, dtype: torch.dtype) -> torch.Tensor:
    arr = np.array(img_pil).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = x * 2.0 - 1.0
    return x.to(device=device, dtype=dtype)


@torch.no_grad()
def encode_image_latents(pipe, img_pil: Image.Image, generator: torch.Generator, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = pil_to_vae_tensor(img_pil, device=device, dtype=dtype)
    lat = pipe.vae.encode(x).latent_dist.sample(generator=generator)
    return lat * pipe.vae.config.scaling_factor


@torch.no_grad()
def decode_latents(pipe, latents: torch.Tensor) -> List[Image.Image]:
    latents = (1.0 / pipe.vae.config.scaling_factor) * latents
    img = pipe.vae.decode(latents).sample
    img = (img / 2 + 0.5).clamp(0, 1)
    img = img.cpu().permute(0, 2, 3, 1).numpy()
    imgs = (img * 255).round().astype(np.uint8)
    return [Image.fromarray(im) for im in imgs]


# ============================================================
# 7) Control weights + schedules
# ============================================================
def preset_weights_A(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.array([0, 0, 0, 0, 0.1, 0.1, 0.2, 0.2, 0.3, 0.6, 0.6, 0.6], dtype=np.float32)
    mid = 0.9
    if len(weights) != n_down:
        weights = np.resize(weights, n_down)
    return weights, mid


def preset_weights_B(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.linspace(0.25, 0.65, n_down).astype(np.float32)
    if n_down >= 12:
        for i in range(4, 9):
            weights[i] = min(0.70, weights[i] + 0.08)
    return weights, 0.55


def control_gate(step_idx: int, num_steps: int, start_frac: float, end_frac: float, kind: str = "cosine") -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    if step_idx < s0 or step_idx >= s1:
        return 0.0
    active = max(1, s1 - s0)
    x = (step_idx - s0) / active
    if kind == "linear":
        return float(1.0 - x)
    return float(math.cos(0.5 * math.pi * x))


def blend_alpha(
    step_idx: int,
    num_steps: int,
    start_frac: float,
    end_frac: float,
    alpha_end: float,
    profile: str = "decay",
) -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    s0 = max(0, min(s0, num_steps - 1))
    s1 = max(0, min(s1, num_steps))

    if profile == "ramp":
        if step_idx <= s0:
            return 0.0
        if step_idx >= s1:
            return float(alpha_end)
        return float(((step_idx - s0) / max(1, s1 - s0)) * alpha_end)

    if step_idx <= s0:
        return 1.0
    if step_idx >= s1:
        return float(alpha_end)
    return float(1.0 - ((step_idx - s0) / max(1, s1 - s0)) * (1.0 - alpha_end))


@torch.no_grad()
def unet_with_optional_control(
    pipe,
    scheduler,
    latents,
    t,
    encoder_hidden_states,
    do_cfg,
    guidance_scale,
    controlnets=None,
    control_images=None,
    control_scales=None,
    layer_weights=None,
    mid_weight=None,
    step_gate=1.0,
    guess_mode=False,
    device="cuda",
    dtype=torch.float16,
) -> torch.Tensor:
    unet = pipe.unet
    latents_in = torch.cat([latents] * 2, dim=0) if do_cfg else latents
    latents_in = scheduler.scale_model_input(latents_in, t)
    down_res, mid_res = None, None

    if controlnets is not None and control_images is not None:
        if control_scales is None:
            control_scales = [1.0] * len(controlnets)

        control_cond_mask = None
        if do_cfg and guess_mode:
            control_cond_mask = torch.zeros((latents_in.shape[0], 1, 1, 1), device=device, dtype=dtype)
            control_cond_mask[latents.shape[0]:] = 1.0

        sum_down, sum_mid = None, None
        for cn, img, scale in zip(controlnets, control_images, control_scales):
            img_in = torch.cat([img] * 2, dim=0) if do_cfg else img
            cn_out = cn(
                latents_in,
                t,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=img_in,
                return_dict=True,
            )

            down = list(cn_out.down_block_res_samples)
            mid = cn_out.mid_block_res_sample

            if layer_weights is not None:
                for i in range(min(len(down), len(layer_weights))):
                    down[i] = down[i] * float(layer_weights[i])

            if mid_weight is not None:
                mid = mid * float(mid_weight)

            if step_gate != 1.0:
                down = [d * float(step_gate) for d in down]
                mid = mid * float(step_gate)

            if control_cond_mask is not None:
                down = [d * control_cond_mask for d in down]
                mid = mid * control_cond_mask

            if scale != 1.0:
                down = [d * float(scale) for d in down]
                mid = mid * float(scale)

            if sum_down is None:
                sum_down, sum_mid = down, mid
            else:
                for i in range(len(sum_down)):
                    sum_down[i] = sum_down[i] + down[i]
                sum_mid = sum_mid + mid

        down_res, mid_res = sum_down, sum_mid

    noise_pred = unet(
        latents_in,
        t,
        encoder_hidden_states=encoder_hidden_states,
        down_block_additional_residuals=down_res,
        mid_block_additional_residual=mid_res,
        return_dict=False,
    )[0]

    if do_cfg:
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + float(guidance_scale) * (noise_cond - noise_uncond)

    return noise_pred


# ============================================================
# 8) Runner
# ============================================================
class CamoRunner:
    def __init__(self, cfg: SimpleNamespace):
        self.cfg = cfg
        self.pipe = None
        self.control_nets = None
        self.hed = None

    def build_pipe(self):
        cfg = self.cfg
        seed_everything(int(cfg.seed))

        cn_canny = ControlNetModel.from_pretrained(cfg.canny_cn_id, torch_dtype=cfg.torch_dtype)
        cn_soft = ControlNetModel.from_pretrained(cfg.soft_cn_id, torch_dtype=cfg.torch_dtype)

        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            cfg.base_model_id,
            controlnet=[cn_canny, cn_soft],
            torch_dtype=cfg.torch_dtype,
            safety_checker=None,
        )
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to(cfg.device)
        pipe.set_progress_bar_config(disable=True)

        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
        pipe.enable_attention_slicing()

        self.pipe = pipe
        self.control_nets = pipe.controlnet.nets if hasattr(pipe.controlnet, "nets") else [pipe.controlnet]
        return self

    def _get_hed(self):
        if self.hed is not None:
            return self.hed
        try:
            from controlnet_aux import HEDdetector

            self.hed = HEDdetector.from_pretrained(self.cfg.hed_annotator_id)
            return self.hed
        except Exception:
            return None

    def preprocess(
        self,
        image_path: str,
        seed: int,
        layout_override: Optional[Tuple[float, float, float]] = None,
    ) -> Dict[str, Any]:
        cfg = self.cfg
        src = load_image(image_path, size=(cfg.width, cfg.height))
        src_cv = pil_to_cv_bgr(src)

        if layout_override is None:
            layout_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
            layout_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
            layout_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
            layout_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
        else:
            values = [float(v) for v in layout_override]
            if len(values) >= 4:
                layout_scale, layout_offset_x, layout_offset_y, layout_rotation_deg = values[:4]
            else:
                layout_scale, layout_offset_x, layout_offset_y = values[:3]
                layout_rotation_deg = 0.0

        canny_hint = canny_clean_norm(
            src_cv,
            low=cfg.canny_low,
            high=cfg.canny_high,
            blur_ks=cfg.canny_blur_ks,
        )

        soft_hint = None
        hed = self._get_hed()
        if hed is not None:
            try:
                hed_map = hed(src)
                hed_np = normalize_0_255(np.array(hed_map.convert("L")))
                soft_hint = np.stack([hed_np, hed_np, hed_np], axis=-1)
            except Exception:
                pass

        if soft_hint is None:
            soft_hint = canny_hint.copy()

        if layout_scale != 1.0 or layout_offset_x != 0.0 or layout_offset_y != 0.0 or layout_rotation_deg != 0.0:
            canny_hint = transform_hint_uint8(
                canny_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_NEAREST,
            )
            soft_hint = transform_hint_uint8(
                soft_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_LINEAR,
            )

        if cfg.use_broken_edges:
            canny_hint = canny_broken(canny_hint, drop_prob=cfg.broken_drop_prob, seed=seed)

        sil_hw = edges_to_silhouette_mask(
            canny_hint,
            dilate=int(cfg.mask_dilate),
            close_ks=int(cfg.mask_close_ks),
            close_iter=int(getattr(cfg, "mask_close_iter", 1)),
            blur=int(cfg.mask_blur),
        )

        feature_hw = extract_internal_feature_mask(
            canny_hint,
            sil_hw,
            thresh=int(cfg.feature_thresh),
            open_ks=int(cfg.feature_open_ks),
            blur=int(cfg.feature_blur),
        )

        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )

        soft_for_mask = soft_hint.copy()
        if bool(cfg.use_soft_grad_mask):
            soft_gray = cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
            soft_grad_pil = kernel_gradient_hint(
                edges_2d=soft_gray,
                dilation_k=int(cfg.soft_grad_dilation_k),
                blur_k=int(cfg.soft_grad_blur_k),
                kernel_shape=str(cfg.soft_grad_kernel_shape),
                iterations=int(cfg.soft_grad_iterations),
                use_skeleton=bool(cfg.soft_grad_use_skeleton),
                skeleton_stage=str(cfg.soft_grad_skeleton_stage),
                skeleton_thresh=cfg.soft_grad_skeleton_thresh,
                post_skel_dilate_k=int(cfg.soft_grad_post_skel_dilate_k),
            )
            soft_for_mask = np.array(soft_grad_pil.convert("RGB"))

        attention_info = resolve_attention_compensation(cfg, sil_hw)
        soft_mask_hw = softedge_to_mask_hw(
            soft_hint=soft_for_mask,
            blur_ks=int(attention_info["effective_blur"]),
            gamma=float(attention_info["effective_gamma"]),
            invert=bool(cfg.soft_mask_invert),
            floor=float(cfg.soft_mask_floor),
            ceiling=float(attention_info["effective_ceiling"]),
        )

        return {
            "src": src,
            "canny_hint": canny_hint,
            "soft_hint": soft_hint,
            "sil_hw": sil_hw,
            "feature_hw": feature_hw,
            "comp_hw": comp_hw,
            "soft_for_mask": soft_for_mask,
            "soft_mask_hw": soft_mask_hw,
            "attention_info": attention_info,
        }

    def _job_out_dir(self, job: AnimalJob) -> str:
        base = os.path.splitext(os.path.basename(job.image_path))[0]
        job_name = job.name or f"{slugify(job.animals)}_{slugify(base)}"
        out_dir = os.path.join(self.cfg.out_dir, job_name)
        ensure_dir(out_dir)
        return out_dir

    def _save_run_config(self, job_out: str, job: AnimalJob, extra: dict) -> None:
        cfg_dict = {k: v for k, v in self.cfg.__dict__.items() if k != "torch_dtype"}
        cfg_dict["job"] = {
            "animals": job.animals,
            "image_path": job.image_path,
            "name": job.name,
            "seed": job.seed,
            **extra,
        }
        save_json(os.path.join(job_out, "run_config.json"), cfg_dict)

    def _probe_weights(self, lat_sub, t0, emb_sub, img_canny, do_cfg) -> Tuple[np.ndarray, float]:
        lat_in = torch.cat([lat_sub] * 2, 0) if do_cfg else lat_sub
        img_in = torch.cat([img_canny] * 2, 0) if do_cfg else img_canny
        probe = self.control_nets[0](
            self.pipe.scheduler.scale_model_input(lat_in, t0),
            t0,
            encoder_hidden_states=emb_sub,
            controlnet_cond=img_in,
            return_dict=True,
        )
        n_down = len(probe.down_block_res_samples)
        return preset_weights_B(n_down) if self.cfg.use_weights == "B" else preset_weights_A(n_down)

    def _init_latents_and_timesteps(self, job_out: str, job: AnimalJob, generator: torch.Generator):
        cfg, pipe = self.cfg, self.pipe
        pipe.scheduler.set_timesteps(int(cfg.steps), device=cfg.device)
        timesteps = pipe.scheduler.timesteps

        scheduler_bg = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_sub = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_bg.set_timesteps(int(cfg.steps), device=cfg.device)
        scheduler_sub.set_timesteps(int(cfg.steps), device=cfg.device)

        bg_path = job.bg_image_path if job.bg_image_path is not None else cfg.bg_image_path
        bg_strength = float(job.bg_strength) if job.bg_strength is not None else float(cfg.bg_strength)

        if bg_path:
            bg_img = load_image(bg_path, size=(cfg.width, cfg.height))
            bg_img.save(os.path.join(job_out, "bg_input.png"))

            lat_img = encode_image_latents(pipe, bg_img, generator, cfg.device, cfg.torch_dtype)
            lat_img = lat_img.repeat(int(cfg.batch), 1, 1, 1)

            num_steps = int(cfg.steps)
            init_timestep = max(1, min(num_steps, int(num_steps * bg_strength)))
            t_start = min(max(num_steps - init_timestep, 0), num_steps - 1)
            t_start_1 = timesteps[t_start : t_start + 1].to(cfg.device).repeat(int(cfg.batch))

            noise = torch.randn(lat_img.shape, generator=generator, device=lat_img.device, dtype=lat_img.dtype)
            lat_bg = pipe.scheduler.add_noise(lat_img, noise, t_start_1)
            lat_sub = lat_bg.clone()

            if float(cfg.sub_init_noise) > 0:
                sub_noise = torch.randn(lat_sub.shape, generator=generator, device=lat_sub.device, dtype=lat_sub.dtype)
                lat_sub = lat_sub + float(cfg.sub_init_noise) * sub_noise

            return lat_bg, lat_sub, timesteps[t_start:], scheduler_bg, scheduler_sub

        lat_bg = prepare_noise_latents(
            pipe,
            int(cfg.batch),
            int(cfg.height),
            int(cfg.width),
            generator,
            cfg.device,
            cfg.torch_dtype,
        )
        return lat_bg, lat_bg.clone(), timesteps, scheduler_bg, scheduler_sub

    def _apply_coupling(self, lat_out, lat_bg_next, lat_sub_next):
        mode = str(self.cfg.couple_mode).lower()
        if mode == "both":
            return lat_out, lat_out
        if mode == "bg_only":
            return lat_out, lat_sub_next
        return lat_bg_next, lat_sub_next

    @torch.no_grad()
    def run_one(self, job: AnimalJob, log_every: Optional[int] = None, show_preview: Optional[bool] = None):
        cfg, pipe = self.cfg, self.pipe
        job_out = self._job_out_dir(job)
        seed = int(job.seed) if job.seed is not None else int(cfg.seed)
        seed_everything(seed)

        gs_bg = float(getattr(cfg, "guidance_bg", getattr(cfg, "guidance_scale", 1.0)))
        gs_sub = float(getattr(cfg, "guidance_sub", getattr(cfg, "guidance_scale", 1.0)))
        do_cfg_bg, do_cfg_sub = gs_bg > 1.0, gs_sub > 1.0
        log_every = int(cfg.log_every if log_every is None else log_every)

        prep = self.preprocess(job.image_path, seed=seed)
        img_canny = to_torch_image_hint(prep["canny_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)
        img_soft = to_torch_image_hint(prep["soft_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)

        prompt_bg = job.prompt_bg or cfg.prompt_bg
        prompt_sub = format_prompt(job.subject_template or cfg.prompt_subject_template, job.animals)
        negative = job.negative or cfg.negative

        emb_bg = encode_prompt(pipe, prompt_bg, negative, int(cfg.batch), do_cfg_bg, cfg.device)
        emb_sub = encode_prompt(pipe, prompt_sub, negative, int(cfg.batch), do_cfg_sub, cfg.device)

        generator = torch.Generator(device=cfg.device).manual_seed(seed)
        lat_bg, lat_sub, timesteps_run, scheduler_bg, scheduler_sub = self._init_latents_and_timesteps(job_out, job, generator)

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            int(cfg.height) // 8,
            int(cfg.width) // 8,
            cfg,
            job_out=job_out,
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        w_down, w_mid = self._probe_weights(lat_sub, timesteps_run[0], emb_sub, img_canny[:1], do_cfg=do_cfg_sub)

        amp_ctx = (
            torch.autocast(device_type="cuda", dtype=cfg.torch_dtype)
            if cfg.device.startswith("cuda")
            else contextlib.nullcontext()
        )

        lat_out = lat_bg.clone()
        num_run = len(timesteps_run)

        with amp_ctx:
            for i, t in enumerate(timesteps_run):
                gate = control_gate(
                    i,
                    num_run,
                    float(cfg.start_frac_control),
                    float(cfg.end_frac_control),
                    str(cfg.control_gate_kind),
                )

                eps_bg = unet_with_optional_control(
                    pipe,
                    scheduler_bg,
                    lat_bg,
                    t,
                    emb_bg,
                    do_cfg_bg,
                    gs_bg,
                    None,
                    None,
                    device=cfg.device,
                    dtype=cfg.torch_dtype,
                )
                lat_bg_next = scheduler_bg.step(eps_bg, t, lat_bg).prev_sample

                eps_sub = unet_with_optional_control(
                    pipe,
                    scheduler_sub,
                    lat_sub,
                    t,
                    emb_sub,
                    do_cfg_sub,
                    gs_sub,
                    self.control_nets,
                    [img_canny, img_soft],
                    list(cfg.control_scales),
                    w_down,
                    w_mid,
                    gate,
                    bool(cfg.guess_mode),
                    cfg.device,
                    cfg.torch_dtype,
                )
                lat_sub_next = scheduler_sub.step(eps_sub, t, lat_sub).prev_sample

                alpha = blend_alpha(
                    i,
                    num_run,
                    float(cfg.blend_start_frac),
                    float(cfg.blend_end_frac),
                    float(cfg.blend_alpha_end),
                    str(cfg.blend_profile),
                )

                lat_out = lat_bg_next * (1.0 - alpha * mask) + lat_sub_next * (alpha * mask)
                lat_bg, lat_sub = self._apply_coupling(lat_out, lat_bg_next, lat_sub_next)

                if log_every and (i + 1) % log_every == 0:
                    print(f"[{job.animals}] step {i + 1:>3}/{num_run} | gate={gate:.3f} | alpha={alpha:.3f}")

        imgs = decode_latents(pipe, lat_out)
        paths = []
        for k, img in enumerate(imgs):
            out_path = os.path.join(job_out, f"out_{slugify(job.animals)}_{k:02d}_seed{seed}.png")
            img.save(out_path)
            paths.append(out_path)

        self._save_run_config(
            job_out,
            job,
            extra={
                "prompt_bg": prompt_bg,
                "prompt_sub": prompt_sub,
                "bg_strength": job.bg_strength or cfg.bg_strength,
            },
        )
        return imgs, paths


# ============================================================
# 9) Scoring / UI integration helpers
# ============================================================
def hidden_score(out_img_pil: Image.Image, outline_like_hw: np.ndarray) -> float:
    img = np.array(out_img_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160).astype(np.float32) / 255.0
    ring = (outline_like_hw > 0.25).astype(np.uint8)
    in_mean = float(edges[ring > 0].mean()) if (ring > 0).any() else 0.0
    out_mean = float(edges[ring == 0].mean())
    return float(-abs((in_mean - out_mean) - 0.01) - 0.5 * in_mean)


# ------------------------------------------------------------
# Demo metrics / quantitative proxy calculations
# ------------------------------------------------------------
def _to_gray01_from_pil(img_pil: Image.Image, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    img = img_pil.convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    arr = np.array(img).astype(np.uint8)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    return np.clip(gray, 0.0, 1.0)


def _to_gray01_from_any(x: Any, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    if isinstance(x, Image.Image):
        return _to_gray01_from_pil(x, size=size)
    arr = np.asarray(x)
    if arr.ndim == 3:
        arr = cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    arr = arr.astype(np.float32)
    if arr.max() > 1.5:
        arr = arr / 255.0
    if size is not None:
        arr = cv2.resize(arr, size, interpolation=cv2.INTER_LINEAR)
    return np.clip(arr, 0.0, 1.0)


def _global_ssim(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    mux = float(x.mean())
    muy = float(y.mean())
    varx = float(((x - mux) ** 2).mean())
    vary = float(((y - muy) ** 2).mean())
    cov = float(((x - mux) * (y - muy)).mean())

    denom = (mux ** 2 + muy ** 2 + c1) * (varx + vary + c2)
    if abs(denom) < 1e-12:
        return 0.0
    return float(((2 * mux * muy + c1) * (2 * cov + c2)) / denom)


def compute_ssim_bg(out_img: Image.Image, bg_img: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_gray = _to_gray01_from_pil(out_img, size=size)
    bg_gray = _to_gray01_from_pil(bg_img, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)
    bg_region = 1.0 - np.clip(mask, 0.0, 1.0)

    x = out_gray * bg_region
    y = bg_gray * bg_region

    try:
        from skimage.metrics import structural_similarity as ssim
        return float(ssim(x, y, data_range=1.0))
    except Exception:
        return _global_ssim(x, y)


def compute_edge_similarity(out_img: Image.Image, subject_edge_hint: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_rgb = np.array(out_img.convert("RGB").resize(size, Image.LANCZOS)).astype(np.uint8)
    out_gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY)
    e_out = cv2.Canny(out_gray, 80, 160).astype(np.float32) / 255.0

    e_sub = _to_gray01_from_pil(subject_edge_hint, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)

    a = (e_sub * mask).reshape(-1).astype(np.float32)
    b = (e_out * mask).reshape(-1).astype(np.float32)

    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom < 1e-8:
        return 0.0
    return float(np.dot(a, b) / denom)


def _image_edge_density(img_pil: Image.Image, mask_img: Optional[Image.Image] = None) -> float:
    gray = _to_gray01_from_pil(img_pil)
    edges = cv2.Canny((gray * 255).astype(np.uint8), 80, 160).astype(np.float32) / 255.0
    if mask_img is None:
        return float(edges.mean())
    mask = _to_gray01_from_pil(mask_img, size=img_pil.size)
    active = mask > 0.25
    if not active.any():
        return 0.0
    return float(edges[active].mean())


def compute_demo_metrics(
    runner: CamoRunner,
    job: AnimalJob,
    out_img: Image.Image,
    previews: Dict[str, Any],
    hidden_score_value: float,
) -> Dict[str, Any]:
    cfg = runner.cfg
    bg_img = previews["preview_bg_original"]
    canny_hint_img = previews["preview_canny"]
    mask_img = previews["preview_mask"]

    mask_np = _to_gray01_from_pil(mask_img)
    active = mask_np > 0.25

    metrics = {
        "seed": int(job.seed),
        "subject_name": str(job.animals),
        "subject_path": str(job.image_path),
        "background_path": str(job.bg_image_path or cfg.bg_image_path),
        "hidden_score": float(hidden_score_value),
        "SSIM_bg": compute_ssim_bg(out_img, bg_img, mask_img),
        "S_edge": compute_edge_similarity(out_img, canny_hint_img, mask_img),
        "mask_mean": float(mask_np.mean()),
        "mask_max": float(mask_np.max()),
        "mask_active_ratio": float(active.mean()),
        "output_edge_density_global": _image_edge_density(out_img),
        "output_edge_density_in_mask": _image_edge_density(out_img, mask_img),
        "layout_effective_scale": float(previews["attention_info"].get("scale_eff", 0.0)),
        "layout_scale": float(cfg.blend_mask_scale),
        "layout_offset_x": float(cfg.blend_mask_offset_x),
        "layout_offset_y": float(cfg.blend_mask_offset_y),
        "layout_rotation_deg": float(cfg.blend_mask_rotation_deg),
        "steps": int(cfg.steps),
        "bg_strength": float(cfg.bg_strength),
        "guidance_bg": float(cfg.guidance_bg),
        "guidance_sub": float(cfg.guidance_sub),
        "canny_scale": float(cfg.control_scales[0]),
        "softedge_scale": float(cfg.control_scales[1]),
        "blend_profile": str(cfg.blend_profile),
        "blend_start_frac": float(cfg.blend_start_frac),
        "blend_end_frac": float(cfg.blend_end_frac),
        "blend_alpha_end": float(cfg.blend_alpha_end),
        "mask_mode": str(cfg.mask_mode),
        "soft_mask_blur": int(cfg.soft_mask_blur),
        "soft_mask_gamma": float(cfg.soft_mask_gamma),
        "soft_mask_floor": float(cfg.soft_mask_floor),
        "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
    }
    return metrics


def print_demo_metrics(metrics: Dict[str, Any]) -> None:
    print("\n" + "=" * 72)
    print("Calculated Demo Metrics")
    print("=" * 72)
    ordered_keys = [
        "seed",
        "subject_name",
        "SSIM_bg",
        "S_edge",
        "hidden_score",
        "mask_mean",
        "mask_max",
        "mask_active_ratio",
        "output_edge_density_global",
        "output_edge_density_in_mask",
        "layout_effective_scale",
        "steps",
        "bg_strength",
        "guidance_bg",
        "guidance_sub",
        "canny_scale",
        "softedge_scale",
        "blend_profile",
        "blend_start_frac",
        "blend_end_frac",
        "blend_alpha_end",
    ]
    for key in ordered_keys:
        value = metrics.get(key)
        if isinstance(value, float):
            print(f"{key:>28}: {value:.6f}")
        else:
            print(f"{key:>28}: {value}")
    print("=" * 72 + "\n")


def build_demo_metrics_html(metrics: Dict[str, Any]) -> str:
    core_rows = [
        ("SSIM_bg", metrics.get("SSIM_bg")),
        ("S_edge", metrics.get("S_edge")),
        ("Hidden Score", metrics.get("hidden_score")),
        ("Mask Mean", metrics.get("mask_mean")),
        ("Mask Active Ratio", metrics.get("mask_active_ratio")),
        ("Output Edge Density", metrics.get("output_edge_density_global")),
        ("Output Edge Density in Mask", metrics.get("output_edge_density_in_mask")),
        ("Layout Effective Scale", metrics.get("layout_effective_scale")),
    ]

    param_rows = [
        ("Seed", metrics.get("seed")),
        ("Steps", metrics.get("steps")),
        ("BG Strength", metrics.get("bg_strength")),
        ("Guidance BG", metrics.get("guidance_bg")),
        ("Guidance Subject", metrics.get("guidance_sub")),
        ("Canny Scale", metrics.get("canny_scale")),
        ("SoftEdge Scale", metrics.get("softedge_scale")),
        ("Blend Profile", metrics.get("blend_profile")),
        ("Blend Start", metrics.get("blend_start_frac")),
        ("Blend End", metrics.get("blend_end_frac")),
        ("Alpha End", metrics.get("blend_alpha_end")),
    ]

    def fmt(v: Any) -> str:
        if isinstance(v, float):
            return f"{v:.6f}"
        return str(v)

    def rows_html(rows: List[Tuple[str, Any]]) -> str:
        return "".join(
            f"<tr><td style='padding:5px 10px; color:#cbd5e1;'>{html_lib.escape(str(k))}</td>"
            f"<td style='padding:5px 10px; color:#f8fafc; font-weight:600;'>{html_lib.escape(fmt(v))}</td></tr>"
            for k, v in rows
        )

    return f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">Calculated Output Metrics</div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:14px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Proxy Metrics</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(core_rows)}</table>
            </div>
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Main Parameters</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(param_rows)}</table>
            </div>
        </div>
    </div>
    """


def apply_ui_params_to_runner(runner: CamoRunner, ui: Dict[str, Any]) -> None:
    cfg = runner.cfg
    cfg.steps = int(ui["steps"])
    cfg.bg_strength = float(ui["bg_strength"])
    cfg.guidance_bg = float(ui["guidance_bg"])
    cfg.guidance_sub = float(ui["guidance_sub"])
    cfg.sub_init_noise = float(ui["sub_init_noise"])
    cfg.couple_mode = str(ui["couple_mode"])

    cfg.canny_low = int(ui["canny_low"])
    cfg.canny_high = int(ui["canny_high"])
    cfg.canny_blur_ks = int(ui["canny_blur_ks"])
    cfg.use_broken_edges = bool(ui["use_broken_edges"])
    cfg.broken_drop_prob = float(ui["broken_drop_prob"])

    cfg.mask_mode = str(ui["mask_mode"])
    cfg.mask_dilate = int(ui["mask_dilate"])
    cfg.mask_close_ks = int(ui["mask_close_ks"])
    cfg.mask_blur = int(ui["mask_blur"])
    cfg.mask_ring_k = int(ui["mask_ring_k"])
    cfg.mask_ring_blur = int(ui["mask_ring_blur"])
    cfg.mask_gamma = float(ui["mask_gamma"])
    cfg.mask_auto_invert = bool(ui["mask_auto_invert"])

    cfg.outer_weight = float(ui["outer_weight"])
    cfg.feature_weight = float(ui["feature_weight"])
    cfg.fill_weight = float(ui["fill_weight"])
    cfg.feature_thresh = int(ui["feature_thresh"])
    cfg.feature_open_ks = int(ui["feature_open_ks"])
    cfg.feature_blur = int(ui["feature_blur"])
    cfg.fill_blur = int(ui["fill_blur"])

    cfg.use_soft_grad_mask = bool(ui["use_soft_grad_mask"])
    cfg.soft_grad_dilation_k = int(ui["soft_grad_dilation_k"])
    cfg.soft_grad_blur_k = int(ui["soft_grad_blur_k"])
    cfg.soft_grad_kernel_shape = str(ui["soft_grad_kernel_shape"])
    cfg.soft_grad_iterations = int(ui["soft_grad_iterations"])
    cfg.soft_grad_use_skeleton = bool(ui["soft_grad_use_skeleton"])
    cfg.soft_grad_skeleton_stage = str(ui["soft_grad_skeleton_stage"])
    cfg.soft_grad_post_skel_dilate_k = int(ui["soft_grad_post_skel_dilate_k"])

    cfg.soft_mask_blur = int(ui["soft_mask_blur"])
    cfg.soft_mask_gamma = float(ui["soft_mask_gamma"])
    cfg.soft_mask_invert = bool(ui["soft_mask_invert"])
    cfg.soft_mask_floor = float(ui["soft_mask_floor"])
    cfg.soft_mask_ceiling = float(ui["soft_mask_ceiling"])

    cfg.control_scales = (float(ui["canny_scale"]), float(ui["soft_scale"]))
    cfg.start_frac_control = float(ui["start_frac"])
    cfg.end_frac_control = float(ui["end_frac"])
    cfg.control_gate_kind = str(ui["gate_kind"])

    cfg.blend_profile = str(ui["blend_profile"])
    cfg.blend_start_frac = float(ui["blend_start"])
    cfg.blend_end_frac = float(ui["blend_end"])
    cfg.blend_alpha_end = float(ui["alpha_end"])
    cfg.auto_attention_compensation = bool(ui["auto_attention_compensation"])
    cfg.attention_scale_ref = float(ui["attention_scale_ref"])
    cfg.attention_gamma_ref = float(ui["attention_gamma_ref"])
    cfg.attention_ceiling_ref = float(ui["attention_ceiling_ref"])
    cfg.attention_blur_ref = int(ui["attention_blur_ref"])
    cfg.blend_mask_scale = float(ui["blend_mask_scale"])
    cfg.blend_mask_offset_x = float(ui["blend_mask_offset_x"])
    cfg.blend_mask_offset_y = float(ui["blend_mask_offset_y"])
    cfg.blend_mask_rotation_deg = float(ui["blend_mask_rotation_deg"])
    cfg.randomize_seed_each_generate = bool(ui["randomize_seed_each_generate"])


def build_job_from_ui(ui: Dict[str, Any]) -> AnimalJob:
    animal_path_raw = str(ui["path_animal"]).strip()
    bg_path_raw = str(ui["path_bg"]).strip()

    resolved_animal = resolve_image_path(animal_path_raw, fallback_dir=KAGGLE_ANIMALS_DIR)
    if resolved_animal is None:
        raise gr.Error(f"No valid subject image found from: {animal_path_raw}")

    resolved_bg = resolve_image_path(bg_path_raw, fallback_dir=KAGGLE_BG_DIR) if bg_path_raw or KAGGLE_BG_DIR else None

    return AnimalJob(
        image_path=resolved_animal,
        bg_image_path=resolved_bg,
        animals=str(ui["animal_name"]),
        seed=int(ui["seed"]),
        prompt_bg=str(ui["prompt_bg"]),
        subject_template=str(ui["prompt_sub"]),
        negative=str(ui["negative_prompt"]),
        name=f"UI_SoftGradMask_{int(time.time())}",
        bg_strength=float(ui["bg_strength"]),
    )



def stage2_canny_breakdown(canny_hint_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    edges = canny_hint_u8[..., 0].astype(np.uint8)

    contour_agg = edges.copy()

    if int(cfg.mask_dilate) > 1:
        k = _odd(int(cfg.mask_dilate))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.dilate(contour_agg, kernel, iterations=1)

    if int(cfg.mask_close_ks) > 1:
        k = _odd(int(cfg.mask_close_ks))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.morphologyEx(
            contour_agg,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=max(1, int(getattr(cfg, "mask_close_iter", 1))),
        )

    inv = cv2.bitwise_not(contour_agg)
    flood = inv.copy()
    h, w = contour_agg.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)
    region_filled = (flood > 0).astype(np.uint8) * 255

    smoothing = region_filled.copy()
    if int(cfg.mask_blur) > 1:
        k = _odd(int(cfg.mask_blur))
        smoothing = cv2.GaussianBlur(smoothing, (k, k), 0)

    return {
        "canny_edges": edges,
        "contour_aggregation": contour_agg,
        "region_filling": region_filled,
        "canny_smoothing": smoothing,
    }


def stage2_softedge_breakdown(soft_input_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    if soft_input_u8.ndim == 3:
        gray = cv2.cvtColor(soft_input_u8.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = soft_input_u8.astype(np.uint8)

    smoothed = gray.astype(np.float32) / 255.0
    if int(cfg.soft_mask_blur) > 1:
        k = _odd(int(cfg.soft_mask_blur))
        smoothed = cv2.GaussianBlur(smoothed, (k, k), 0)

    remap_input = smoothed.copy()
    if bool(cfg.soft_mask_invert):
        remap_input = 1.0 - remap_input
    remap_input = np.clip(remap_input, 0.0, 1.0)

    gamma_remap = remap_input.copy()
    gamma = float(cfg.soft_mask_gamma)
    if gamma != 1.0:
        gamma_remap = gamma_remap ** gamma

    floor = float(np.clip(cfg.soft_mask_floor, 0.0, 1.0))
    ceiling = float(np.clip(cfg.soft_mask_ceiling, 0.0, 1.0))
    clipped = gamma_remap.copy()
    if ceiling > floor:
        clipped = np.clip((clipped - floor) / (ceiling - floor), 0.0, 1.0)

    final_soft_mask = np.clip(clipped, 0.0, 1.0)

    return {
        "soft_gray": gray01_to_u8(gray.astype(np.float32) / 255.0),
        "soft_smoothing": gray01_to_u8(smoothed),
        "soft_after_gamma_remap": gray01_to_u8(gamma_remap),
        "soft_after_clip": gray01_to_u8(final_soft_mask),
    }


def _stats_from_array(name: str, arr: np.ndarray) -> Dict[str, Any]:
    x = np.asarray(arr).astype(np.float32)
    if x.ndim == 3:
        x = x.mean(axis=2)

    nonzero = float((x > 0).sum())
    total = float(x.size)
    ratio = nonzero / total if total > 0 else 0.0

    return {
        "name": name,
        "shape": tuple(x.shape),
        "min": float(x.min()) if x.size else 0.0,
        "max": float(x.max()) if x.size else 0.0,
        "mean": float(x.mean()) if x.size else 0.0,
        "std": float(x.std()) if x.size else 0.0,
        "nonzero_ratio": float(ratio),
    }


def build_stage2_value_dict(
    runner: CamoRunner,
    job: AnimalJob,
    prep: Dict[str, Any],
    mask: torch.Tensor,
    canny_steps: Dict[str, np.ndarray],
    soft_steps: Dict[str, np.ndarray],
) -> Dict[str, Any]:
    cfg = runner.cfg

    mask_pil = mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height)
    mask_np = np.array(mask_pil.convert("L"))

    info = {
        "paths": {
            "animal_image": job.image_path,
            "background_image": job.bg_image_path or cfg.bg_image_path,
            "out_dir": runner._job_out_dir(job),
        },
        "layout": {
            "blend_mask_scale": float(cfg.blend_mask_scale),
            "blend_mask_offset_x": float(cfg.blend_mask_offset_x),
            "blend_mask_offset_y": float(cfg.blend_mask_offset_y),
            "blend_mask_rotation_deg": float(cfg.blend_mask_rotation_deg),
        },
        "canny_params": {
            "canny_low": int(cfg.canny_low),
            "canny_high": int(cfg.canny_high),
            "canny_blur_ks": int(cfg.canny_blur_ks),
            "use_broken_edges": bool(cfg.use_broken_edges),
            "broken_drop_prob": float(cfg.broken_drop_prob),
        },
        "support_params": {
            "mask_dilate": int(cfg.mask_dilate),
            "mask_close_ks": int(cfg.mask_close_ks),
            "mask_close_iter": int(getattr(cfg, "mask_close_iter", 1)),
            "mask_blur": int(cfg.mask_blur),
            "mask_mode": str(cfg.mask_mode),
        },
        "soft_mask_params": {
            "use_soft_grad_mask": bool(cfg.use_soft_grad_mask),
            "soft_grad_dilation_k": int(cfg.soft_grad_dilation_k),
            "soft_grad_blur_k": int(cfg.soft_grad_blur_k),
            "soft_grad_kernel_shape": str(cfg.soft_grad_kernel_shape),
            "soft_grad_iterations": int(cfg.soft_grad_iterations),
            "soft_grad_use_skeleton": bool(cfg.soft_grad_use_skeleton),
            "soft_grad_skeleton_stage": str(cfg.soft_grad_skeleton_stage),
            "soft_grad_post_skel_dilate_k": int(cfg.soft_grad_post_skel_dilate_k),
            "soft_mask_blur": int(cfg.soft_mask_blur),
            "soft_mask_gamma": float(cfg.soft_mask_gamma),
            "soft_mask_invert": bool(cfg.soft_mask_invert),
            "soft_mask_floor": float(cfg.soft_mask_floor),
            "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
        },
        "attention_info": prep["attention_info"],
        "arrays": {
            "canny_hint": _stats_from_array("canny_hint", prep["canny_hint"]),
            "contour_aggregation": _stats_from_array("contour_aggregation", canny_steps["contour_aggregation"]),
            "region_filling": _stats_from_array("region_filling", canny_steps["region_filling"]),
            "canny_smoothing": _stats_from_array("canny_smoothing", canny_steps["canny_smoothing"]),
            "sil_hw": _stats_from_array("sil_hw", prep["sil_hw"]),
            "feature_hw": _stats_from_array("feature_hw", prep["feature_hw"]),
            "comp_hw": _stats_from_array("comp_hw", prep["comp_hw"]),
            "soft_hint": _stats_from_array("soft_hint", prep["soft_hint"]),
            "soft_for_mask": _stats_from_array("soft_for_mask", prep["soft_for_mask"]),
            "soft_gray": _stats_from_array("soft_gray", soft_steps["soft_gray"]),
            "soft_smoothing": _stats_from_array("soft_smoothing", soft_steps["soft_smoothing"]),
            "soft_after_gamma_remap": _stats_from_array("soft_after_gamma_remap", soft_steps["soft_after_gamma_remap"]),
            "soft_after_clip": _stats_from_array("soft_after_clip", soft_steps["soft_after_clip"]),
            "soft_mask_hw": _stats_from_array("soft_mask_hw", prep["soft_mask_hw"]),
            "latent_mask": _stats_from_array("latent_mask", mask_np),
        },
    }
    return info


def build_stage2_debug_html(stage2_info: Dict[str, Any]) -> str:
    def render_dict(d: Dict[str, Any]) -> str:
        rows = []
        for k, v in d.items():
            if isinstance(v, dict):
                rows.append(
                    f"<tr><td colspan='2' style='padding-top:10px; font-weight:700; color:#93c5fd'>{k}</td></tr>"
                )
                for kk, vv in v.items():
                    rows.append(
                        f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{kk}</td>"
                        f"<td style='padding:4px 10px; color:#f8fafc'>{vv}</td></tr>"
                    )
            else:
                rows.append(
                    f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{k}</td>"
                    f"<td style='padding:4px 10px; color:#f8fafc'>{v}</td></tr>"
                )
        return "".join(rows)

    arrays_html = []
    for name, stats in stage2_info["arrays"].items():
        arrays_html.append(f"""
        <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
            <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">{name}</div>
            <table style="width:100%; font-size:13px; border-collapse:collapse;">
                <tr><td style="padding:3px 8px; color:#cbd5e1;">shape</td><td style="padding:3px 8px; color:#f8fafc;">{stats['shape']}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">min</td><td style="padding:3px 8px; color:#f8fafc;">{stats['min']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">max</td><td style="padding:3px 8px; color:#f8fafc;">{stats['max']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">mean</td><td style="padding:3px 8px; color:#f8fafc;">{stats['mean']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">std</td><td style="padding:3px 8px; color:#f8fafc;">{stats['std']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">nonzero_ratio</td><td style="padding:3px 8px; color:#f8fafc;">{stats['nonzero_ratio']:.4f}</td></tr>
            </table>
        </div>
        """)

    html = f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">
            Stage-2 Diagnostic Values
        </div>

        <div style="display:grid; grid-template-columns:1fr 1fr; gap:16px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Config Summary</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict({
                        "paths": stage2_info["paths"],
                        "layout": stage2_info["layout"],
                        "canny_params": stage2_info["canny_params"],
                        "support_params": stage2_info["support_params"],
                        "soft_mask_params": stage2_info["soft_mask_params"],
                    })}
                </table>
            </div>

            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Attention / Compensation</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict(stage2_info["attention_info"])}
                </table>
            </div>
        </div>

        <div style="margin-top:14px; display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px;">
            {''.join(arrays_html)}
        </div>
    </div>
    """
    return html


def make_debug_previews(runner: CamoRunner, job: AnimalJob):
    prep = runner.preprocess(job.image_path, seed=int(job.seed))
    job_out = runner._job_out_dir(job)
    latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8

    mask = build_mask_latent(
        prep["sil_hw"],
        prep["feature_hw"],
        latent_h,
        latent_w,
        runner.cfg,
        job_out=job_out,
        soft_hint=prep["soft_hint"],
        soft_for_mask=prep["soft_for_mask"],
        soft_mask_hw=prep["soft_mask_hw"],
    )

    canny_steps = stage2_canny_breakdown(prep["canny_hint"], runner.cfg)
    soft_steps = stage2_softedge_breakdown(prep["soft_for_mask"], runner.cfg)

    stage2_info = build_stage2_value_dict(
        runner=runner,
        job=job,
        prep=prep,
        mask=mask,
        canny_steps=canny_steps,
        soft_steps=soft_steps,
    )

    score_base = prep["sil_hw"].copy()
    stage2_json = json.dumps(stage2_info, ensure_ascii=False, indent=2)
    stage2_html = build_stage2_debug_html(stage2_info)

    previews = {
        "preview_bg_original": load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height)),
        "preview_input": prep["src"],
        "preview_canny": rgb_uint8_to_pil(prep["canny_hint"]),
        "preview_soft": rgb_uint8_to_pil(prep["soft_hint"]),
        "preview_sil": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_feat": gray_to_pil((prep["feature_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_comp": gray_to_pil((prep["comp_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_for_mask": rgb_uint8_to_pil(prep["soft_for_mask"]),
        "preview_soft_hw": gray_to_pil((prep["soft_mask_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_mask": mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height),
        "preview_contour_agg": gray_to_pil(canny_steps["contour_aggregation"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_region_filling": gray_to_pil(canny_steps["region_filling"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_canny_smoothing": gray_to_pil(canny_steps["canny_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_aux_support": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_gray": gray_to_pil(soft_steps["soft_gray"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_smoothing": gray_to_pil(soft_steps["soft_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_remap": gray_to_pil(soft_steps["soft_after_gamma_remap"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_clip": gray_to_pil(soft_steps["soft_after_clip"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_stage2_report_html": stage2_html,
        "preview_stage2_report_json": stage2_json,
        "score_base": score_base,
        "attention_info": prep["attention_info"],
    }
    return previews


# ============================================================
# 10) Main / Gradio UI
# ============================================================
if __name__ == "__main__":
    cfg = build_cfg(CFG)
    runner = CamoRunner(cfg).build_pipe()


    # ------------------------------------------------------------
    # Upload helpers
    # ------------------------------------------------------------
    # The original pipeline expects image paths. These helpers let the
    # Gradio UI accept uploaded PIL images, save them into out_dir, and
    # then reuse the existing path-based pipeline without changing the
    # core model code.
    def save_uploaded_image(img_pil: Optional[Image.Image], prefix: str) -> Optional[str]:
        if img_pil is None:
            return None

        upload_dir = os.path.join(runner.cfg.out_dir, "_ui_uploads")
        ensure_dir(upload_dir)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        path = os.path.join(upload_dir, f"{prefix}_{timestamp}.png")
        img_pil.convert("RGB").save(path)
        return path


    def apply_uploads_to_ui(
        ui: Dict[str, Any],
        subject_upload: Optional[Image.Image] = None,
        bg_upload: Optional[Image.Image] = None,
    ) -> Dict[str, Any]:
        ui = dict(ui)

        subject_path = save_uploaded_image(subject_upload, "subject")
        bg_path = save_uploaded_image(bg_upload, "background")

        # If the user uploads images, uploaded images override Textbox paths.
        # If no upload exists, the old Textbox path behavior remains unchanged.
        if subject_path is not None:
            ui["path_animal"] = subject_path

        if bg_path is not None:
            ui["path_bg"] = bg_path

        return ui


    def render_layout_editor_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed), layout_override=(1.0, 0.0, 0.0, 0.0))

        latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8
        cfg_preview = SimpleNamespace(**vars(runner.cfg))
        cfg_preview.save_mask_images = False
        cfg_preview.blend_mask_scale = 1.0
        cfg_preview.blend_mask_offset_x = 0.0
        cfg_preview.blend_mask_offset_y = 0.0
        cfg_preview.blend_mask_rotation_deg = 0.0

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            latent_h,
            latent_w,
            cfg_preview,
            job_out=runner._job_out_dir(job),
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        bg_img = load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height))
        overlay_img = mask_preview_to_rgba(mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height))
        return build_interactive_layout_html(
            bg_img=bg_img,
            overlay_img=overlay_img,
            width=runner.cfg.width,
            height=runner.cfg.height,
            init_scale=float(ui["blend_mask_scale"]),
            init_offset_x=float(ui["blend_mask_offset_x"]),
            init_offset_y=float(ui["blend_mask_offset_y"]),
            init_rotation_deg=float(ui["blend_mask_rotation_deg"]),
        )

    def summarize_layout_attention_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        return format_attention_status(prep["attention_info"], runner.cfg.mask_mode)

    def refresh_layout_editor(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        html = render_layout_editor_from_ui(ui)
        status = "Layout canvas has been refreshed from the Canny/SoftEdge support. " + summarize_layout_attention_from_ui(ui)
        return html, status

    def capture_current_scale_reference(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        scale_eff = float(prep["attention_info"]["scale_eff"])
        status = f"Captured the current layout scale as the reference: {scale_eff:.3f}"
        return scale_eff, status

    def safe_initial_layout_html() -> str:
        try:
            return render_layout_editor_from_ui(normalize_ui_params(UI_DEFAULTS))
        except Exception as e:
            return (
                "<div style='padding:12px;border:1px solid rgba(255,255,255,.1);border-radius:12px'>"
                f"Could not create the interactive layout canvas. Please check the image paths and click <b>Refresh Interactive Canvas</b>.<br><small>{e}</small></div>"
            )

    JS_SYNC_CANVAS_TO_ARGS = r"""
    (...args) => {
      const state = window.camoPlacementState;
      const syncHost = (elemId, value) => {
        const host = document.getElementById(elemId);
        if (!host) return;
        host.querySelectorAll('input').forEach((input) => {
          input.value = String(Number(value).toFixed(4));
          input.dispatchEvent(new Event('input', { bubbles: true }));
          input.dispatchEvent(new Event('change', { bubbles: true }));
        });
      };
      if (state) {
        const n = args.length;
        if (n >= 4) {
          args[n - 4] = Number(state.scale);
          args[n - 3] = Number(state.offsetX);
          args[n - 2] = Number(state.offsetY);
          args[n - 1] = Number(state.rotationDeg);
        }
        syncHost('blend-mask-scale', state.scale);
        syncHost('blend-mask-offset-x', state.offsetX);
        syncHost('blend-mask-offset-y', state.offsetY);
        syncHost('blend-mask-rotation-deg', state.rotationDeg);
      }
      return args;
    }
    """

    def generate_preview(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        if bool(ui["randomize_seed_each_generate"]):
            ui["seed"] = fresh_seed()

        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)

        print(f"\\n[UI] Generating {job.animals} - Seed {job.seed} - Steps {runner.cfg.steps}...")

        previews = make_debug_previews(runner, job)
        imgs, _ = runner.run_one(job, log_every=0, show_preview=False)
        score = hidden_score(imgs[0], previews["score_base"])
        demo_metrics = compute_demo_metrics(runner, job, imgs[0], previews, hidden_score_value=score)
        print_demo_metrics(demo_metrics)
        metrics_json = json.dumps(demo_metrics, ensure_ascii=False, indent=2)
        metrics_html = build_demo_metrics_html(demo_metrics)
        save_json(os.path.join(runner._job_out_dir(job), "calculated_demo_metrics.json"), demo_metrics)

        status = (
            f"Generation completed | seed={job.seed} | "
            f"SSIM_bg={demo_metrics['SSIM_bg']:.4f} | "
            f"S_edge={demo_metrics['S_edge']:.4f}. "
            + format_attention_status(previews["attention_info"], runner.cfg.mask_mode)
        )

        return (
            int(job.seed),
            imgs[0],
            previews["preview_bg_original"],
            previews["preview_input"],
            previews["preview_canny"],
            previews["preview_soft"],
            previews["preview_sil"],
            previews["preview_feat"],
            previews["preview_comp"],
            previews["preview_soft_for_mask"],
            previews["preview_soft_hw"],
            previews["preview_mask"],
            previews["preview_contour_agg"],
            previews["preview_region_filling"],
            previews["preview_canny_smoothing"],
            previews["preview_aux_support"],
            previews["preview_soft_gray"],
            previews["preview_soft_smoothing"],
            previews["preview_soft_remap"],
            previews["preview_soft_clip"],
            f"Hidden Score: {score:.4f}",
            status,
            metrics_html,
            metrics_json,
            previews["preview_stage2_report_html"],
            previews["preview_stage2_report_json"],
        )

    def save_current_config(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filepath = os.path.join(runner.cfg.out_dir, f"ui_config_{timestamp}.json")
        payload = {
            "timestamp": timestamp,
            "version": 5,
            "layout_stage": "preprocess_hints",
            "params": ui,
            "layout_params": {
                "layout_scale": ui["blend_mask_scale"],
                "layout_offset_x": ui["blend_mask_offset_x"],
                "layout_offset_y": ui["blend_mask_offset_y"],
                "layout_rotation_deg": ui["blend_mask_rotation_deg"],
            },
            "attention_compensation": {
                "enabled": ui["auto_attention_compensation"],
                "reference_scale": ui["attention_scale_ref"],
                "reference_gamma": ui["attention_gamma_ref"],
                "reference_ceiling": ui["attention_ceiling_ref"],
                "reference_blur": ui["attention_blur_ref"],
            },
            "legacy_params": [ui[name] for name in UI_FIELD_NAMES],
        }
        save_json(filepath, payload)
        return f"Configuration saved at: {filepath}"

    def load_config_to_ui(config_path: str):
        config_path = str(config_path).strip()
        if not config_path:
            raise gr.Error("Please enter a JSON config path.")
        if not os.path.exists(config_path):
            raise gr.Error(f"File not found: {config_path}")

        with open(config_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        raw_params = payload.get("params", payload) if isinstance(payload, dict) else payload
        missing_keys = []
        if isinstance(raw_params, dict):
            missing_keys = [name for name in UI_FIELD_NAMES if name not in raw_params]
        elif isinstance(raw_params, list):
            missing_keys = UI_FIELD_NAMES[len(raw_params):]

        ui = normalize_ui_params(raw_params)
        auto_filled = [k for k in missing_keys if k in {"auto_attention_compensation", "attention_scale_ref", "attention_gamma_ref", "attention_ceiling_ref", "attention_blur_ref", "blend_mask_scale", "blend_mask_offset_x", "blend_mask_offset_y", "blend_mask_rotation_deg", "randomize_seed_each_generate"}]
        status = f"Loaded config: {config_path}"
        if auto_filled:
            status += " | Auto-filled defaults: " + ", ".join(auto_filled)
        layout_html = render_layout_editor_from_ui(ui)
        return [ui[name] for name in UI_FIELD_NAMES] + [status, layout_html]

    with gr.Blocks(theme=gr.themes.Base()) as demo:
        gr.Markdown("## Camouflage Master Tuning UI — Direct Layout on Canny / SoftEdge")

        with gr.Row():
            with gr.Column(scale=5):
                with gr.Tab("Inputs & Prompts"):
                    gr.Markdown(
                        "Upload images directly if you want to provide custom inputs. "
                        "If no image is uploaded, the code will use the two path fields as in the previous version."
                    )
                    subject_upload = gr.Image(label="Upload Subject Image", type="pil")
                    bg_upload = gr.Image(label="Upload Background Image", type="pil")

                    path_animal = gr.Textbox(value=UI_DEFAULTS["path_animal"], label="Animal Image Path / Subject Folder")
                    path_bg = gr.Textbox(value=UI_DEFAULTS["path_bg"], label="Background Image Path / Background Folder")
                    animal_name = gr.Textbox(value=UI_DEFAULTS["animal_name"], label="Animal Name")
                    prompt_bg = gr.Textbox(value=UI_DEFAULTS["prompt_bg"], label="Prompt Background", lines=2)
                    prompt_sub = gr.Textbox(value=UI_DEFAULTS["prompt_sub"], label="Prompt Subject Template", lines=2)
                    negative_prompt = gr.Textbox(value=UI_DEFAULTS["negative_prompt"], label="Negative Prompt", lines=2)

                with gr.Tab("Core Generation"):
                    steps = gr.Slider(1, 100, value=UI_DEFAULTS["steps"], step=1, label="Steps")
                    with gr.Row():
                        seed = gr.Number(value=UI_DEFAULTS["seed"], label="Seed", precision=0)
                        randomize_seed_each_generate = gr.Checkbox(value=UI_DEFAULTS["randomize_seed_each_generate"], label="Random Seed Each Generate")
                    bg_strength = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["bg_strength"], step=0.01, label="BG Strength")
                    guidance_bg = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_bg"], step=0.1, label="Guidance BG")
                    guidance_sub = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_sub"], step=0.1, label="Guidance Sub")
                    sub_init_noise = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["sub_init_noise"], step=0.01, label="Sub Init Noise")
                    couple_mode = gr.Dropdown(
                        choices=["none", "bg_only", "both"],
                        value=UI_DEFAULTS["couple_mode"],
                        label="Couple Mode",
                    )

                with gr.Tab("🎭 Edges & Mask"):
                    with gr.Row():
                        canny_low = gr.Slider(0, 255, value=UI_DEFAULTS["canny_low"], step=1, label="Canny Low")
                        canny_high = gr.Slider(0, 255, value=UI_DEFAULTS["canny_high"], step=1, label="Canny High")
                    canny_blur_ks = gr.Slider(1, 15, value=UI_DEFAULTS["canny_blur_ks"], step=2, label="Canny Blur KS")
                    use_broken_edges = gr.Checkbox(value=UI_DEFAULTS["use_broken_edges"], label="Use Broken Edges")
                    broken_drop_prob = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["broken_drop_prob"], step=0.01, label="Broken Drop Prob")

                    gr.Markdown("---")
                    mask_mode = gr.Dropdown(
                        choices=["silhouette", "outline", "composite", "softedge"],
                        value=UI_DEFAULTS["mask_mode"],
                        label="Mask Mode",
                    )
                    mask_dilate = gr.Slider(1, 31, value=UI_DEFAULTS["mask_dilate"], step=2, label="Mask Dilate")
                    mask_close_ks = gr.Slider(1, 31, value=UI_DEFAULTS["mask_close_ks"], step=2, label="Mask Close KS")
                    mask_blur = gr.Slider(1, 51, value=UI_DEFAULTS["mask_blur"], step=2, label="Mask Blur")
                    mask_ring_k = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_k"], step=2, label="Mask Ring K")
                    mask_ring_blur = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_blur"], step=2, label="Mask Ring Blur")
                    mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["mask_gamma"], step=0.1, label="Mask Gamma (old modes)")
                    mask_auto_invert = gr.Checkbox(value=UI_DEFAULTS["mask_auto_invert"], label="Mask Auto Invert")

                    gr.Markdown("#### Composite Mask Params (old modes)")
                    outer_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["outer_weight"], step=0.05, label="Outer Weight")
                    feature_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["feature_weight"], step=0.05, label="Feature Weight")
                    fill_weight = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["fill_weight"], step=0.05, label="Fill Weight")
                    feature_thresh = gr.Slider(0, 255, value=UI_DEFAULTS["feature_thresh"], step=1, label="Feature Thresh")
                    feature_open_ks = gr.Slider(1, 15, value=UI_DEFAULTS["feature_open_ks"], step=2, label="Feature Open KS")
                    feature_blur = gr.Slider(1, 21, value=UI_DEFAULTS["feature_blur"], step=2, label="Feature Blur")
                    fill_blur = gr.Slider(1, 31, value=UI_DEFAULTS["fill_blur"], step=2, label="Fill Blur")

                    gr.Markdown("#### SoftEdge → Kernel Gradient (for subject layout + final mask)")
                    use_soft_grad_mask = gr.Checkbox(value=UI_DEFAULTS["use_soft_grad_mask"], label="Use Kernel Gradient for Layout/Mask")
                    soft_grad_dilation_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_dilation_k"], step=2, label="Soft Grad Dilation K")
                    soft_grad_blur_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_blur_k"], step=2, label="Soft Grad Blur K")
                    soft_grad_kernel_shape = gr.Dropdown(
                        choices=["ellipse", "rect", "cross"],
                        value=UI_DEFAULTS["soft_grad_kernel_shape"],
                        label="Soft Grad Kernel Shape",
                    )
                    soft_grad_iterations = gr.Slider(1, 5, value=UI_DEFAULTS["soft_grad_iterations"], step=1, label="Soft Grad Iterations")
                    soft_grad_use_skeleton = gr.Checkbox(value=UI_DEFAULTS["soft_grad_use_skeleton"], label="Soft Grad Use Skeleton")
                    soft_grad_skeleton_stage = gr.Dropdown(
                        choices=["pre", "post"],
                        value=UI_DEFAULTS["soft_grad_skeleton_stage"],
                        label="Soft Grad Skeleton Stage",
                    )
                    soft_grad_post_skel_dilate_k = gr.Slider(0, 11, value=UI_DEFAULTS["soft_grad_post_skel_dilate_k"], step=1, label="Post Skeleton Dilate K")

                    gr.Markdown("#### Final Soft Mask Remap")
                    soft_mask_blur = gr.Slider(1, 31, value=UI_DEFAULTS["soft_mask_blur"], step=2, label="Soft Mask Blur")
                    soft_mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["soft_mask_gamma"], step=0.1, label="Soft Mask Gamma")
                    soft_mask_invert = gr.Checkbox(value=UI_DEFAULTS["soft_mask_invert"], label="Soft Mask Invert")
                    soft_mask_floor = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_floor"], step=0.01, label="Soft Mask Floor")
                    soft_mask_ceiling = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_ceiling"], step=0.01, label="Soft Mask Ceiling")

                    gr.Markdown("#### Auto Attention Compensation")
                    gr.Markdown("Enable this option to automatically compensate the soft-mask strength according to the effective silhouette size after dragging or resizing. When the object is smaller than the reference scale, the system reduces gamma/ceiling and slightly reduces blur so the subject remains visible.")
                    auto_attention_compensation = gr.Checkbox(value=UI_DEFAULTS["auto_attention_compensation"], label="Auto Attention Compensation")
                    with gr.Row():
                        attention_scale_ref = gr.Slider(0.05, 0.95, value=UI_DEFAULTS["attention_scale_ref"], step=0.01, label="Reference Scale")
                        btn_capture_scale_ref = gr.Button("Use Current Layout as Ref Scale")
                    with gr.Row():
                        attention_gamma_ref = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["attention_gamma_ref"], step=0.05, label="Reference Gamma")
                        attention_ceiling_ref = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["attention_ceiling_ref"], step=0.01, label="Reference Ceiling")
                    attention_blur_ref = gr.Slider(1, 31, value=UI_DEFAULTS["attention_blur_ref"], step=2, label="Reference Blur")

                with gr.Tab("⏱️ Control & Blending"):
                    canny_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["canny_scale"], step=0.01, label="Control Scale: Canny")
                    soft_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["soft_scale"], step=0.01, label="Control Scale: Soft Edge")
                    start_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["start_frac"], step=0.01, label="Start Frac Control")
                    end_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["end_frac"], step=0.01, label="End Frac Control")
                    gate_kind = gr.Dropdown(choices=["cosine", "linear"], value=UI_DEFAULTS["gate_kind"], label="Control Gate Kind")

                    gr.Markdown("---")
                    blend_profile = gr.Dropdown(choices=["decay", "ramp"], value=UI_DEFAULTS["blend_profile"], label="Blend Profile")
                    blend_start = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_start"], step=0.01, label="Blend Start Frac")
                    blend_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_end"], step=0.01, label="Blend End Frac")
                    alpha_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["alpha_end"], step=0.01, label="Blend Alpha End")

                    gr.Markdown("#### Subject Layout Placement (applied from Canny / SoftEdge)")
                    gr.Markdown("Use the drag-and-drop canvas on the right as the main layout control. These fields control the subject position, scale, and rotation directly from the Canny and SoftEdge stage, and are kept for config load/save.")
                    blend_mask_scale = gr.Slider(0.10, 2.50, value=UI_DEFAULTS["blend_mask_scale"], step=0.01, label="Layout Scale", elem_id="blend-mask-scale")
                    blend_mask_offset_x = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_x"], step=0.01, label="Layout Offset X (+ = right)", elem_id="blend-mask-offset-x")
                    blend_mask_offset_y = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_y"], step=0.01, label="Layout Offset Y (+ = down)", elem_id="blend-mask-offset-y")
                    blend_mask_rotation_deg = gr.Slider(-180.0, 180.0, value=UI_DEFAULTS["blend_mask_rotation_deg"], step=1.0, label="Layout Rotation (deg)", elem_id="blend-mask-rotation-deg")

                with gr.Tab("Config JSON"):
                    config_path = gr.Textbox(label="Config JSON Path", placeholder="/kaggle/working/outputs_camouflage_softgradmask/ui_config_20260314_150633.json")
                    btn_load = gr.Button("Load Config JSON")

                with gr.Row():
                    btn_generate = gr.Button("Generate Image", variant="primary", scale=2)
                    btn_save = gr.Button("Save Config", scale=1)

            with gr.Column(scale=4):
                out_img = gr.Image(label="Output Image", type="pil")
                out_score = gr.Textbox(label="Hidden Score", interactive=False)
                status_box = gr.Textbox(label="System Status", interactive=False)

                gr.Markdown("### Calculated Output Metrics")
                out_metrics_report = gr.HTML()
                out_metrics_json = gr.Code(label="Calculated Metrics (JSON)", language="json", interactive=False)

                gr.Markdown("### Interactive Subject Layout")
                btn_refresh_layout = gr.Button("Refresh Interactive Canvas")
                layout_editor = gr.HTML(value=safe_initial_layout_html())

                gr.Markdown("### Debug Views")
                with gr.Row():
                    dbg_bg_original = gr.Image(label="Original BG", type="pil")
                    dbg_input = gr.Image(label="Input Image", type="pil")
                with gr.Row():
                    dbg_canny = gr.Image(label="Canny Hint", type="pil")
                    dbg_soft = gr.Image(label="Soft Hint (Raw)", type="pil")
                with gr.Row():
                    dbg_sil = gr.Image(label="Silhouette Mask", type="pil")
                    dbg_feat = gr.Image(label="Feature Mask", type="pil")
                with gr.Row():
                    dbg_comp = gr.Image(label="Composite HW Mask", type="pil")
                    dbg_soft_for_mask = gr.Image(label="Soft For Mask (After Kernel Gradient)", type="pil")
                with gr.Row():
                    dbg_soft_hw = gr.Image(label="SoftEdge HW Mask", type="pil")
                    dbg_latent_mask = gr.Image(label="Final Latent Mask (after preprocess layout)", type="pil")

                gr.Markdown("### Stage 2 Breakdown Views")
                with gr.Row():
                    dbg_contour_agg = gr.Image(label="Contour Aggregation", type="pil")
                    dbg_region_filling = gr.Image(label="Region Filling", type="pil")
                with gr.Row():
                    dbg_canny_smoothing = gr.Image(label="Smoothing (Canny Path)", type="pil")
                    dbg_aux_support = gr.Image(label="Auxiliary Support S", type="pil")
                with gr.Row():
                    dbg_soft_gray = gr.Image(label="Soft Gray", type="pil")
                    dbg_soft_smoothing = gr.Image(label="Smoothing (Soft Path)", type="pil")
                with gr.Row():
                    dbg_soft_remap = gr.Image(label="Nonlinear Remapping", type="pil")
                    dbg_soft_clip = gr.Image(label="Clip to [0,1]", type="pil")

                gr.Markdown("### Stage 2 Values / Diagnostics")
                dbg_stage2_report = gr.HTML()
                dbg_stage2_json = gr.Code(label="Stage 2 Values (JSON)", language="json", interactive=False)

        all_inputs = [
            path_animal,
            path_bg,
            animal_name,
            prompt_bg,
            prompt_sub,
            negative_prompt,
            steps,
            seed,
            randomize_seed_each_generate,
            bg_strength,
            guidance_bg,
            guidance_sub,
            sub_init_noise,
            couple_mode,
            canny_low,
            canny_high,
            canny_blur_ks,
            use_broken_edges,
            broken_drop_prob,
            mask_mode,
            mask_dilate,
            mask_close_ks,
            mask_blur,
            mask_ring_k,
            mask_ring_blur,
            mask_gamma,
            mask_auto_invert,
            outer_weight,
            feature_weight,
            fill_weight,
            feature_thresh,
            feature_open_ks,
            feature_blur,
            fill_blur,
            use_soft_grad_mask,
            soft_grad_dilation_k,
            soft_grad_blur_k,
            soft_grad_kernel_shape,
            soft_grad_iterations,
            soft_grad_use_skeleton,
            soft_grad_skeleton_stage,
            soft_grad_post_skel_dilate_k,
            soft_mask_blur,
            soft_mask_gamma,
            soft_mask_invert,
            soft_mask_floor,
            soft_mask_ceiling,
            canny_scale,
            soft_scale,
            start_frac,
            end_frac,
            gate_kind,
            blend_profile,
            blend_start,
            blend_end,
            alpha_end,
            auto_attention_compensation,
            attention_scale_ref,
            attention_gamma_ref,
            attention_ceiling_ref,
            attention_blur_ref,
            blend_mask_scale,
            blend_mask_offset_x,
            blend_mask_offset_y,
            blend_mask_rotation_deg,
        ]

        btn_generate.click(
            fn=generate_preview,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[
                seed,
                out_img,
                dbg_bg_original,
                dbg_input,
                dbg_canny,
                dbg_soft,
                dbg_sil,
                dbg_feat,
                dbg_comp,
                dbg_soft_for_mask,
                dbg_soft_hw,
                dbg_latent_mask,
                dbg_contour_agg,
                dbg_region_filling,
                dbg_canny_smoothing,
                dbg_aux_support,
                dbg_soft_gray,
                dbg_soft_smoothing,
                dbg_soft_remap,
                dbg_soft_clip,
                out_score,
                status_box,
                out_metrics_report,
                out_metrics_json,
                dbg_stage2_report,
                dbg_stage2_json,
            ],
        )

        btn_capture_scale_ref.click(
            fn=capture_current_scale_reference,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[attention_scale_ref, status_box],
        )

        btn_save.click(
            fn=save_current_config,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=status_box,
        )

        btn_refresh_layout.click(
            fn=refresh_layout_editor,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[layout_editor, status_box],
        )

        btn_load.click(
            fn=load_config_to_ui,
            inputs=[config_path],
            outputs=all_inputs + [status_box, layout_editor],
        )

    demo.launch(share=True if os.path.exists("/kaggle/working") else True, debug=True)


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
/tmp/ipykernel_58/392154876.py:3077: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f02a572bcba33469c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f02a572bcba33469c9.gradio.live


In [ ]:
# ============================================================
# Kaggle dependency setup
# If you run this in a Kaggle notebook, you can also run this once
# in a separate cell:
# !pip -q install -U diffusers transformers accelerate safetensors opencv-python controlnet-aux gradio scikit-image
# ============================================================

#NEW ENGLISH
# ============================================================
# Camouflage Latent Blending + Dual ControlNet + Gradio UI
# Clean version with JSON config load/save support
# Compatible with legacy ui_config_*.json (params as positional list)
# ============================================================
import os
import re
import math
import json
import time
import base64
import io
import html as html_lib
import contextlib
import shutil
import traceback
import zipfile
from dataclasses import dataclass
from datetime import datetime
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont, ImageOps
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)


# ============================================================
# 0) Small utils
# ============================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

# You can paste image from clipboard 
KAGGLE_ANIMALS_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/animals" 
KAGGLE_BG_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/bg"
DEFAULT_OUT_DIR = "/kaggle/working/outputs_camouflage_softgradmask" if os.path.exists("/kaggle/working") else "./outputs_camouflage_softgradmask"

def list_image_files(path: str) -> List[str]:
    path = str(path or "").strip()
    if not path or not os.path.exists(path):
        return []
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    if os.path.isfile(path):
        return [path] if os.path.splitext(path)[1].lower() in exts else []
    files = []
    for name in sorted(os.listdir(path)):
        full = os.path.join(path, name)
        if os.path.isfile(full) and os.path.splitext(name)[1].lower() in exts:
            files.append(full)
    return files


def resolve_image_path(path_or_dir: Optional[str], fallback_dir: Optional[str] = None) -> Optional[str]:
    candidate = str(path_or_dir or "").strip()
    if candidate:
        if os.path.isfile(candidate):
            return candidate
        if os.path.isdir(candidate):
            files = list_image_files(candidate)
            if files:
                return files[0]
    fallback = str(fallback_dir or "").strip()
    if fallback:
        if os.path.isfile(fallback):
            return fallback
        if os.path.isdir(fallback):
            files = list_image_files(fallback)
            if files:
                return files[0]
    return None


def _odd(k: int) -> int:
    k = int(k)
    return k if k % 2 == 1 else k + 1


def _ensure_odd(k: int, minimum: int = 1) -> int:
    k = max(int(k), int(minimum))
    return k if k % 2 == 1 else k + 1


def _get_kernel(shape: str, k: int) -> np.ndarray:
    k = _ensure_odd(k, 1)
    shape = str(shape).lower().strip()

    if shape == "rect":
        return cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    if shape == "cross":
        return cv2.getStructuringElement(cv2.MORPH_CROSS, (k, k))
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def seed_everything(seed: int) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def slugify(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"[^a-z0-9\-_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "item"


def format_prompt(template: str, animal: str) -> str:
    return (template or "").format(animals=animal, animal=animal)


def save_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def fresh_seed() -> int:
    seed = int.from_bytes(os.urandom(8), "big") % 2147483647
    return seed if seed > 0 else 1


def normalize_0_255(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-6:
        return np.zeros_like(x, dtype=np.uint8)
    x = (x - mn) / (mx - mn)
    return (x * 255.0).clip(0, 255).astype(np.uint8)


def gray01_to_u8(x: np.ndarray) -> np.ndarray:
    return (np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)


def pil_to_data_uri(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    encoded = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/{fmt.lower()};base64,{encoded}"


def make_checkerboard(size: Tuple[int, int], tile: int = 32) -> Image.Image:
    w, h = size
    tile = max(int(tile), 4)
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    c0 = np.array([38, 38, 42], dtype=np.uint8)
    c1 = np.array([58, 58, 64], dtype=np.uint8)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            use_c0 = ((x // tile) + (y // tile)) % 2 == 0
            arr[y : y + tile, x : x + tile] = c0 if use_c0 else c1
    return Image.fromarray(arr, mode="RGB")


def load_rgb_preview_image(path: Optional[str], size: Tuple[int, int]) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved and os.path.exists(resolved):
        return Image.open(resolved).convert("RGB").resize(size, Image.LANCZOS)
    return make_checkerboard(size)


def mask_preview_to_rgba(mask_pil: Image.Image, alpha_scale: float = 0.82) -> Image.Image:
    gray = np.array(mask_pil.convert("L"), dtype=np.uint8)
    edges = cv2.Canny(gray, 32, 128)

    rgba = np.zeros((gray.shape[0], gray.shape[1], 4), dtype=np.uint8)
    rgba[..., 0] = 255
    rgba[..., 1] = 255
    rgba[..., 2] = 255
    rgba[..., 3] = np.clip(gray.astype(np.float32) * float(alpha_scale), 0, 255).astype(np.uint8)

    outline = edges > 0
    rgba[outline, 0] = 255
    rgba[outline, 1] = 96
    rgba[outline, 2] = 96
    rgba[outline, 3] = 255
    return Image.fromarray(rgba, mode="RGBA")


def build_interactive_layout_html(
    bg_img: Image.Image,
    overlay_img: Image.Image,
    width: int,
    height: int,
    init_scale: float,
    init_offset_x: float,
    init_offset_y: float,
    init_rotation_deg: float,
) -> str:
    bg_uri = pil_to_data_uri(bg_img.convert("RGB"), fmt="PNG")
    overlay_uri = pil_to_data_uri(overlay_img.convert("RGBA"), fmt="PNG")
    canvas_w = int(width)
    canvas_h = int(height)
    display_w = min(max(canvas_w, 320), 640)
    display_h = max(int(display_w * canvas_h / max(canvas_w, 1)), 220)

    iframe_id = f"camo-placement-frame-{int(time.time() * 1000)}"

    inner_html = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <style>
    html, body { margin: 0; padding: 0; background: transparent; font-family: ui-sans-serif, system-ui, sans-serif; color: #e5e7eb; overflow: hidden; }
    .wrap { border: 1px solid rgba(255,255,255,0.08); border-radius: 14px; padding: 12px; background: #111827; color: #e5e7eb; box-sizing: border-box; width: 100%; min-height: 100vh; }
    .hint { font-size: 13px; line-height: 1.5; color: #cbd5e1; margin-bottom: 10px; }
    .canvas-wrap { position: relative; width: __DISPLAY_W__px; max-width: 100%; }
    canvas { width: 100%; height: auto; display: block; border-radius: 12px; cursor: grab; background: #0f172a; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.08); }
    canvas.dragging, canvas.resizing, canvas.rotating { cursor: grabbing; }
    .toolbar { margin-top: 10px; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }
    button { border: 0; border-radius: 10px; padding: 8px 12px; background: #1f2937; color: #f9fafb; cursor: pointer; }
    button:hover { background: #374151; }
    .stats { font-size: 12px; color: #cbd5e1; }
    .badge { display: inline-flex; align-items: center; gap: 6px; padding: 6px 10px; border-radius: 999px; background: rgba(255,255,255,0.06); }
    .status { font-size: 12px; color: #93c5fd; margin-top: 8px; }
  </style>
</head>
<body>
  <div class="wrap">
    <div class="hint">Drag the box to move the subject. Drag the blue handle at the bottom-right corner to resize, drag the yellow handle above the box to rotate, and use the mouse wheel to zoom. The Layout sliders on the left panel are synchronized <b>in real time</b>.</div>
    <div class="canvas-wrap">
      <canvas id="canvas" width="__CANVAS_W__" height="__CANVAS_H__"></canvas>
    </div>
    <div class="toolbar">
      <button type="button" data-action="center">Center</button>
      <button type="button" data-action="fit">Fit 0.7</button>
      <button type="button" data-action="reset">Reset</button>
      <span class="badge stats">scale=<span data-key="scale"></span></span>
      <span class="badge stats">offsetX=<span data-key="offsetX"></span></span>
      <span class="badge stats">offsetY=<span data-key="offsetY"></span></span>
      <span class="badge stats">rotation=<span data-key="rotationDeg"></span>°</span>
    </div>
    <div class="status" data-role="status">Canvas state is ready.</div>
  </div>
  <script>
    (() => {
      const canvas = document.getElementById('canvas');
      const ctx = canvas.getContext('2d');
      const statusEl = document.querySelector('[data-role="status"]');
      const bg = new Image();
      const overlay = new Image();
      bg.src = '__BG_URI__';
      overlay.src = '__OVERLAY_URI__';

      const clamp = (v, lo, hi) => Math.min(Math.max(v, lo), hi);
      const wrapDeg = (deg) => {
        let out = Number(deg) || 0;
        while (out > 180) out -= 360;
        while (out <= -180) out += 360;
        return out;
      };
      const wrapRad = (rad) => {
        let out = Number(rad) || 0;
        while (out > Math.PI) out -= Math.PI * 2;
        while (out <= -Math.PI) out += Math.PI * 2;
        return out;
      };
      const nearlyEqual = (a, b, eps=1e-4) => Math.abs((Number(a)||0) - (Number(b)||0)) <= eps;

      const state = {
        scale: clamp(Number('__INIT_SCALE__') || 1, 0.10, 2.50),
        offsetX: clamp(Number('__INIT_OFFSET_X__') || 0, -1.00, 1.00),
        offsetY: clamp(Number('__INIT_OFFSET_Y__') || 0, -1.00, 1.00),
        rotationDeg: wrapDeg(Number('__INIT_ROTATION_DEG__') || 0),
      };

      let parentDoc = null;
      try { parentDoc = window.parent && window.parent.document ? window.parent.document : null; } catch (_) { parentDoc = null; }

      function setStatus(msg) {
        if (statusEl) statusEl.textContent = msg;
      }

      function syncOneSlider(elemId, value) {
        if (!parentDoc) return false;
        const host = parentDoc.getElementById(elemId);
        if (!host) return false;
        const inputs = host.querySelectorAll('input');
        if (!inputs.length) return false;
        const fixed = elemId === 'blend-mask-rotation-deg' ? String(Number(value).toFixed(1)) : String(Number(value).toFixed(4));
        inputs.forEach((input) => {
          if (input.value !== fixed) {
            input.value = fixed;
            input.dispatchEvent(new Event('input', { bubbles: true }));
            input.dispatchEvent(new Event('change', { bubbles: true }));
          }
        });
        return true;
      }

      function readOneSlider(elemId, fallback) {
        if (!parentDoc) return fallback;
        const host = parentDoc.getElementById(elemId);
        if (!host) return fallback;
        const input = host.querySelector('input');
        if (!input) return fallback;
        const val = Number(input.value);
        return Number.isFinite(val) ? val : fallback;
      }

      function pushStateToParent() {
        if (window.parent) {
          window.parent.camoPlacementState = { ...state };
        }
        syncOneSlider('blend-mask-scale', state.scale);
        syncOneSlider('blend-mask-offset-x', state.offsetX);
        syncOneSlider('blend-mask-offset-y', state.offsetY);
        syncOneSlider('blend-mask-rotation-deg', state.rotationDeg);
      }

      function pullStateFromParent(force=false) {
        const nextScale = clamp(readOneSlider('blend-mask-scale', state.scale), 0.10, 2.50);
        const nextOffsetX = clamp(readOneSlider('blend-mask-offset-x', state.offsetX), -1.00, 1.00);
        const nextOffsetY = clamp(readOneSlider('blend-mask-offset-y', state.offsetY), -1.00, 1.00);
        const nextRotation = wrapDeg(readOneSlider('blend-mask-rotation-deg', state.rotationDeg));
        const changed = force || !nearlyEqual(nextScale, state.scale) || !nearlyEqual(nextOffsetX, state.offsetX) || !nearlyEqual(nextOffsetY, state.offsetY) || !nearlyEqual(nextRotation, state.rotationDeg, 0.11);
        if (changed) {
          state.scale = nextScale;
          state.offsetX = nextOffsetX;
          state.offsetY = nextOffsetY;
          state.rotationDeg = nextRotation;
          syncBadges();
          draw();
        }
      }

      let mode = null;
      let dragStart = null;
      let rotateStart = null;
      const handleR = 11;
      const rotateHandleGap = 42;

      function syncBadges() {
        document.querySelector('[data-key="scale"]').textContent = state.scale.toFixed(3);
        document.querySelector('[data-key="offsetX"]').textContent = (state.offsetX).toFixed(3);
        document.querySelector('[data-key="offsetY"]').textContent = state.offsetY.toFixed(3);
        document.querySelector('[data-key="rotationDeg"]').textContent = state.rotationDeg.toFixed(1);
      }

      function getGeometry() {
        const cw = canvas.width;
        const ch = canvas.height;
        const dw = cw * state.scale;
        const dh = ch * state.scale;
        const cx = (cw / 2) + state.offsetX * cw;
        const cy = (ch / 2) + state.offsetY * ch;
        const rad = state.rotationDeg * Math.PI / 180.0;
        const cos = Math.cos(rad);
        const sin = Math.sin(rad);

        const localToWorld = (lx, ly) => ({
          x: cx + lx * cos - ly * sin,
          y: cy + lx * sin + ly * cos,
        });
        const worldToLocal = (px, py) => {
          const dx = px - cx;
          const dy = py - cy;
          return {
            x: dx * cos + dy * sin,
            y: -dx * sin + dy * cos,
          };
        };

        const halfW = dw / 2;
        const halfH = dh / 2;
        const corners = [
          localToWorld(-halfW, -halfH),
          localToWorld( halfW, -halfH),
          localToWorld( halfW,  halfH),
          localToWorld(-halfW,  halfH),
        ];
        const resizeHandle = localToWorld(halfW, halfH);
        const rotateHandle = localToWorld(0, -halfH - rotateHandleGap);

        return {
          cw, ch, dw, dh, cx, cy, rad, halfW, halfH,
          corners, resizeHandle, rotateHandle, localToWorld, worldToLocal,
        };
      }

      function drawGuides(geom) {
        ctx.save();
        ctx.strokeStyle = 'rgba(255,255,255,0.15)';
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(geom.cw / 2, 0);
        ctx.lineTo(geom.cw / 2, geom.ch);
        ctx.moveTo(0, geom.ch / 2);
        ctx.lineTo(geom.cw, geom.ch / 2);
        ctx.stroke();

        ctx.setLineDash([8, 8]);
        ctx.strokeStyle = 'rgba(255,255,255,0.70)';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(geom.corners[0].x, geom.corners[0].y);
        for (let i = 1; i < geom.corners.length; i++) ctx.lineTo(geom.corners[i].x, geom.corners[i].y);
        ctx.closePath();
        ctx.stroke();
        ctx.setLineDash([]);

        const topMid = geom.localToWorld(0, -geom.halfH);
        ctx.beginPath();
        ctx.moveTo(topMid.x, topMid.y);
        ctx.lineTo(geom.rotateHandle.x, geom.rotateHandle.y);
        ctx.strokeStyle = 'rgba(251,191,36,0.9)';
        ctx.lineWidth = 2;
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#60a5fa';
        ctx.arc(geom.resizeHandle.x, geom.resizeHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#fbbf24';
        ctx.arc(geom.rotateHandle.x, geom.rotateHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();
        ctx.restore();
      }

      function draw() {
        const geom = getGeometry();
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        if (bg.complete) ctx.drawImage(bg, 0, 0, canvas.width, canvas.height);
        if (overlay.complete) {
          ctx.save();
          ctx.translate(geom.cx, geom.cy);
          ctx.rotate(geom.rad);
          ctx.drawImage(overlay, -geom.dw / 2, -geom.dh / 2, geom.dw, geom.dh);
          ctx.restore();
        }
        drawGuides(geom);
      }

      function localPoint(evt) {
        const rect = canvas.getBoundingClientRect();
        return {
          x: (evt.clientX - rect.left) * (canvas.width / rect.width),
          y: (evt.clientY - rect.top) * (canvas.height / rect.height),
        };
      }

      function startAction(evt) {
        evt.preventDefault();
        const p = localPoint(evt);
        const geom = getGeometry();
        const distResize = Math.hypot(p.x - geom.resizeHandle.x, p.y - geom.resizeHandle.y);
        const distRotate = Math.hypot(p.x - geom.rotateHandle.x, p.y - geom.rotateHandle.y);
        const local = geom.worldToLocal(p.x, p.y);
        const inside = Math.abs(local.x) <= geom.halfW && Math.abs(local.y) <= geom.halfH;

        if (distRotate <= handleR * 1.6) {
          mode = 'rotate';
          rotateStart = {
            baseRotation: state.rotationDeg,
            baseAngle: Math.atan2(p.y - geom.cy, p.x - geom.cx),
          };
          canvas.classList.add('rotating');
        } else if (distResize <= handleR * 1.6) {
          mode = 'resize';
          canvas.classList.add('resizing');
        } else if (inside) {
          mode = 'drag';
          dragStart = {
            startX: p.x,
            startY: p.y,
            baseOffsetX: state.offsetX,
            baseOffsetY: state.offsetY,
          };
          canvas.classList.add('dragging');
        }
      }

      function afterCanvasEdit(message) {
        syncBadges();
        pushStateToParent();
        draw();
        setStatus(message || 'Layout has been synchronized with the left panel.');
      }

      function moveAction(evt) {
        if (!mode) return;
        const p = localPoint(evt);
        const geom = getGeometry();

        if (mode === 'drag') {
          const dx = p.x - dragStart.startX;
          const dy = p.y - dragStart.startY;
          state.offsetX = clamp(dragStart.baseOffsetX + dx / canvas.width, -1.00, 1.00);
          state.offsetY = clamp(dragStart.baseOffsetY + dy / canvas.height, -1.00, 1.00);
        } else if (mode === 'resize') {
          const local = geom.worldToLocal(p.x, p.y);
          const sx = Math.abs(local.x) / (canvas.width / 2);
          const sy = Math.abs(local.y) / (canvas.height / 2);
          state.scale = clamp(Math.max(sx, sy, 0.10), 0.10, 2.50);
        } else if (mode === 'rotate') {
          const angle = Math.atan2(p.y - geom.cy, p.x - geom.cx);
          const delta = wrapRad(angle - rotateStart.baseAngle);
          state.rotationDeg = wrapDeg(rotateStart.baseRotation - delta * 180.0 / Math.PI);
        }

        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }

      function endAction() {
        mode = null;
        dragStart = null;
        rotateStart = null;
        canvas.classList.remove('dragging');
        canvas.classList.remove('resizing');
        canvas.classList.remove('rotating');
      }

      canvas.addEventListener('mousedown', startAction);
      window.addEventListener('mousemove', moveAction);
      window.addEventListener('mouseup', endAction);
      canvas.addEventListener('mouseleave', endAction);
      canvas.addEventListener('wheel', (evt) => {
        evt.preventDefault();
        const factor = evt.deltaY < 0 ? 1.05 : 0.95;
        state.scale = clamp(state.scale * factor, 0.10, 2.50);
        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }, { passive: false });

      document.querySelector('[data-action="center"]').addEventListener('click', () => {
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        afterCanvasEdit('Centered and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="fit"]').addEventListener('click', () => {
        state.scale = 0.70;
        afterCanvasEdit('Set scale to 0.7 and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="reset"]').addEventListener('click', () => {
        state.scale = 1.0;
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        state.rotationDeg = 0.0;
        afterCanvasEdit('Reset and synchronized with the layout sliders.');
      });

      bg.onload = draw;
      overlay.onload = draw;
      syncBadges();
      pushStateToParent();
      draw();
      setStatus(parentDoc ? 'Canvas is ready and synchronized in real time with the left panel.' : 'Canvas is ready. The parent panel is not accessible, so only internal synchronization is available.');
      setInterval(() => {
        if (!mode) pullStateFromParent(false);
      }, 120);
    })();
  </script>
</body>
</html>
"""

    inner_html = (
        inner_html.replace("__DISPLAY_W__", str(display_w))
        .replace("__CANVAS_W__", str(canvas_w))
        .replace("__CANVAS_H__", str(canvas_h))
        .replace("__BG_URI__", bg_uri)
        .replace("__OVERLAY_URI__", overlay_uri)
        .replace("__INIT_SCALE__", f"{float(init_scale):.6f}")
        .replace("__INIT_OFFSET_X__", f"{float(init_offset_x):.6f}")
        .replace("__INIT_OFFSET_Y__", f"{float(init_offset_y):.6f}")
        .replace("__INIT_ROTATION_DEG__", f"{float(init_rotation_deg):.6f}")
    )
    srcdoc = html_lib.escape(inner_html, quote=True)
    return f"""<div style='width:100%'>
  <iframe id="{iframe_id}" srcdoc="{srcdoc}" style="width:100%; height:{display_h + 150}px; border:0; background:transparent; border-radius:14px;"></iframe>
</div>"""


# ============================================================
# 1) Core config
# ============================================================
CFG: Dict[str, Any] = dict(
    out_dir=DEFAULT_OUT_DIR,
    base_model_id="runwayml/stable-diffusion-v1-5",
    canny_cn_id="lllyasviel/sd-controlnet-canny",
    soft_cn_id="lllyasviel/control_v11p_sd15_softedge",
    hed_annotator_id="lllyasviel/Annotators",
    height=512,
    width=512,
    steps=57,
    batch=1,
    seed=10,
    guidance_scale=2.0,
    guidance_bg=1.2,
    guidance_sub=2.3,
    guess_mode=True,
    canny_low=60,
    canny_high=160,
    canny_blur_ks=5,
    use_broken_edges=True,
    broken_drop_prob=0.30,
    mask_dilate=1,
    mask_close_ks=1,
    mask_close_iter=1,
    mask_blur=1,
    mask_mode="softedge",  # silhouette / outline / composite / softedge
    mask_ring_k=1,
    mask_ring_blur=1,
    mask_gamma=0.1,
    mask_auto_invert=True,
    outer_weight=1.00,
    feature_weight=0.45,
    fill_weight=0.10,
    feature_thresh=32,
    feature_open_ks=3,
    feature_blur=5,
    fill_blur=15,
    use_soft_grad_mask=False,
    soft_grad_dilation_k=3,
    soft_grad_blur_k=3,
    soft_grad_kernel_shape="ellipse",
    soft_grad_iterations=1,
    soft_grad_use_skeleton=False,
    soft_grad_skeleton_stage="post",
    soft_grad_skeleton_thresh="otsu",
    soft_grad_post_skel_dilate_k=0,
    soft_mask_blur=11,
    soft_mask_gamma=0.5,
    soft_mask_invert=False,
    soft_mask_floor=0.05,
    soft_mask_ceiling=0.90,
    debug_mask=False,
    save_mask_images=True,
    control_scales=(0.02, 1.5),
    use_weights="A",
    start_frac_control=0.0,
    end_frac_control=0.84,
    control_gate_kind="cosine",
    blend_profile="decay",
    blend_start_frac=0.05,
    blend_end_frac=0.56,
    blend_alpha_end=0.54,
    auto_attention_compensation=False,
    attention_scale_ref=0.32,
    attention_gamma_ref=0.5,
    attention_ceiling_ref=0.90,
    attention_blur_ref=11,
    blend_mask_scale=1.0,
    blend_mask_offset_x=0.0,
    blend_mask_offset_y=0.0,
    blend_mask_rotation_deg=0.0,
    randomize_seed_each_generate=False,
    bg_image_path=KAGGLE_BG_DIR,
    bg_strength=0.52,
    sub_init_noise=0.0,
    couple_mode="bg_only",
    prompt_bg=(
        ""
    ),
    prompt_subject_template=(
        "a hidden {animal} silhouette seamlessly integrated into the background, subtle facial cues, optical illusion, camouflage"
    ),
    negative=(
        "sticker, pasted object, sharp outline, cartoon, text, watermark, logo, "
        "extra limbs, deformed, low quality, high contrast foreground object"
    ),
    seed_stride=1000,
    log_every=10,
    show_preview=True,
)


def build_cfg(raw_cfg: Dict[str, Any]) -> SimpleNamespace:
    cfg = SimpleNamespace(**raw_cfg)

    if not getattr(cfg, "device", None):
        cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    if not getattr(cfg, "dtype", None):
        cfg.dtype = "float16" if cfg.device.startswith("cuda") and torch.cuda.is_available() else "float32"

    cfg.torch_dtype = torch.float16 if cfg.dtype == "float16" else torch.float32
    ensure_dir(cfg.out_dir)

    odd_fields = [
        "canny_blur_ks",
        "mask_dilate",
        "mask_close_ks",
        "mask_blur",
        "mask_ring_k",
        "mask_ring_blur",
        "feature_open_ks",
        "feature_blur",
        "fill_blur",
        "soft_mask_blur",
        "attention_blur_ref",
        "soft_grad_dilation_k",
        "soft_grad_blur_k",
        "soft_grad_post_skel_dilate_k",
    ]
    for name in odd_fields:
        value = getattr(cfg, name, 0)
        setattr(cfg, name, _odd(value) if value and value > 1 else value)

    cfg.control_scales = tuple(getattr(cfg, "control_scales", (1.0, 1.0)))
    cfg.seed_stride = int(getattr(cfg, "seed_stride", 1000))
    cfg.log_every = int(getattr(cfg, "log_every", 10))
    cfg.show_preview = bool(getattr(cfg, "show_preview", False))
    cfg.debug_mask = bool(getattr(cfg, "debug_mask", False))
    cfg.save_mask_images = bool(getattr(cfg, "save_mask_images", True))
    cfg.use_broken_edges = bool(getattr(cfg, "use_broken_edges", False))
    cfg.broken_drop_prob = float(getattr(cfg, "broken_drop_prob", 0.2))

    cfg.mask_mode = str(getattr(cfg, "mask_mode", "softedge")).lower().strip()
    cfg.mask_gamma = float(getattr(cfg, "mask_gamma", 2.0))
    cfg.mask_auto_invert = bool(getattr(cfg, "mask_auto_invert", True))

    cfg.feature_weight = float(getattr(cfg, "feature_weight", 0.45))
    cfg.fill_weight = float(getattr(cfg, "fill_weight", 0.10))
    cfg.outer_weight = float(getattr(cfg, "outer_weight", 1.00))
    cfg.feature_thresh = int(getattr(cfg, "feature_thresh", 32))
    cfg.feature_open_ks = int(getattr(cfg, "feature_open_ks", 3))
    cfg.feature_blur = int(getattr(cfg, "feature_blur", 5))
    cfg.fill_blur = int(getattr(cfg, "fill_blur", 15))

    cfg.soft_mask_blur = int(getattr(cfg, "soft_mask_blur", 5))
    cfg.soft_mask_gamma = float(getattr(cfg, "soft_mask_gamma", 1.4))
    cfg.soft_mask_invert = bool(getattr(cfg, "soft_mask_invert", False))
    cfg.soft_mask_floor = float(getattr(cfg, "soft_mask_floor", 0.0))
    cfg.soft_mask_ceiling = float(getattr(cfg, "soft_mask_ceiling", 1.0))

    cfg.auto_attention_compensation = bool(getattr(cfg, "auto_attention_compensation", False))
    cfg.attention_scale_ref = float(getattr(cfg, "attention_scale_ref", 0.32))
    cfg.attention_gamma_ref = float(getattr(cfg, "attention_gamma_ref", cfg.soft_mask_gamma))
    cfg.attention_ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", cfg.soft_mask_ceiling))
    cfg.attention_blur_ref = int(getattr(cfg, "attention_blur_ref", cfg.soft_mask_blur))

    cfg.use_soft_grad_mask = bool(getattr(cfg, "use_soft_grad_mask", True))
    cfg.soft_grad_dilation_k = int(getattr(cfg, "soft_grad_dilation_k", 5))
    cfg.soft_grad_blur_k = int(getattr(cfg, "soft_grad_blur_k", 5))
    cfg.soft_grad_kernel_shape = str(getattr(cfg, "soft_grad_kernel_shape", "ellipse")).lower().strip()
    cfg.soft_grad_iterations = int(getattr(cfg, "soft_grad_iterations", 1))
    cfg.soft_grad_use_skeleton = bool(getattr(cfg, "soft_grad_use_skeleton", False))
    cfg.soft_grad_skeleton_stage = str(getattr(cfg, "soft_grad_skeleton_stage", "post")).lower().strip()
    cfg.soft_grad_skeleton_thresh = getattr(cfg, "soft_grad_skeleton_thresh", "otsu")
    cfg.soft_grad_post_skel_dilate_k = int(getattr(cfg, "soft_grad_post_skel_dilate_k", 0))

    cfg.start_frac_control = float(getattr(cfg, "start_frac_control", 0.0))
    cfg.end_frac_control = float(getattr(cfg, "end_frac_control", 1.0))
    cfg.control_gate_kind = str(getattr(cfg, "control_gate_kind", "cosine")).lower().strip()
    cfg.blend_profile = str(getattr(cfg, "blend_profile", "decay")).lower().strip()
    cfg.blend_start_frac = float(getattr(cfg, "blend_start_frac", 0.05))
    cfg.blend_end_frac = float(getattr(cfg, "blend_end_frac", 0.70))
    cfg.blend_alpha_end = float(getattr(cfg, "blend_alpha_end", 0.45))
    cfg.blend_mask_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
    cfg.blend_mask_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
    cfg.blend_mask_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
    cfg.blend_mask_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
    cfg.randomize_seed_each_generate = bool(getattr(cfg, "randomize_seed_each_generate", False))

    cfg.bg_image_path = getattr(cfg, "bg_image_path", None)
    cfg.bg_strength = float(getattr(cfg, "bg_strength", 0.85))
    cfg.sub_init_noise = float(getattr(cfg, "sub_init_noise", 0.0))
    cfg.couple_mode = str(getattr(cfg, "couple_mode", "none")).lower().strip()
    cfg.use_weights = str(getattr(cfg, "use_weights", "A")).upper().strip()

    cfg.guidance_scale = float(getattr(cfg, "guidance_scale", 1.0))
    cfg.guidance_bg = float(getattr(cfg, "guidance_bg", cfg.guidance_scale))
    cfg.guidance_sub = float(getattr(cfg, "guidance_sub", cfg.guidance_scale))
    cfg.guess_mode = bool(getattr(cfg, "guess_mode", True))
    return cfg


# ============================================================
# 2) JSON UI config compatibility layer
# ============================================================
UI_FIELD_NAMES: List[str] = [
    "path_animal",
    "path_bg",
    "animal_name",
    "prompt_bg",
    "prompt_sub",
    "negative_prompt",
    "steps",
    "seed",
    "randomize_seed_each_generate",
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "couple_mode",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "use_broken_edges",
    "broken_drop_prob",
    "mask_mode",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "mask_gamma",
    "mask_auto_invert",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "use_soft_grad_mask",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_kernel_shape",
    "soft_grad_iterations",
    "soft_grad_use_skeleton",
    "soft_grad_skeleton_stage",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "soft_mask_gamma",
    "soft_mask_invert",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "gate_kind",
    "blend_profile",
    "blend_start",
    "blend_end",
    "alpha_end",
    "auto_attention_compensation",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "attention_blur_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
]


UI_DEFAULTS: Dict[str, Any] = {
    "path_animal": KAGGLE_ANIMALS_DIR,
    "path_bg": KAGGLE_BG_DIR,
    "animal_name": "cat",
    "prompt_bg": CFG["prompt_bg"],
    "prompt_sub": CFG["prompt_subject_template"],
    "negative_prompt": CFG["negative"],
    "steps": 57,
    "seed": 10,
    "randomize_seed_each_generate": False,
    "bg_strength": 0.52,
    "guidance_bg": 1.2,
    "guidance_sub": 2.3,
    "sub_init_noise": 0.0,
    "couple_mode": "bg_only",
    "canny_low": 60,
    "canny_high": 160,
    "canny_blur_ks": 5,
    "use_broken_edges": True,
    "broken_drop_prob": 0.3,
    "mask_mode": "softedge",
    "mask_dilate": 1,
    "mask_close_ks": 1,
    "mask_blur": 1,
    "mask_ring_k": 1,
    "mask_ring_blur": 1,
    "mask_gamma": 0.1,
    "mask_auto_invert": True,
    "outer_weight": 1.0,
    "feature_weight": 0.45,
    "fill_weight": 0.10,
    "feature_thresh": 32,
    "feature_open_ks": 3,
    "feature_blur": 5,
    "fill_blur": 15,
    "use_soft_grad_mask": False,
    "soft_grad_dilation_k": 3,
    "soft_grad_blur_k": 3,
    "soft_grad_kernel_shape": "ellipse",
    "soft_grad_iterations": 1,
    "soft_grad_use_skeleton": False,
    "soft_grad_skeleton_stage": "post",
    "soft_grad_post_skel_dilate_k": 0,
    "soft_mask_blur": 11,
    "soft_mask_gamma": 0.5,
    "soft_mask_invert": False,
    "soft_mask_floor": 0.05,
    "soft_mask_ceiling": 0.90,
    "canny_scale": 0.02,
    "soft_scale": 1.5,
    "start_frac": 0.0,
    "end_frac": 0.84,
    "gate_kind": "cosine",
    "blend_profile": "decay",
    "blend_start": 0.05,
    "blend_end": 0.56,
    "alpha_end": 0.54,
    "auto_attention_compensation": False,
    "attention_scale_ref": 0.32,
    "attention_gamma_ref": 0.5,
    "attention_ceiling_ref": 0.90,
    "attention_blur_ref": 11,
    "blend_mask_scale": 1.0,
    "blend_mask_offset_x": 0.0,
    "blend_mask_offset_y": 0.0,
    "blend_mask_rotation_deg": 0.0,
}


LAYOUT_ALIAS_TO_UI_FIELD = {
    "layout_scale": "blend_mask_scale",
    "layout_offset_x": "blend_mask_offset_x",
    "layout_offset_y": "blend_mask_offset_y",
    "layout_rotation_deg": "blend_mask_rotation_deg",
}

NUMERIC_INT_FIELDS = {
    "steps",
    "seed",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_iterations",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "attention_blur_ref",
}

NUMERIC_FLOAT_FIELDS = {
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "broken_drop_prob",
    "mask_gamma",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "soft_mask_gamma",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "blend_start",
    "blend_end",
    "alpha_end",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
}

BOOL_FIELDS = {
    "use_broken_edges",
    "mask_auto_invert",
    "use_soft_grad_mask",
    "soft_grad_use_skeleton",
    "soft_mask_invert",
    "auto_attention_compensation",
    "randomize_seed_each_generate",
}


def coerce_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y", "on"}
    return bool(value)


def coerce_ui_value(field: str, value: Any) -> Any:
    if field in BOOL_FIELDS:
        return coerce_bool(value)
    if field in NUMERIC_INT_FIELDS:
        return int(value)
    if field in NUMERIC_FLOAT_FIELDS:
        return float(value)
    return value


def normalize_ui_params(raw_params: Any) -> Dict[str, Any]:
    if isinstance(raw_params, list):
        params = dict(zip(UI_FIELD_NAMES, raw_params))
    elif isinstance(raw_params, dict):
        params = dict(raw_params)
        for alias_name, ui_name in LAYOUT_ALIAS_TO_UI_FIELD.items():
            if alias_name in params and ui_name not in params:
                params[ui_name] = params[alias_name]
    else:
        raise ValueError("Unsupported config format: 'params' must be a list or dict.")

    merged = dict(UI_DEFAULTS)
    merged.update(params)
    return {key: coerce_ui_value(key, merged[key]) for key in UI_FIELD_NAMES}


def load_ui_params_from_json(config_path: str) -> Dict[str, Any]:
    with open(config_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict) and "params" in payload:
        return normalize_ui_params(payload["params"])

    if isinstance(payload, dict):
        return normalize_ui_params(payload)

    raise ValueError("JSON config must be a dict or contain a 'params' field.")


def build_ui_dict_from_args(args: Sequence[Any]) -> Dict[str, Any]:
    if len(args) != len(UI_FIELD_NAMES):
        raise ValueError(f"Expected {len(UI_FIELD_NAMES)} UI values, got {len(args)}")
    raw = dict(zip(UI_FIELD_NAMES, args))
    return {key: coerce_ui_value(key, value) for key, value in raw.items()}


# ============================================================
# 3) Job definition
# ============================================================
@dataclass
class AnimalJob:
    image_path: str
    animals: str
    name: Optional[str] = None
    seed: Optional[int] = None
    subject_template: Optional[str] = None
    prompt_bg: Optional[str] = None
    negative: Optional[str] = None
    bg_image_path: Optional[str] = None
    bg_strength: Optional[float] = None


# ============================================================
# 4) Skeleton + kernel-gradient hint
# ============================================================
def skeletonize_u8(img_u8: np.ndarray, thresh: str | int = "otsu") -> np.ndarray:
    if img_u8.dtype != np.uint8:
        img_u8 = np.clip(img_u8, 0, 255).astype(np.uint8)

    if thresh == "otsu":
        _, bin_u8 = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, bin_u8 = cv2.threshold(img_u8, int(thresh), 255, cv2.THRESH_BINARY)

    mask = bin_u8 > 0

    try:
        from skimage.morphology import skeletonize

        return skeletonize(mask).astype(np.uint8) * 255
    except Exception:
        try:
            return cv2.ximgproc.thinning(mask.astype(np.uint8) * 255).astype(np.uint8)
        except Exception as e:
            raise RuntimeError(
                "Skeletonization failed: scikit-image and/or cv2.ximgproc is missing. "
                "Install: pip install scikit-image or opencv-contrib-python."
            ) from e


def kernel_gradient_hint(
    edges_2d: np.ndarray,
    dilation_k: int = 3,
    blur_k: int = 3,
    kernel_shape: str = "ellipse",
    iterations: int = 1,
    use_skeleton: bool = True,
    skeleton_stage: str = "pre",
    skeleton_thresh: str | int = "otsu",
    post_skel_dilate_k: int = 0,
) -> Image.Image:
    if edges_2d.dtype != np.uint8:
        edges_2d = np.clip(edges_2d, 0, 255).astype(np.uint8)

    x = edges_2d
    if use_skeleton and skeleton_stage.lower() == "pre":
        x = skeletonize_u8(x, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            x = cv2.dilate(x, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad = cv2.morphologyEx(
        x,
        cv2.MORPH_GRADIENT,
        _get_kernel(kernel_shape, dilation_k),
        iterations=iterations,
    )

    if use_skeleton and skeleton_stage.lower() == "post":
        grad = skeletonize_u8(grad, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            grad = cv2.dilate(grad, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad_blur = cv2.GaussianBlur(grad, (_ensure_odd(blur_k, 3), _ensure_odd(blur_k, 3)), 0)
    return Image.fromarray(grad_blur).convert("RGB")


# ============================================================
# 5) Image + edge + mask helpers
# ============================================================
def load_image(path: str, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved is None:
        raise FileNotFoundError(f"No valid image file or image folder found: {path}")
    img = Image.open(resolved).convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    return img


def pil_to_cv_bgr(img_pil: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def gray_to_pil(gray: np.ndarray, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    img = Image.fromarray(normalize_0_255(gray))
    if size is not None:
        img = img.resize(size, Image.NEAREST)
    return img


def rgb_uint8_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8))


def canny_clean_norm(img_bgr: np.ndarray, low: int = 80, high: int = 160, blur_ks: int = 3) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if blur_ks and blur_ks > 1:
        gray = cv2.GaussianBlur(gray, (blur_ks, blur_ks), 0)
    edges = cv2.Canny(gray, low, high)
    hint = np.stack([edges, edges, edges], axis=-1)
    return normalize_0_255(hint)


def canny_broken(edges_3ch: np.ndarray, drop_prob: float = 0.5, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    edges = edges_3ch[..., 0] > 0
    keep = rng.random(edges.shape) > drop_prob
    out = (edges & keep).astype(np.uint8) * 255
    return np.stack([out, out, out], axis=-1)


def edges_to_silhouette_mask(
    edges_3ch: np.ndarray,
    dilate: int = 17,
    close_ks: int = 17,
    close_iter: int = 1,
    blur: int = 41,
) -> np.ndarray:
    edges = (edges_3ch[..., 0] > 0).astype(np.uint8) * 255

    if dilate and dilate > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(dilate), _odd(dilate)))
        edges = cv2.dilate(edges, kernel, iterations=1)

    if close_ks and close_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(close_ks), _odd(close_ks)))
        edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=max(1, int(close_iter)))

    inv = cv2.bitwise_not(edges)
    flood = inv.copy()
    h, w = edges.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)

    fg = (flood > 0).astype(np.uint8) * 255
    if blur and blur > 1:
        fg = cv2.GaussianBlur(fg, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(fg).astype(np.float32) / 255.0


def extract_internal_feature_mask(
    edges_3ch: np.ndarray,
    sil_hw: np.ndarray,
    thresh: int = 32,
    open_ks: int = 3,
    blur: int = 5,
) -> np.ndarray:
    edges = edges_3ch[..., 0].astype(np.uint8)
    sil_bin = (sil_hw > 0.35).astype(np.uint8) * 255

    if sil_bin.max() == 0:
        return np.zeros_like(sil_hw, dtype=np.float32)

    feat = np.where(sil_bin > 0, edges, 0).astype(np.uint8)
    inner = sil_bin.copy()

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        inner = cv2.erode(inner, kernel, iterations=2)

    feat = np.where(inner > 0, feat, 0).astype(np.uint8)
    feat = (feat > thresh).astype(np.uint8) * 255

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        feat = cv2.morphologyEx(feat, cv2.MORPH_OPEN, kernel, iterations=1)

    if blur and blur > 1:
        feat = cv2.GaussianBlur(feat, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(feat).astype(np.float32) / 255.0


def build_composite_mask_hw(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    outer_ring_k: int = 11,
    outer_ring_blur: int = 7,
    outer_weight: float = 1.0,
    feature_weight: float = 0.45,
    fill_weight: float = 0.10,
    fill_blur: int = 15,
) -> np.ndarray:
    sil_u8 = gray01_to_u8(sil_hw)

    if outer_ring_k and outer_ring_k > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(outer_ring_k), _odd(outer_ring_k)))
        dil = cv2.dilate(sil_u8, kernel, iterations=1)
        ero = cv2.erode(sil_u8, kernel, iterations=1)
        ring = cv2.subtract(dil, ero)
    else:
        ring = sil_u8.copy()

    if outer_ring_blur and outer_ring_blur > 1:
        ring = cv2.GaussianBlur(ring, (_odd(outer_ring_blur), _odd(outer_ring_blur)), 0)

    ring = normalize_0_255(ring).astype(np.float32) / 255.0

    fill = sil_u8.copy()
    if fill_blur and fill_blur > 1:
        fill = cv2.GaussianBlur(fill, (_odd(fill_blur), _odd(fill_blur)), 0)
    fill = normalize_0_255(fill).astype(np.float32) / 255.0

    feat = np.clip(feature_hw.astype(np.float32), 0.0, 1.0)
    out = outer_weight * ring + feature_weight * feat + fill_weight * fill
    return np.clip(out, 0.0, 1.0)


def softedge_to_mask_hw(
    soft_hint: np.ndarray,
    blur_ks: int = 0,
    gamma: float = 1.0,
    invert: bool = False,
    floor: float = 0.0,
    ceiling: float = 1.0,
) -> np.ndarray:
    gray = (
        cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
        if soft_hint.ndim == 3
        else soft_hint.astype(np.uint8)
    )
    mask = gray.astype(np.float32) / 255.0

    if blur_ks and blur_ks > 1:
        mask = cv2.GaussianBlur(mask, (_odd(blur_ks), _odd(blur_ks)), 0)

    if invert:
        mask = 1.0 - mask

    mask = np.clip(mask, 0.0, 1.0)
    if gamma != 1.0:
        mask = mask ** gamma

    floor = float(np.clip(floor, 0.0, 1.0))
    ceiling = float(np.clip(ceiling, 0.0, 1.0))
    if ceiling > floor:
        mask = np.clip((mask - floor) / (ceiling - floor), 0.0, 1.0)

    return np.clip(mask, 0.0, 1.0).astype(np.float32)


# ============================================================
# 6) Torch helpers
# ============================================================
def to_torch_image_hint(hint_uint8: np.ndarray, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = torch.from_numpy(hint_uint8).to(device=device).float() / 255.0
    return x.permute(2, 0, 1).unsqueeze(0).contiguous().to(dtype=dtype)


def make_latent_mask(mask_hw: np.ndarray, latent_h: int, latent_w: int, device: str, dtype: torch.dtype) -> torch.Tensor:
    mask = torch.from_numpy(mask_hw).float().unsqueeze(0).unsqueeze(0)
    mask = F.interpolate(mask, size=(latent_h, latent_w), mode="bilinear", align_corners=False)
    return mask.to(device=device, dtype=dtype)


def _dilate_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.max_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def _erode_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return -F.max_pool2d(-x, kernel_size=k, stride=1, padding=k // 2)


def _blur_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.avg_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def save_mask_img(mask_1x1hw: torch.Tensor, out_path: str, width: int, height: int) -> None:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(arr).resize((width, height), Image.NEAREST).save(out_path)


def mask_tensor_to_pil(mask_1x1hw: torch.Tensor, width: int, height: int) -> Image.Image:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr).resize((width, height), Image.NEAREST)


def has_layout_transform(cfg: SimpleNamespace) -> bool:
    return (
        abs(float(getattr(cfg, "blend_mask_scale", 1.0)) - 1.0) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_x", 0.0))) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_y", 0.0))) > 1e-6
    )


def transform_frame(
    arr: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
    border_value: float | Tuple[float, ...] = 0.0,
) -> np.ndarray:
    src = np.asarray(arr)
    h, w = src.shape[:2]
    scale = max(float(scale), 1e-3)
    # UI/canvas convention: +offset_x => move right, +offset_y => move down.
    # Because cv2.warpAffine samples via an inverse map, the translation terms here
    # need the opposite sign so the rendered output moves in the same direction as
    # the interactive canvas preview.
    tx = -float(offset_x_frac) * float(w)
    ty = -float(offset_y_frac) * float(h)
    theta = math.radians(float(rotation_deg))
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    cx = (float(w) - 1.0) * 0.5
    cy = (float(h) - 1.0) * 0.5

    t_neg = np.array([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [0.0, 0.0, 1.0]], dtype=np.float32)
    s_mat = np.array([[scale, 0.0, 0.0], [0.0, scale, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    r_mat = np.array([[cos_t, -sin_t, 0.0], [sin_t, cos_t, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    t_pos = np.array([[1.0, 0.0, cx + tx], [0.0, 1.0, cy + ty], [0.0, 0.0, 1.0]], dtype=np.float32)

    forward = t_pos @ r_mat @ s_mat @ t_neg
    m = np.linalg.inv(forward)[:2].astype(np.float32)

    out = cv2.warpAffine(
        src,
        m,
        (w, h),
        flags=interp,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )
    return out


def transform_mask_hw(

    mask_hw: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(mask_hw, dtype=np.float32),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0.0,
    )
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def transform_hint_uint8(
    hint: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(hint, dtype=np.uint8),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0,
    )
    return np.clip(out, 0, 255).astype(np.uint8)


def compute_layout_scale_eff(mask_hw: np.ndarray, threshold: float = 0.35) -> float:
    mask = np.asarray(mask_hw, dtype=np.float32)
    if mask.ndim != 2 or mask.size == 0:
        return 0.0

    total = float(mask.shape[0] * mask.shape[1])
    area = float((mask > float(threshold)).sum())
    if area <= 0.0:
        area = float(np.clip(mask, 0.0, 1.0).sum())
    if total <= 0.0 or area <= 0.0:
        return 0.0
    return float(math.sqrt(area / total))


def resolve_attention_compensation(cfg: SimpleNamespace, sil_hw: np.ndarray) -> Dict[str, Any]:
    scale_eff = compute_layout_scale_eff(sil_hw, threshold=0.35)
    scale_ref = max(float(getattr(cfg, "attention_scale_ref", 0.32)), 1e-4)
    ratio = max(scale_eff / scale_ref, 1e-4) if scale_eff > 0.0 else 1.0

    gamma_ref = float(getattr(cfg, "attention_gamma_ref", getattr(cfg, "soft_mask_gamma", 1.0)))
    ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", getattr(cfg, "soft_mask_ceiling", 1.0)))
    blur_ref = int(getattr(cfg, "attention_blur_ref", getattr(cfg, "soft_mask_blur", 5)))

    effective_gamma = float(getattr(cfg, "soft_mask_gamma", gamma_ref))
    effective_ceiling = float(getattr(cfg, "soft_mask_ceiling", ceiling_ref))
    effective_blur = int(getattr(cfg, "soft_mask_blur", blur_ref))

    enabled = bool(getattr(cfg, "auto_attention_compensation", False))
    applies = enabled and str(getattr(cfg, "mask_mode", "softedge")).lower() == "softedge"

    if applies:
        effective_gamma = float(np.clip(gamma_ref * (ratio ** 0.70), 0.25, 0.95))
        effective_ceiling = float(np.clip(ceiling_ref * (ratio ** 0.25), 0.70, 0.97))
        blur_val = int(round(blur_ref * (ratio ** 0.50)))
        effective_blur = int(np.clip(_ensure_odd(blur_val, minimum=1), 1, 31))

    return {
        "enabled": enabled,
        "applies": applies,
        "scale_eff": float(scale_eff),
        "scale_ref": float(scale_ref),
        "ratio": float(ratio),
        "gamma_ref": float(gamma_ref),
        "ceiling_ref": float(ceiling_ref),
        "blur_ref": int(blur_ref),
        "effective_gamma": float(effective_gamma),
        "effective_ceiling": float(effective_ceiling),
        "effective_blur": int(effective_blur),
    }


def format_attention_status(info: Dict[str, Any], mask_mode: str) -> str:
    base = (
        f"scale_eff={float(info.get('scale_eff', 0.0)):.3f} | "
        f"scale_ref={float(info.get('scale_ref', 0.0)):.3f} | "
        f"ratio={float(info.get('ratio', 1.0)):.3f}"
    )
    if not bool(info.get("enabled", False)):
        return base + " | Auto attention compensation: OFF"
    if not bool(info.get("applies", False)):
        return base + f" | Auto attention compensation: ON, but mask_mode='{mask_mode}' so it is not applied to the soft mask."
    return (
        base
        + " | effective soft mask: "
        + f"gamma={float(info.get('effective_gamma', 0.0)):.3f}, "
        + f"ceiling={float(info.get('effective_ceiling', 0.0)):.3f}, "
        + f"blur={int(info.get('effective_blur', 0))}"
    )


def build_mask_latent(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    latent_h: int,
    latent_w: int,
    cfg: SimpleNamespace,
    job_out: str,
    soft_hint: Optional[np.ndarray] = None,
    soft_for_mask: Optional[np.ndarray] = None,
    soft_mask_hw: Optional[np.ndarray] = None,
) -> torch.Tensor:
    sil_mask = make_latent_mask(sil_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)

    if cfg.mask_auto_invert and float(sil_mask.mean()) > 0.60:
        sil_mask = 1.0 - sil_mask

    if cfg.save_mask_images:
        save_mask_img(sil_mask, os.path.join(job_out, "mask_silhouette.png"), cfg.width, cfg.height)

    mode = str(cfg.mask_mode).lower()

    if mode == "silhouette":
        mask = sil_mask
    elif mode == "outline":
        ring = (_dilate_t(sil_mask, int(cfg.mask_ring_k)) - _erode_t(sil_mask, int(cfg.mask_ring_k))).clamp(0, 1)
        if int(cfg.mask_ring_blur) > 1:
            ring = _blur_t(ring, int(cfg.mask_ring_blur)).clamp(0, 1)
        mask = ring
    elif mode == "composite":
        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )
        mask = make_latent_mask(comp_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    elif mode == "softedge":
        if soft_mask_hw is None:
            src_for_mask = soft_for_mask if soft_for_mask is not None else soft_hint
            if src_for_mask is None:
                raise ValueError("mask_mode='softedge' but soft source is None")
            soft_mask_hw = softedge_to_mask_hw(
                soft_hint=src_for_mask,
                blur_ks=int(cfg.soft_mask_blur),
                gamma=float(cfg.soft_mask_gamma),
                invert=bool(cfg.soft_mask_invert),
                floor=float(cfg.soft_mask_floor),
                ceiling=float(cfg.soft_mask_ceiling),
            )
        mask = make_latent_mask(soft_mask_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    else:
        raise ValueError(f"Unknown mask_mode: {cfg.mask_mode}")

    if mode != "softedge":
        gamma = float(cfg.mask_gamma)
        if gamma != 1.0:
            mask = (mask.clamp(0, 1) ** gamma).clamp(0, 1)

    mask = mask.clamp(0, 1)

    if cfg.save_mask_images:
        save_mask_img(mask, os.path.join(job_out, "mask_latent.png"), cfg.width, cfg.height)
    return mask


def encode_prompt(pipe, prompt: str, negative_prompt: str, batch_size: int, do_cfg: bool, device: str) -> torch.Tensor:
    cond, uncond = pipe.encode_prompt(
        prompt=[prompt] * batch_size,
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=do_cfg,
        negative_prompt=[negative_prompt] * batch_size,
    )
    return torch.cat([uncond, cond], dim=0) if do_cfg else cond


def prepare_noise_latents(
    pipe,
    batch_size: int,
    height: int,
    width: int,
    generator: torch.Generator,
    device: str,
    dtype: torch.dtype,
) -> torch.Tensor:
    latent_h, latent_w = height // 8, width // 8
    shape = (batch_size, pipe.unet.config.in_channels, latent_h, latent_w)
    latents = torch.randn(shape, generator=generator, device=device, dtype=dtype)
    return latents * pipe.scheduler.init_noise_sigma


def pil_to_vae_tensor(img_pil: Image.Image, device: str, dtype: torch.dtype) -> torch.Tensor:
    arr = np.array(img_pil).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = x * 2.0 - 1.0
    return x.to(device=device, dtype=dtype)


@torch.no_grad()
def encode_image_latents(pipe, img_pil: Image.Image, generator: torch.Generator, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = pil_to_vae_tensor(img_pil, device=device, dtype=dtype)
    lat = pipe.vae.encode(x).latent_dist.sample(generator=generator)
    return lat * pipe.vae.config.scaling_factor


@torch.no_grad()
def decode_latents(pipe, latents: torch.Tensor) -> List[Image.Image]:
    latents = (1.0 / pipe.vae.config.scaling_factor) * latents
    img = pipe.vae.decode(latents).sample
    img = (img / 2 + 0.5).clamp(0, 1)
    img = img.cpu().permute(0, 2, 3, 1).numpy()
    imgs = (img * 255).round().astype(np.uint8)
    return [Image.fromarray(im) for im in imgs]


# ============================================================
# 7) Control weights + schedules
# ============================================================
def preset_weights_A(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.array([0, 0, 0, 0, 0.1, 0.1, 0.2, 0.2, 0.3, 0.6, 0.6, 0.6], dtype=np.float32)
    mid = 0.9
    if len(weights) != n_down:
        weights = np.resize(weights, n_down)
    return weights, mid


def preset_weights_B(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.linspace(0.25, 0.65, n_down).astype(np.float32)
    if n_down >= 12:
        for i in range(4, 9):
            weights[i] = min(0.70, weights[i] + 0.08)
    return weights, 0.55


def control_gate(step_idx: int, num_steps: int, start_frac: float, end_frac: float, kind: str = "cosine") -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    if step_idx < s0 or step_idx >= s1:
        return 0.0
    active = max(1, s1 - s0)
    x = (step_idx - s0) / active
    if kind == "linear":
        return float(1.0 - x)
    return float(math.cos(0.5 * math.pi * x))


def blend_alpha(
    step_idx: int,
    num_steps: int,
    start_frac: float,
    end_frac: float,
    alpha_end: float,
    profile: str = "decay",
) -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    s0 = max(0, min(s0, num_steps - 1))
    s1 = max(0, min(s1, num_steps))

    if profile == "ramp":
        if step_idx <= s0:
            return 0.0
        if step_idx >= s1:
            return float(alpha_end)
        return float(((step_idx - s0) / max(1, s1 - s0)) * alpha_end)

    if step_idx <= s0:
        return 1.0
    if step_idx >= s1:
        return float(alpha_end)
    return float(1.0 - ((step_idx - s0) / max(1, s1 - s0)) * (1.0 - alpha_end))


@torch.no_grad()
def unet_with_optional_control(
    pipe,
    scheduler,
    latents,
    t,
    encoder_hidden_states,
    do_cfg,
    guidance_scale,
    controlnets=None,
    control_images=None,
    control_scales=None,
    layer_weights=None,
    mid_weight=None,
    step_gate=1.0,
    guess_mode=False,
    device="cuda",
    dtype=torch.float16,
) -> torch.Tensor:
    unet = pipe.unet
    latents_in = torch.cat([latents] * 2, dim=0) if do_cfg else latents
    latents_in = scheduler.scale_model_input(latents_in, t)
    down_res, mid_res = None, None

    if controlnets is not None and control_images is not None:
        if control_scales is None:
            control_scales = [1.0] * len(controlnets)

        control_cond_mask = None
        if do_cfg and guess_mode:
            control_cond_mask = torch.zeros((latents_in.shape[0], 1, 1, 1), device=device, dtype=dtype)
            control_cond_mask[latents.shape[0]:] = 1.0

        sum_down, sum_mid = None, None
        for cn, img, scale in zip(controlnets, control_images, control_scales):
            img_in = torch.cat([img] * 2, dim=0) if do_cfg else img
            cn_out = cn(
                latents_in,
                t,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=img_in,
                return_dict=True,
            )

            down = list(cn_out.down_block_res_samples)
            mid = cn_out.mid_block_res_sample

            if layer_weights is not None:
                for i in range(min(len(down), len(layer_weights))):
                    down[i] = down[i] * float(layer_weights[i])

            if mid_weight is not None:
                mid = mid * float(mid_weight)

            if step_gate != 1.0:
                down = [d * float(step_gate) for d in down]
                mid = mid * float(step_gate)

            if control_cond_mask is not None:
                down = [d * control_cond_mask for d in down]
                mid = mid * control_cond_mask

            if scale != 1.0:
                down = [d * float(scale) for d in down]
                mid = mid * float(scale)

            if sum_down is None:
                sum_down, sum_mid = down, mid
            else:
                for i in range(len(sum_down)):
                    sum_down[i] = sum_down[i] + down[i]
                sum_mid = sum_mid + mid

        down_res, mid_res = sum_down, sum_mid

    noise_pred = unet(
        latents_in,
        t,
        encoder_hidden_states=encoder_hidden_states,
        down_block_additional_residuals=down_res,
        mid_block_additional_residual=mid_res,
        return_dict=False,
    )[0]

    if do_cfg:
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + float(guidance_scale) * (noise_cond - noise_uncond)

    return noise_pred


# ============================================================
# 8) Runner
# ============================================================
class CamoRunner:
    def __init__(self, cfg: SimpleNamespace):
        self.cfg = cfg
        self.pipe = None
        self.control_nets = None
        self.hed = None

    def build_pipe(self):
        cfg = self.cfg
        seed_everything(int(cfg.seed))

        cn_canny = ControlNetModel.from_pretrained(cfg.canny_cn_id, torch_dtype=cfg.torch_dtype)
        cn_soft = ControlNetModel.from_pretrained(cfg.soft_cn_id, torch_dtype=cfg.torch_dtype)

        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            cfg.base_model_id,
            controlnet=[cn_canny, cn_soft],
            torch_dtype=cfg.torch_dtype,
            safety_checker=None,
        )
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to(cfg.device)
        pipe.set_progress_bar_config(disable=True)

        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
        pipe.enable_attention_slicing()

        self.pipe = pipe
        self.control_nets = pipe.controlnet.nets if hasattr(pipe.controlnet, "nets") else [pipe.controlnet]
        return self

    def _get_hed(self):
        if self.hed is not None:
            return self.hed
        try:
            from controlnet_aux import HEDdetector

            self.hed = HEDdetector.from_pretrained(self.cfg.hed_annotator_id)
            return self.hed
        except Exception:
            return None

    def preprocess(
        self,
        image_path: str,
        seed: int,
        layout_override: Optional[Tuple[float, float, float]] = None,
    ) -> Dict[str, Any]:
        cfg = self.cfg
        src = load_image(image_path, size=(cfg.width, cfg.height))
        src_cv = pil_to_cv_bgr(src)

        if layout_override is None:
            layout_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
            layout_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
            layout_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
            layout_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
        else:
            values = [float(v) for v in layout_override]
            if len(values) >= 4:
                layout_scale, layout_offset_x, layout_offset_y, layout_rotation_deg = values[:4]
            else:
                layout_scale, layout_offset_x, layout_offset_y = values[:3]
                layout_rotation_deg = 0.0

        canny_hint = canny_clean_norm(
            src_cv,
            low=cfg.canny_low,
            high=cfg.canny_high,
            blur_ks=cfg.canny_blur_ks,
        )

        soft_hint = None
        hed = self._get_hed()
        if hed is not None:
            try:
                hed_map = hed(src)
                hed_np = normalize_0_255(np.array(hed_map.convert("L")))
                soft_hint = np.stack([hed_np, hed_np, hed_np], axis=-1)
            except Exception:
                pass

        if soft_hint is None:
            soft_hint = canny_hint.copy()

        if layout_scale != 1.0 or layout_offset_x != 0.0 or layout_offset_y != 0.0 or layout_rotation_deg != 0.0:
            canny_hint = transform_hint_uint8(
                canny_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_NEAREST,
            )
            soft_hint = transform_hint_uint8(
                soft_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_LINEAR,
            )

        if cfg.use_broken_edges:
            canny_hint = canny_broken(canny_hint, drop_prob=cfg.broken_drop_prob, seed=seed)

        sil_hw = edges_to_silhouette_mask(
            canny_hint,
            dilate=int(cfg.mask_dilate),
            close_ks=int(cfg.mask_close_ks),
            close_iter=int(getattr(cfg, "mask_close_iter", 1)),
            blur=int(cfg.mask_blur),
        )

        feature_hw = extract_internal_feature_mask(
            canny_hint,
            sil_hw,
            thresh=int(cfg.feature_thresh),
            open_ks=int(cfg.feature_open_ks),
            blur=int(cfg.feature_blur),
        )

        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )

        soft_for_mask = soft_hint.copy()
        if bool(cfg.use_soft_grad_mask):
            soft_gray = cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
            soft_grad_pil = kernel_gradient_hint(
                edges_2d=soft_gray,
                dilation_k=int(cfg.soft_grad_dilation_k),
                blur_k=int(cfg.soft_grad_blur_k),
                kernel_shape=str(cfg.soft_grad_kernel_shape),
                iterations=int(cfg.soft_grad_iterations),
                use_skeleton=bool(cfg.soft_grad_use_skeleton),
                skeleton_stage=str(cfg.soft_grad_skeleton_stage),
                skeleton_thresh=cfg.soft_grad_skeleton_thresh,
                post_skel_dilate_k=int(cfg.soft_grad_post_skel_dilate_k),
            )
            soft_for_mask = np.array(soft_grad_pil.convert("RGB"))

        attention_info = resolve_attention_compensation(cfg, sil_hw)
        soft_mask_hw = softedge_to_mask_hw(
            soft_hint=soft_for_mask,
            blur_ks=int(attention_info["effective_blur"]),
            gamma=float(attention_info["effective_gamma"]),
            invert=bool(cfg.soft_mask_invert),
            floor=float(cfg.soft_mask_floor),
            ceiling=float(attention_info["effective_ceiling"]),
        )

        return {
            "src": src,
            "canny_hint": canny_hint,
            "soft_hint": soft_hint,
            "sil_hw": sil_hw,
            "feature_hw": feature_hw,
            "comp_hw": comp_hw,
            "soft_for_mask": soft_for_mask,
            "soft_mask_hw": soft_mask_hw,
            "attention_info": attention_info,
        }

    def _job_out_dir(self, job: AnimalJob) -> str:
        base = os.path.splitext(os.path.basename(job.image_path))[0]
        job_name = job.name or f"{slugify(job.animals)}_{slugify(base)}"
        out_dir = os.path.join(self.cfg.out_dir, job_name)
        ensure_dir(out_dir)
        return out_dir

    def _save_run_config(self, job_out: str, job: AnimalJob, extra: dict) -> None:
        cfg_dict = {k: v for k, v in self.cfg.__dict__.items() if k != "torch_dtype"}
        cfg_dict["job"] = {
            "animals": job.animals,
            "image_path": job.image_path,
            "name": job.name,
            "seed": job.seed,
            **extra,
        }
        save_json(os.path.join(job_out, "run_config.json"), cfg_dict)

    def _probe_weights(self, lat_sub, t0, emb_sub, img_canny, do_cfg) -> Tuple[np.ndarray, float]:
        lat_in = torch.cat([lat_sub] * 2, 0) if do_cfg else lat_sub
        img_in = torch.cat([img_canny] * 2, 0) if do_cfg else img_canny
        probe = self.control_nets[0](
            self.pipe.scheduler.scale_model_input(lat_in, t0),
            t0,
            encoder_hidden_states=emb_sub,
            controlnet_cond=img_in,
            return_dict=True,
        )
        n_down = len(probe.down_block_res_samples)
        return preset_weights_B(n_down) if self.cfg.use_weights == "B" else preset_weights_A(n_down)

    def _init_latents_and_timesteps(self, job_out: str, job: AnimalJob, generator: torch.Generator):
        cfg, pipe = self.cfg, self.pipe
        pipe.scheduler.set_timesteps(int(cfg.steps), device=cfg.device)
        timesteps = pipe.scheduler.timesteps

        scheduler_bg = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_sub = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_bg.set_timesteps(int(cfg.steps), device=cfg.device)
        scheduler_sub.set_timesteps(int(cfg.steps), device=cfg.device)

        bg_path = job.bg_image_path if job.bg_image_path is not None else cfg.bg_image_path
        bg_strength = float(job.bg_strength) if job.bg_strength is not None else float(cfg.bg_strength)

        if bg_path:
            bg_img = load_image(bg_path, size=(cfg.width, cfg.height))
            bg_img.save(os.path.join(job_out, "bg_input.png"))

            lat_img = encode_image_latents(pipe, bg_img, generator, cfg.device, cfg.torch_dtype)
            lat_img = lat_img.repeat(int(cfg.batch), 1, 1, 1)

            num_steps = int(cfg.steps)
            init_timestep = max(1, min(num_steps, int(num_steps * bg_strength)))
            t_start = min(max(num_steps - init_timestep, 0), num_steps - 1)
            t_start_1 = timesteps[t_start : t_start + 1].to(cfg.device).repeat(int(cfg.batch))

            noise = torch.randn(lat_img.shape, generator=generator, device=lat_img.device, dtype=lat_img.dtype)
            lat_bg = pipe.scheduler.add_noise(lat_img, noise, t_start_1)
            lat_sub = lat_bg.clone()

            if float(cfg.sub_init_noise) > 0:
                sub_noise = torch.randn(lat_sub.shape, generator=generator, device=lat_sub.device, dtype=lat_sub.dtype)
                lat_sub = lat_sub + float(cfg.sub_init_noise) * sub_noise

            return lat_bg, lat_sub, timesteps[t_start:], scheduler_bg, scheduler_sub

        lat_bg = prepare_noise_latents(
            pipe,
            int(cfg.batch),
            int(cfg.height),
            int(cfg.width),
            generator,
            cfg.device,
            cfg.torch_dtype,
        )
        return lat_bg, lat_bg.clone(), timesteps, scheduler_bg, scheduler_sub

    def _apply_coupling(self, lat_out, lat_bg_next, lat_sub_next):
        mode = str(self.cfg.couple_mode).lower()
        if mode == "both":
            return lat_out, lat_out
        if mode == "bg_only":
            return lat_out, lat_sub_next
        return lat_bg_next, lat_sub_next

    @torch.no_grad()
    def run_one(self, job: AnimalJob, log_every: Optional[int] = None, show_preview: Optional[bool] = None):
        cfg, pipe = self.cfg, self.pipe
        job_out = self._job_out_dir(job)
        seed = int(job.seed) if job.seed is not None else int(cfg.seed)
        seed_everything(seed)

        gs_bg = float(getattr(cfg, "guidance_bg", getattr(cfg, "guidance_scale", 1.0)))
        gs_sub = float(getattr(cfg, "guidance_sub", getattr(cfg, "guidance_scale", 1.0)))
        do_cfg_bg, do_cfg_sub = gs_bg > 1.0, gs_sub > 1.0
        log_every = int(cfg.log_every if log_every is None else log_every)

        prep = self.preprocess(job.image_path, seed=seed)
        img_canny = to_torch_image_hint(prep["canny_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)
        img_soft = to_torch_image_hint(prep["soft_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)

        prompt_bg = job.prompt_bg or cfg.prompt_bg
        prompt_sub = format_prompt(job.subject_template or cfg.prompt_subject_template, job.animals)
        negative = job.negative or cfg.negative

        emb_bg = encode_prompt(pipe, prompt_bg, negative, int(cfg.batch), do_cfg_bg, cfg.device)
        emb_sub = encode_prompt(pipe, prompt_sub, negative, int(cfg.batch), do_cfg_sub, cfg.device)

        generator = torch.Generator(device=cfg.device).manual_seed(seed)
        lat_bg, lat_sub, timesteps_run, scheduler_bg, scheduler_sub = self._init_latents_and_timesteps(job_out, job, generator)

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            int(cfg.height) // 8,
            int(cfg.width) // 8,
            cfg,
            job_out=job_out,
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        w_down, w_mid = self._probe_weights(lat_sub, timesteps_run[0], emb_sub, img_canny[:1], do_cfg=do_cfg_sub)

        amp_ctx = (
            torch.autocast(device_type="cuda", dtype=cfg.torch_dtype)
            if cfg.device.startswith("cuda")
            else contextlib.nullcontext()
        )

        lat_out = lat_bg.clone()
        num_run = len(timesteps_run)

        with amp_ctx:
            for i, t in enumerate(timesteps_run):
                gate = control_gate(
                    i,
                    num_run,
                    float(cfg.start_frac_control),
                    float(cfg.end_frac_control),
                    str(cfg.control_gate_kind),
                )

                eps_bg = unet_with_optional_control(
                    pipe,
                    scheduler_bg,
                    lat_bg,
                    t,
                    emb_bg,
                    do_cfg_bg,
                    gs_bg,
                    None,
                    None,
                    device=cfg.device,
                    dtype=cfg.torch_dtype,
                )
                lat_bg_next = scheduler_bg.step(eps_bg, t, lat_bg).prev_sample

                eps_sub = unet_with_optional_control(
                    pipe,
                    scheduler_sub,
                    lat_sub,
                    t,
                    emb_sub,
                    do_cfg_sub,
                    gs_sub,
                    self.control_nets,
                    [img_canny, img_soft],
                    list(cfg.control_scales),
                    w_down,
                    w_mid,
                    gate,
                    bool(cfg.guess_mode),
                    cfg.device,
                    cfg.torch_dtype,
                )
                lat_sub_next = scheduler_sub.step(eps_sub, t, lat_sub).prev_sample

                alpha = blend_alpha(
                    i,
                    num_run,
                    float(cfg.blend_start_frac),
                    float(cfg.blend_end_frac),
                    float(cfg.blend_alpha_end),
                    str(cfg.blend_profile),
                )

                lat_out = lat_bg_next * (1.0 - alpha * mask) + lat_sub_next * (alpha * mask)
                lat_bg, lat_sub = self._apply_coupling(lat_out, lat_bg_next, lat_sub_next)

                if log_every and (i + 1) % log_every == 0:
                    print(f"[{job.animals}] step {i + 1:>3}/{num_run} | gate={gate:.3f} | alpha={alpha:.3f}")

        imgs = decode_latents(pipe, lat_out)
        paths = []
        for k, img in enumerate(imgs):
            out_path = os.path.join(job_out, f"out_{slugify(job.animals)}_{k:02d}_seed{seed}.png")
            img.save(out_path)
            paths.append(out_path)

        self._save_run_config(
            job_out,
            job,
            extra={
                "prompt_bg": prompt_bg,
                "prompt_sub": prompt_sub,
                "bg_strength": job.bg_strength or cfg.bg_strength,
            },
        )
        return imgs, paths


# ============================================================
# 9) Scoring / UI integration helpers
# ============================================================
def hidden_score(out_img_pil: Image.Image, outline_like_hw: np.ndarray) -> float:
    img = np.array(out_img_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160).astype(np.float32) / 255.0
    ring = (outline_like_hw > 0.25).astype(np.uint8)
    in_mean = float(edges[ring > 0].mean()) if (ring > 0).any() else 0.0
    out_mean = float(edges[ring == 0].mean())
    return float(-abs((in_mean - out_mean) - 0.01) - 0.5 * in_mean)


# ------------------------------------------------------------
# Demo metrics / quantitative proxy calculations
# ------------------------------------------------------------
def _to_gray01_from_pil(img_pil: Image.Image, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    img = img_pil.convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    arr = np.array(img).astype(np.uint8)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    return np.clip(gray, 0.0, 1.0)


def _to_gray01_from_any(x: Any, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    if isinstance(x, Image.Image):
        return _to_gray01_from_pil(x, size=size)
    arr = np.asarray(x)
    if arr.ndim == 3:
        arr = cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    arr = arr.astype(np.float32)
    if arr.max() > 1.5:
        arr = arr / 255.0
    if size is not None:
        arr = cv2.resize(arr, size, interpolation=cv2.INTER_LINEAR)
    return np.clip(arr, 0.0, 1.0)


def _global_ssim(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    mux = float(x.mean())
    muy = float(y.mean())
    varx = float(((x - mux) ** 2).mean())
    vary = float(((y - muy) ** 2).mean())
    cov = float(((x - mux) * (y - muy)).mean())

    denom = (mux ** 2 + muy ** 2 + c1) * (varx + vary + c2)
    if abs(denom) < 1e-12:
        return 0.0
    return float(((2 * mux * muy + c1) * (2 * cov + c2)) / denom)


def compute_ssim_bg(out_img: Image.Image, bg_img: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_gray = _to_gray01_from_pil(out_img, size=size)
    bg_gray = _to_gray01_from_pil(bg_img, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)
    bg_region = 1.0 - np.clip(mask, 0.0, 1.0)

    x = out_gray * bg_region
    y = bg_gray * bg_region

    try:
        from skimage.metrics import structural_similarity as ssim
        return float(ssim(x, y, data_range=1.0))
    except Exception:
        return _global_ssim(x, y)


def compute_edge_similarity(out_img: Image.Image, subject_edge_hint: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_rgb = np.array(out_img.convert("RGB").resize(size, Image.LANCZOS)).astype(np.uint8)
    out_gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY)
    e_out = cv2.Canny(out_gray, 80, 160).astype(np.float32) / 255.0

    e_sub = _to_gray01_from_pil(subject_edge_hint, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)

    a = (e_sub * mask).reshape(-1).astype(np.float32)
    b = (e_out * mask).reshape(-1).astype(np.float32)

    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom < 1e-8:
        return 0.0
    return float(np.dot(a, b) / denom)


def _image_edge_density(img_pil: Image.Image, mask_img: Optional[Image.Image] = None) -> float:
    gray = _to_gray01_from_pil(img_pil)
    edges = cv2.Canny((gray * 255).astype(np.uint8), 80, 160).astype(np.float32) / 255.0
    if mask_img is None:
        return float(edges.mean())
    mask = _to_gray01_from_pil(mask_img, size=img_pil.size)
    active = mask > 0.25
    if not active.any():
        return 0.0
    return float(edges[active].mean())


def compute_demo_metrics(
    runner: CamoRunner,
    job: AnimalJob,
    out_img: Image.Image,
    previews: Dict[str, Any],
    hidden_score_value: float,
) -> Dict[str, Any]:
    cfg = runner.cfg
    bg_img = previews["preview_bg_original"]
    canny_hint_img = previews["preview_canny"]
    mask_img = previews["preview_mask"]

    mask_np = _to_gray01_from_pil(mask_img)
    active = mask_np > 0.25

    metrics = {
        "seed": int(job.seed),
        "subject_name": str(job.animals),
        "subject_path": str(job.image_path),
        "background_path": str(job.bg_image_path or cfg.bg_image_path),
        "hidden_score": float(hidden_score_value),
        "SSIM_bg": compute_ssim_bg(out_img, bg_img, mask_img),
        "S_edge": compute_edge_similarity(out_img, canny_hint_img, mask_img),
        "mask_mean": float(mask_np.mean()),
        "mask_max": float(mask_np.max()),
        "mask_active_ratio": float(active.mean()),
        "output_edge_density_global": _image_edge_density(out_img),
        "output_edge_density_in_mask": _image_edge_density(out_img, mask_img),
        "layout_effective_scale": float(previews["attention_info"].get("scale_eff", 0.0)),
        "layout_scale": float(cfg.blend_mask_scale),
        "layout_offset_x": float(cfg.blend_mask_offset_x),
        "layout_offset_y": float(cfg.blend_mask_offset_y),
        "layout_rotation_deg": float(cfg.blend_mask_rotation_deg),
        "steps": int(cfg.steps),
        "bg_strength": float(cfg.bg_strength),
        "guidance_bg": float(cfg.guidance_bg),
        "guidance_sub": float(cfg.guidance_sub),
        "canny_scale": float(cfg.control_scales[0]),
        "softedge_scale": float(cfg.control_scales[1]),
        "blend_profile": str(cfg.blend_profile),
        "blend_start_frac": float(cfg.blend_start_frac),
        "blend_end_frac": float(cfg.blend_end_frac),
        "blend_alpha_end": float(cfg.blend_alpha_end),
        "mask_mode": str(cfg.mask_mode),
        "soft_mask_blur": int(cfg.soft_mask_blur),
        "soft_mask_gamma": float(cfg.soft_mask_gamma),
        "soft_mask_floor": float(cfg.soft_mask_floor),
        "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
    }
    return metrics


def print_demo_metrics(metrics: Dict[str, Any]) -> None:
    print("\n" + "=" * 72)
    print("Calculated Demo Metrics")
    print("=" * 72)
    ordered_keys = [
        "seed",
        "subject_name",
        "SSIM_bg",
        "S_edge",
        "hidden_score",
        "mask_mean",
        "mask_max",
        "mask_active_ratio",
        "output_edge_density_global",
        "output_edge_density_in_mask",
        "layout_effective_scale",
        "steps",
        "bg_strength",
        "guidance_bg",
        "guidance_sub",
        "canny_scale",
        "softedge_scale",
        "blend_profile",
        "blend_start_frac",
        "blend_end_frac",
        "blend_alpha_end",
    ]
    for key in ordered_keys:
        value = metrics.get(key)
        if isinstance(value, float):
            print(f"{key:>28}: {value:.6f}")
        else:
            print(f"{key:>28}: {value}")
    print("=" * 72 + "\n")


def build_demo_metrics_html(metrics: Dict[str, Any]) -> str:
    core_rows = [
        ("SSIM_bg", metrics.get("SSIM_bg")),
        ("S_edge", metrics.get("S_edge")),
        ("Hidden Score", metrics.get("hidden_score")),
        ("Mask Mean", metrics.get("mask_mean")),
        ("Mask Active Ratio", metrics.get("mask_active_ratio")),
        ("Output Edge Density", metrics.get("output_edge_density_global")),
        ("Output Edge Density in Mask", metrics.get("output_edge_density_in_mask")),
        ("Layout Effective Scale", metrics.get("layout_effective_scale")),
    ]

    param_rows = [
        ("Seed", metrics.get("seed")),
        ("Steps", metrics.get("steps")),
        ("BG Strength", metrics.get("bg_strength")),
        ("Guidance BG", metrics.get("guidance_bg")),
        ("Guidance Subject", metrics.get("guidance_sub")),
        ("Canny Scale", metrics.get("canny_scale")),
        ("SoftEdge Scale", metrics.get("softedge_scale")),
        ("Blend Profile", metrics.get("blend_profile")),
        ("Blend Start", metrics.get("blend_start_frac")),
        ("Blend End", metrics.get("blend_end_frac")),
        ("Alpha End", metrics.get("blend_alpha_end")),
    ]

    def fmt(v: Any) -> str:
        if isinstance(v, float):
            return f"{v:.6f}"
        return str(v)

    def rows_html(rows: List[Tuple[str, Any]]) -> str:
        return "".join(
            f"<tr><td style='padding:5px 10px; color:#cbd5e1;'>{html_lib.escape(str(k))}</td>"
            f"<td style='padding:5px 10px; color:#f8fafc; font-weight:600;'>{html_lib.escape(fmt(v))}</td></tr>"
            for k, v in rows
        )

    return f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">Calculated Output Metrics</div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:14px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Proxy Metrics</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(core_rows)}</table>
            </div>
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Main Parameters</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(param_rows)}</table>
            </div>
        </div>
    </div>
    """


def apply_ui_params_to_runner(runner: CamoRunner, ui: Dict[str, Any]) -> None:
    cfg = runner.cfg
    cfg.steps = int(ui["steps"])
    cfg.bg_strength = float(ui["bg_strength"])
    cfg.guidance_bg = float(ui["guidance_bg"])
    cfg.guidance_sub = float(ui["guidance_sub"])
    cfg.sub_init_noise = float(ui["sub_init_noise"])
    cfg.couple_mode = str(ui["couple_mode"])

    cfg.canny_low = int(ui["canny_low"])
    cfg.canny_high = int(ui["canny_high"])
    cfg.canny_blur_ks = int(ui["canny_blur_ks"])
    cfg.use_broken_edges = bool(ui["use_broken_edges"])
    cfg.broken_drop_prob = float(ui["broken_drop_prob"])

    cfg.mask_mode = str(ui["mask_mode"])
    cfg.mask_dilate = int(ui["mask_dilate"])
    cfg.mask_close_ks = int(ui["mask_close_ks"])
    cfg.mask_blur = int(ui["mask_blur"])
    cfg.mask_ring_k = int(ui["mask_ring_k"])
    cfg.mask_ring_blur = int(ui["mask_ring_blur"])
    cfg.mask_gamma = float(ui["mask_gamma"])
    cfg.mask_auto_invert = bool(ui["mask_auto_invert"])

    cfg.outer_weight = float(ui["outer_weight"])
    cfg.feature_weight = float(ui["feature_weight"])
    cfg.fill_weight = float(ui["fill_weight"])
    cfg.feature_thresh = int(ui["feature_thresh"])
    cfg.feature_open_ks = int(ui["feature_open_ks"])
    cfg.feature_blur = int(ui["feature_blur"])
    cfg.fill_blur = int(ui["fill_blur"])

    cfg.use_soft_grad_mask = bool(ui["use_soft_grad_mask"])
    cfg.soft_grad_dilation_k = int(ui["soft_grad_dilation_k"])
    cfg.soft_grad_blur_k = int(ui["soft_grad_blur_k"])
    cfg.soft_grad_kernel_shape = str(ui["soft_grad_kernel_shape"])
    cfg.soft_grad_iterations = int(ui["soft_grad_iterations"])
    cfg.soft_grad_use_skeleton = bool(ui["soft_grad_use_skeleton"])
    cfg.soft_grad_skeleton_stage = str(ui["soft_grad_skeleton_stage"])
    cfg.soft_grad_post_skel_dilate_k = int(ui["soft_grad_post_skel_dilate_k"])

    cfg.soft_mask_blur = int(ui["soft_mask_blur"])
    cfg.soft_mask_gamma = float(ui["soft_mask_gamma"])
    cfg.soft_mask_invert = bool(ui["soft_mask_invert"])
    cfg.soft_mask_floor = float(ui["soft_mask_floor"])
    cfg.soft_mask_ceiling = float(ui["soft_mask_ceiling"])

    cfg.control_scales = (float(ui["canny_scale"]), float(ui["soft_scale"]))
    cfg.start_frac_control = float(ui["start_frac"])
    cfg.end_frac_control = float(ui["end_frac"])
    cfg.control_gate_kind = str(ui["gate_kind"])

    cfg.blend_profile = str(ui["blend_profile"])
    cfg.blend_start_frac = float(ui["blend_start"])
    cfg.blend_end_frac = float(ui["blend_end"])
    cfg.blend_alpha_end = float(ui["alpha_end"])
    cfg.auto_attention_compensation = bool(ui["auto_attention_compensation"])
    cfg.attention_scale_ref = float(ui["attention_scale_ref"])
    cfg.attention_gamma_ref = float(ui["attention_gamma_ref"])
    cfg.attention_ceiling_ref = float(ui["attention_ceiling_ref"])
    cfg.attention_blur_ref = int(ui["attention_blur_ref"])
    cfg.blend_mask_scale = float(ui["blend_mask_scale"])
    cfg.blend_mask_offset_x = float(ui["blend_mask_offset_x"])
    cfg.blend_mask_offset_y = float(ui["blend_mask_offset_y"])
    cfg.blend_mask_rotation_deg = float(ui["blend_mask_rotation_deg"])
    cfg.randomize_seed_each_generate = bool(ui["randomize_seed_each_generate"])


def build_job_from_ui(ui: Dict[str, Any]) -> AnimalJob:
    animal_path_raw = str(ui["path_animal"]).strip()
    bg_path_raw = str(ui["path_bg"]).strip()

    resolved_animal = resolve_image_path(animal_path_raw, fallback_dir=KAGGLE_ANIMALS_DIR)
    if resolved_animal is None:
        raise gr.Error(f"No valid subject image found from: {animal_path_raw}")

    resolved_bg = resolve_image_path(bg_path_raw, fallback_dir=KAGGLE_BG_DIR) if bg_path_raw or KAGGLE_BG_DIR else None

    return AnimalJob(
        image_path=resolved_animal,
        bg_image_path=resolved_bg,
        animals=str(ui["animal_name"]),
        seed=int(ui["seed"]),
        prompt_bg=str(ui["prompt_bg"]),
        subject_template=str(ui["prompt_sub"]),
        negative=str(ui["negative_prompt"]),
        name=f"UI_SoftGradMask_{int(time.time())}",
        bg_strength=float(ui["bg_strength"]),
    )



def stage2_canny_breakdown(canny_hint_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    edges = canny_hint_u8[..., 0].astype(np.uint8)

    contour_agg = edges.copy()

    if int(cfg.mask_dilate) > 1:
        k = _odd(int(cfg.mask_dilate))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.dilate(contour_agg, kernel, iterations=1)

    if int(cfg.mask_close_ks) > 1:
        k = _odd(int(cfg.mask_close_ks))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.morphologyEx(
            contour_agg,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=max(1, int(getattr(cfg, "mask_close_iter", 1))),
        )

    inv = cv2.bitwise_not(contour_agg)
    flood = inv.copy()
    h, w = contour_agg.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)
    region_filled = (flood > 0).astype(np.uint8) * 255

    smoothing = region_filled.copy()
    if int(cfg.mask_blur) > 1:
        k = _odd(int(cfg.mask_blur))
        smoothing = cv2.GaussianBlur(smoothing, (k, k), 0)

    return {
        "canny_edges": edges,
        "contour_aggregation": contour_agg,
        "region_filling": region_filled,
        "canny_smoothing": smoothing,
    }


def stage2_softedge_breakdown(soft_input_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    if soft_input_u8.ndim == 3:
        gray = cv2.cvtColor(soft_input_u8.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = soft_input_u8.astype(np.uint8)

    smoothed = gray.astype(np.float32) / 255.0
    if int(cfg.soft_mask_blur) > 1:
        k = _odd(int(cfg.soft_mask_blur))
        smoothed = cv2.GaussianBlur(smoothed, (k, k), 0)

    remap_input = smoothed.copy()
    if bool(cfg.soft_mask_invert):
        remap_input = 1.0 - remap_input
    remap_input = np.clip(remap_input, 0.0, 1.0)

    gamma_remap = remap_input.copy()
    gamma = float(cfg.soft_mask_gamma)
    if gamma != 1.0:
        gamma_remap = gamma_remap ** gamma

    floor = float(np.clip(cfg.soft_mask_floor, 0.0, 1.0))
    ceiling = float(np.clip(cfg.soft_mask_ceiling, 0.0, 1.0))
    clipped = gamma_remap.copy()
    if ceiling > floor:
        clipped = np.clip((clipped - floor) / (ceiling - floor), 0.0, 1.0)

    final_soft_mask = np.clip(clipped, 0.0, 1.0)

    return {
        "soft_gray": gray01_to_u8(gray.astype(np.float32) / 255.0),
        "soft_smoothing": gray01_to_u8(smoothed),
        "soft_after_gamma_remap": gray01_to_u8(gamma_remap),
        "soft_after_clip": gray01_to_u8(final_soft_mask),
    }


def _stats_from_array(name: str, arr: np.ndarray) -> Dict[str, Any]:
    x = np.asarray(arr).astype(np.float32)
    if x.ndim == 3:
        x = x.mean(axis=2)

    nonzero = float((x > 0).sum())
    total = float(x.size)
    ratio = nonzero / total if total > 0 else 0.0

    return {
        "name": name,
        "shape": tuple(x.shape),
        "min": float(x.min()) if x.size else 0.0,
        "max": float(x.max()) if x.size else 0.0,
        "mean": float(x.mean()) if x.size else 0.0,
        "std": float(x.std()) if x.size else 0.0,
        "nonzero_ratio": float(ratio),
    }


def build_stage2_value_dict(
    runner: CamoRunner,
    job: AnimalJob,
    prep: Dict[str, Any],
    mask: torch.Tensor,
    canny_steps: Dict[str, np.ndarray],
    soft_steps: Dict[str, np.ndarray],
) -> Dict[str, Any]:
    cfg = runner.cfg

    mask_pil = mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height)
    mask_np = np.array(mask_pil.convert("L"))

    info = {
        "paths": {
            "animal_image": job.image_path,
            "background_image": job.bg_image_path or cfg.bg_image_path,
            "out_dir": runner._job_out_dir(job),
        },
        "layout": {
            "blend_mask_scale": float(cfg.blend_mask_scale),
            "blend_mask_offset_x": float(cfg.blend_mask_offset_x),
            "blend_mask_offset_y": float(cfg.blend_mask_offset_y),
            "blend_mask_rotation_deg": float(cfg.blend_mask_rotation_deg),
        },
        "canny_params": {
            "canny_low": int(cfg.canny_low),
            "canny_high": int(cfg.canny_high),
            "canny_blur_ks": int(cfg.canny_blur_ks),
            "use_broken_edges": bool(cfg.use_broken_edges),
            "broken_drop_prob": float(cfg.broken_drop_prob),
        },
        "support_params": {
            "mask_dilate": int(cfg.mask_dilate),
            "mask_close_ks": int(cfg.mask_close_ks),
            "mask_close_iter": int(getattr(cfg, "mask_close_iter", 1)),
            "mask_blur": int(cfg.mask_blur),
            "mask_mode": str(cfg.mask_mode),
        },
        "soft_mask_params": {
            "use_soft_grad_mask": bool(cfg.use_soft_grad_mask),
            "soft_grad_dilation_k": int(cfg.soft_grad_dilation_k),
            "soft_grad_blur_k": int(cfg.soft_grad_blur_k),
            "soft_grad_kernel_shape": str(cfg.soft_grad_kernel_shape),
            "soft_grad_iterations": int(cfg.soft_grad_iterations),
            "soft_grad_use_skeleton": bool(cfg.soft_grad_use_skeleton),
            "soft_grad_skeleton_stage": str(cfg.soft_grad_skeleton_stage),
            "soft_grad_post_skel_dilate_k": int(cfg.soft_grad_post_skel_dilate_k),
            "soft_mask_blur": int(cfg.soft_mask_blur),
            "soft_mask_gamma": float(cfg.soft_mask_gamma),
            "soft_mask_invert": bool(cfg.soft_mask_invert),
            "soft_mask_floor": float(cfg.soft_mask_floor),
            "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
        },
        "attention_info": prep["attention_info"],
        "arrays": {
            "canny_hint": _stats_from_array("canny_hint", prep["canny_hint"]),
            "contour_aggregation": _stats_from_array("contour_aggregation", canny_steps["contour_aggregation"]),
            "region_filling": _stats_from_array("region_filling", canny_steps["region_filling"]),
            "canny_smoothing": _stats_from_array("canny_smoothing", canny_steps["canny_smoothing"]),
            "sil_hw": _stats_from_array("sil_hw", prep["sil_hw"]),
            "feature_hw": _stats_from_array("feature_hw", prep["feature_hw"]),
            "comp_hw": _stats_from_array("comp_hw", prep["comp_hw"]),
            "soft_hint": _stats_from_array("soft_hint", prep["soft_hint"]),
            "soft_for_mask": _stats_from_array("soft_for_mask", prep["soft_for_mask"]),
            "soft_gray": _stats_from_array("soft_gray", soft_steps["soft_gray"]),
            "soft_smoothing": _stats_from_array("soft_smoothing", soft_steps["soft_smoothing"]),
            "soft_after_gamma_remap": _stats_from_array("soft_after_gamma_remap", soft_steps["soft_after_gamma_remap"]),
            "soft_after_clip": _stats_from_array("soft_after_clip", soft_steps["soft_after_clip"]),
            "soft_mask_hw": _stats_from_array("soft_mask_hw", prep["soft_mask_hw"]),
            "latent_mask": _stats_from_array("latent_mask", mask_np),
        },
    }
    return info


def build_stage2_debug_html(stage2_info: Dict[str, Any]) -> str:
    def render_dict(d: Dict[str, Any]) -> str:
        rows = []
        for k, v in d.items():
            if isinstance(v, dict):
                rows.append(
                    f"<tr><td colspan='2' style='padding-top:10px; font-weight:700; color:#93c5fd'>{k}</td></tr>"
                )
                for kk, vv in v.items():
                    rows.append(
                        f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{kk}</td>"
                        f"<td style='padding:4px 10px; color:#f8fafc'>{vv}</td></tr>"
                    )
            else:
                rows.append(
                    f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{k}</td>"
                    f"<td style='padding:4px 10px; color:#f8fafc'>{v}</td></tr>"
                )
        return "".join(rows)

    arrays_html = []
    for name, stats in stage2_info["arrays"].items():
        arrays_html.append(f"""
        <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
            <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">{name}</div>
            <table style="width:100%; font-size:13px; border-collapse:collapse;">
                <tr><td style="padding:3px 8px; color:#cbd5e1;">shape</td><td style="padding:3px 8px; color:#f8fafc;">{stats['shape']}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">min</td><td style="padding:3px 8px; color:#f8fafc;">{stats['min']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">max</td><td style="padding:3px 8px; color:#f8fafc;">{stats['max']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">mean</td><td style="padding:3px 8px; color:#f8fafc;">{stats['mean']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">std</td><td style="padding:3px 8px; color:#f8fafc;">{stats['std']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">nonzero_ratio</td><td style="padding:3px 8px; color:#f8fafc;">{stats['nonzero_ratio']:.4f}</td></tr>
            </table>
        </div>
        """)

    html = f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">
            Stage-2 Diagnostic Values
        </div>

        <div style="display:grid; grid-template-columns:1fr 1fr; gap:16px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Config Summary</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict({
                        "paths": stage2_info["paths"],
                        "layout": stage2_info["layout"],
                        "canny_params": stage2_info["canny_params"],
                        "support_params": stage2_info["support_params"],
                        "soft_mask_params": stage2_info["soft_mask_params"],
                    })}
                </table>
            </div>

            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Attention / Compensation</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict(stage2_info["attention_info"])}
                </table>
            </div>
        </div>

        <div style="margin-top:14px; display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px;">
            {''.join(arrays_html)}
        </div>
    </div>
    """
    return html


def make_debug_previews(runner: CamoRunner, job: AnimalJob):
    prep = runner.preprocess(job.image_path, seed=int(job.seed))
    job_out = runner._job_out_dir(job)
    latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8

    mask = build_mask_latent(
        prep["sil_hw"],
        prep["feature_hw"],
        latent_h,
        latent_w,
        runner.cfg,
        job_out=job_out,
        soft_hint=prep["soft_hint"],
        soft_for_mask=prep["soft_for_mask"],
        soft_mask_hw=prep["soft_mask_hw"],
    )

    canny_steps = stage2_canny_breakdown(prep["canny_hint"], runner.cfg)
    soft_steps = stage2_softedge_breakdown(prep["soft_for_mask"], runner.cfg)

    stage2_info = build_stage2_value_dict(
        runner=runner,
        job=job,
        prep=prep,
        mask=mask,
        canny_steps=canny_steps,
        soft_steps=soft_steps,
    )

    score_base = prep["sil_hw"].copy()
    stage2_json = json.dumps(stage2_info, ensure_ascii=False, indent=2)
    stage2_html = build_stage2_debug_html(stage2_info)

    previews = {
        "preview_bg_original": load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height)),
        "preview_input": prep["src"],
        "preview_canny": rgb_uint8_to_pil(prep["canny_hint"]),
        "preview_soft": rgb_uint8_to_pil(prep["soft_hint"]),
        "preview_sil": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_feat": gray_to_pil((prep["feature_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_comp": gray_to_pil((prep["comp_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_for_mask": rgb_uint8_to_pil(prep["soft_for_mask"]),
        "preview_soft_hw": gray_to_pil((prep["soft_mask_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_mask": mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height),
        "preview_contour_agg": gray_to_pil(canny_steps["contour_aggregation"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_region_filling": gray_to_pil(canny_steps["region_filling"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_canny_smoothing": gray_to_pil(canny_steps["canny_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_aux_support": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_gray": gray_to_pil(soft_steps["soft_gray"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_smoothing": gray_to_pil(soft_steps["soft_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_remap": gray_to_pil(soft_steps["soft_after_gamma_remap"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_clip": gray_to_pil(soft_steps["soft_after_clip"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_stage2_report_html": stage2_html,
        "preview_stage2_report_json": stage2_json,
        "score_base": score_base,
        "attention_info": prep["attention_info"],
    }
    return previews


# ============================================================
# 10) Main / Gradio UI
# ============================================================
if __name__ == "__main__":
    cfg = build_cfg(CFG)
    runner = CamoRunner(cfg).build_pipe()


    # ------------------------------------------------------------
    # Upload helpers
    # ------------------------------------------------------------
    # The original pipeline expects image paths. These helpers let the
    # Gradio UI accept uploaded PIL images, save them into out_dir, and
    # then reuse the existing path-based pipeline without changing the
    # core model code.
    def save_uploaded_image(img_pil: Optional[Image.Image], prefix: str) -> Optional[str]:
        if img_pil is None:
            return None

        upload_dir = os.path.join(runner.cfg.out_dir, "_ui_uploads")
        ensure_dir(upload_dir)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        path = os.path.join(upload_dir, f"{prefix}_{timestamp}.png")
        img_pil.convert("RGB").save(path)
        return path


    def apply_uploads_to_ui(
        ui: Dict[str, Any],
        subject_upload: Optional[Image.Image] = None,
        bg_upload: Optional[Image.Image] = None,
    ) -> Dict[str, Any]:
        ui = dict(ui)

        subject_path = save_uploaded_image(subject_upload, "subject")
        bg_path = save_uploaded_image(bg_upload, "background")

        # If the user uploads images, uploaded images override Textbox paths.
        # If no upload exists, the old Textbox path behavior remains unchanged.
        if subject_path is not None:
            ui["path_animal"] = subject_path

        if bg_path is not None:
            ui["path_bg"] = bg_path

        return ui


    def render_layout_editor_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed), layout_override=(1.0, 0.0, 0.0, 0.0))

        latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8
        cfg_preview = SimpleNamespace(**vars(runner.cfg))
        cfg_preview.save_mask_images = False
        cfg_preview.blend_mask_scale = 1.0
        cfg_preview.blend_mask_offset_x = 0.0
        cfg_preview.blend_mask_offset_y = 0.0
        cfg_preview.blend_mask_rotation_deg = 0.0

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            latent_h,
            latent_w,
            cfg_preview,
            job_out=runner._job_out_dir(job),
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        bg_img = load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height))
        overlay_img = mask_preview_to_rgba(mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height))
        return build_interactive_layout_html(
            bg_img=bg_img,
            overlay_img=overlay_img,
            width=runner.cfg.width,
            height=runner.cfg.height,
            init_scale=float(ui["blend_mask_scale"]),
            init_offset_x=float(ui["blend_mask_offset_x"]),
            init_offset_y=float(ui["blend_mask_offset_y"]),
            init_rotation_deg=float(ui["blend_mask_rotation_deg"]),
        )

    def summarize_layout_attention_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        return format_attention_status(prep["attention_info"], runner.cfg.mask_mode)

    def refresh_layout_editor(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        html = render_layout_editor_from_ui(ui)
        status = "Layout canvas has been refreshed from the Canny/SoftEdge support. " + summarize_layout_attention_from_ui(ui)
        return html, status

    def capture_current_scale_reference(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        scale_eff = float(prep["attention_info"]["scale_eff"])
        status = f"Captured the current layout scale as the reference: {scale_eff:.3f}"
        return scale_eff, status

    def safe_initial_layout_html() -> str:
        try:
            return render_layout_editor_from_ui(normalize_ui_params(UI_DEFAULTS))
        except Exception as e:
            return (
                "<div style='padding:12px;border:1px solid rgba(255,255,255,.1);border-radius:12px'>"
                f"Could not create the interactive layout canvas. Please check the image paths and click <b>Refresh Interactive Canvas</b>.<br><small>{e}</small></div>"
            )

    JS_SYNC_CANVAS_TO_ARGS = r"""
    (...args) => {
      const state = window.camoPlacementState;
      const syncHost = (elemId, value) => {
        const host = document.getElementById(elemId);
        if (!host) return;
        host.querySelectorAll('input').forEach((input) => {
          input.value = String(Number(value).toFixed(4));
          input.dispatchEvent(new Event('input', { bubbles: true }));
          input.dispatchEvent(new Event('change', { bubbles: true }));
        });
      };
      if (state) {
        const n = args.length;
        if (n >= 4) {
          args[n - 4] = Number(state.scale);
          args[n - 3] = Number(state.offsetX);
          args[n - 2] = Number(state.offsetY);
          args[n - 1] = Number(state.rotationDeg);
        }
        syncHost('blend-mask-scale', state.scale);
        syncHost('blend-mask-offset-x', state.offsetX);
        syncHost('blend-mask-offset-y', state.offsetY);
        syncHost('blend-mask-rotation-deg', state.rotationDeg);
      }
      return args;
    }
    """

    def generate_preview(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        if bool(ui["randomize_seed_each_generate"]):
            ui["seed"] = fresh_seed()

        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)

        print(f"\\n[UI] Generating {job.animals} - Seed {job.seed} - Steps {runner.cfg.steps}...")

        previews = make_debug_previews(runner, job)
        imgs, _ = runner.run_one(job, log_every=0, show_preview=False)
        score = hidden_score(imgs[0], previews["score_base"])
        demo_metrics = compute_demo_metrics(runner, job, imgs[0], previews, hidden_score_value=score)
        print_demo_metrics(demo_metrics)
        metrics_json = json.dumps(demo_metrics, ensure_ascii=False, indent=2)
        metrics_html = build_demo_metrics_html(demo_metrics)
        save_json(os.path.join(runner._job_out_dir(job), "calculated_demo_metrics.json"), demo_metrics)

        status = (
            f"Generation completed | seed={job.seed} | "
            f"SSIM_bg={demo_metrics['SSIM_bg']:.4f} | "
            f"S_edge={demo_metrics['S_edge']:.4f}. "
            + format_attention_status(previews["attention_info"], runner.cfg.mask_mode)
        )

        return (
            int(job.seed),
            imgs[0],
            previews["preview_bg_original"],
            previews["preview_input"],
            previews["preview_canny"],
            previews["preview_soft"],
            previews["preview_sil"],
            previews["preview_feat"],
            previews["preview_comp"],
            previews["preview_soft_for_mask"],
            previews["preview_soft_hw"],
            previews["preview_mask"],
            previews["preview_contour_agg"],
            previews["preview_region_filling"],
            previews["preview_canny_smoothing"],
            previews["preview_aux_support"],
            previews["preview_soft_gray"],
            previews["preview_soft_smoothing"],
            previews["preview_soft_remap"],
            previews["preview_soft_clip"],
            f"Hidden Score: {score:.4f}",
            status,
            metrics_html,
            metrics_json,
            previews["preview_stage2_report_html"],
            previews["preview_stage2_report_json"],
        )

    def save_current_config(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filepath = os.path.join(runner.cfg.out_dir, f"ui_config_{timestamp}.json")
        payload = {
            "timestamp": timestamp,
            "version": 5,
            "layout_stage": "preprocess_hints",
            "params": ui,
            "layout_params": {
                "layout_scale": ui["blend_mask_scale"],
                "layout_offset_x": ui["blend_mask_offset_x"],
                "layout_offset_y": ui["blend_mask_offset_y"],
                "layout_rotation_deg": ui["blend_mask_rotation_deg"],
            },
            "attention_compensation": {
                "enabled": ui["auto_attention_compensation"],
                "reference_scale": ui["attention_scale_ref"],
                "reference_gamma": ui["attention_gamma_ref"],
                "reference_ceiling": ui["attention_ceiling_ref"],
                "reference_blur": ui["attention_blur_ref"],
            },
            "legacy_params": [ui[name] for name in UI_FIELD_NAMES],
        }
        save_json(filepath, payload)
        return f"Configuration saved at: {filepath}"

    def load_config_to_ui(config_path: str):
        config_path = str(config_path).strip()
        if not config_path:
            raise gr.Error("Please enter a JSON config path.")
        if not os.path.exists(config_path):
            raise gr.Error(f"File not found: {config_path}")

        with open(config_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        raw_params = payload.get("params", payload) if isinstance(payload, dict) else payload
        missing_keys = []
        if isinstance(raw_params, dict):
            missing_keys = [name for name in UI_FIELD_NAMES if name not in raw_params]
        elif isinstance(raw_params, list):
            missing_keys = UI_FIELD_NAMES[len(raw_params):]

        ui = normalize_ui_params(raw_params)
        auto_filled = [k for k in missing_keys if k in {"auto_attention_compensation", "attention_scale_ref", "attention_gamma_ref", "attention_ceiling_ref", "attention_blur_ref", "blend_mask_scale", "blend_mask_offset_x", "blend_mask_offset_y", "blend_mask_rotation_deg", "randomize_seed_each_generate"}]
        status = f"Loaded config: {config_path}"
        if auto_filled:
            status += " | Auto-filled defaults: " + ", ".join(auto_filled)
        layout_html = render_layout_editor_from_ui(ui)
        return [ui[name] for name in UI_FIELD_NAMES] + [status, layout_html]


    # ------------------------------------------------------------
    # Batch helpers: folder / multiple files / ZIP
    # ------------------------------------------------------------
    BATCH_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


    def resolve_gradio_file_path(file_value: Any) -> str:
        if isinstance(file_value, str):
            path = file_value
        elif hasattr(file_value, "name"):
            path = str(file_value.name)
        else:
            path = str(file_value or "")

        path = path.strip()
        if not path or not os.path.isfile(path):
            raise FileNotFoundError(f"Uploaded file not found: {path}")
        return path


    def normalize_uploaded_values(value: Any) -> List[Any]:
        if value is None:
            return []
        if isinstance(value, (list, tuple)):
            return list(value)
        return [value]


    def list_images_recursive(path: str) -> List[str]:
        path = os.path.abspath(str(path))
        if os.path.isfile(path):
            ext = os.path.splitext(path)[1].lower()
            return [path] if ext in BATCH_IMAGE_EXTENSIONS else []

        if not os.path.isdir(path):
            return []

        results = []
        for root, _, names in os.walk(path):
            for name in sorted(names):
                full = os.path.join(root, name)
                if os.path.splitext(name)[1].lower() in BATCH_IMAGE_EXTENSIONS:
                    results.append(full)
        return results


    def extract_image_zip(zip_value: Any, extract_root: str, label: str) -> List[str]:
        values = normalize_uploaded_values(zip_value)
        if not values:
            return []

        if len(values) > 1:
            raise gr.Error(f"Only one {label} ZIP file is supported per run.")

        zip_path = resolve_gradio_file_path(values[0])
        if os.path.splitext(zip_path)[1].lower() != ".zip":
            raise gr.Error(f"The {label} archive must be a .zip file.")

        target_dir = os.path.join(extract_root, slugify(label))
        ensure_dir(target_dir)
        target_root = os.path.abspath(target_dir)
        extracted = []

        with zipfile.ZipFile(zip_path, "r") as archive:
            for info in archive.infolist():
                if info.is_dir():
                    continue

                member_name = info.filename.replace("\\", "/")
                ext = os.path.splitext(member_name)[1].lower()
                if ext not in BATCH_IMAGE_EXTENSIONS:
                    continue

                base_name = os.path.basename(member_name)
                if not base_name:
                    continue

                safe_name = f"{len(extracted):05d}_{slugify(os.path.splitext(base_name)[0])}{ext}"
                destination = os.path.abspath(os.path.join(target_root, safe_name))

                if os.path.commonpath([target_root, destination]) != target_root:
                    raise gr.Error(f"Unsafe path found in {label} ZIP.")

                with archive.open(info, "r") as source, open(destination, "wb") as target:
                    shutil.copyfileobj(source, target)

                extracted.append(destination)

        if not extracted:
            raise gr.Error(f"No supported images were found in the {label} ZIP.")

        return extracted


    def collect_batch_image_paths(
        multiple_value: Any,
        directory_value: Any,
        zip_value: Any,
        extract_root: str,
        label: str,
    ) -> List[str]:
        candidates: List[str] = []

        # gr.File(file_count='multiple') and file_count='directory'
        # both return lists of temporary file paths.
        for value in normalize_uploaded_values(multiple_value):
            candidates.extend(list_images_recursive(resolve_gradio_file_path(value)))

        for value in normalize_uploaded_values(directory_value):
            try:
                path = resolve_gradio_file_path(value)
                candidates.extend(list_images_recursive(path))
            except FileNotFoundError:
                raw_path = str(value or "").strip()
                if raw_path and os.path.isdir(raw_path):
                    candidates.extend(list_images_recursive(raw_path))
                else:
                    raise

        candidates.extend(
            extract_image_zip(
                zip_value=zip_value,
                extract_root=extract_root,
                label=label,
            )
        )

        unique_paths = []
        seen = set()
        for path in candidates:
            normalized = os.path.abspath(path)
            if normalized in seen:
                continue
            seen.add(normalized)
            unique_paths.append(normalized)

        if not unique_paths:
            raise gr.Error(
                f"No {label} images were found. Upload a folder, select multiple images, or upload a ZIP."
            )

        return sorted(unique_paths)


    def get_batch_grid_font(size: int = 16):
        candidates = [
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
        ]
        for font_path in candidates:
            if os.path.exists(font_path):
                try:
                    return ImageFont.truetype(font_path, size=size)
                except Exception:
                    pass
        return ImageFont.load_default()


    def create_batch_grid(
        items: List[Dict[str, Any]],
        save_path: str,
        columns: int = 3,
        tile_size: int = 300,
        caption_height: int = 54,
    ) -> Image.Image:
        if not items:
            raise ValueError("No images are available for the grid.")

        columns = max(1, int(columns))
        rows = int(math.ceil(len(items) / columns))
        canvas = Image.new(
            "RGB",
            (columns * tile_size, rows * (tile_size + caption_height)),
            "white",
        )
        draw = ImageDraw.Draw(canvas)
        font = get_batch_grid_font(16)

        for index, item in enumerate(items):
            row = index // columns
            col = index % columns
            x = col * tile_size
            y = row * (tile_size + caption_height)

            image = item["image"].convert("RGB")
            fitted = ImageOps.contain(
                image,
                (tile_size, tile_size),
                method=Image.Resampling.LANCZOS,
            )
            tile = Image.new("RGB", (tile_size, tile_size), "white")
            tile.paste(
                fitted,
                ((tile_size - fitted.width) // 2, (tile_size - fitted.height) // 2),
            )
            canvas.paste(tile, (x, y))

            label = str(item.get("label", ""))
            if len(label) > 42:
                label = label[:39] + "..."
            draw.text(
                (x + 8, y + tile_size + 14),
                label,
                fill="black",
                font=font,
            )

        canvas.save(save_path)
        return canvas


    def generate_batch_all_pairs(
        batch_subject_files,
        batch_subject_directory,
        batch_subject_zip,
        batch_background_files,
        batch_background_directory,
        batch_background_zip,
        use_filename_as_animal,
        batch_seed_mode,
        batch_grid_columns,
        *args,
        progress=gr.Progress(),
    ):
        ui_base = build_ui_dict_from_args(args)
        apply_ui_params_to_runner(runner, ui_base)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        batch_name = f"BATCH_{timestamp}"
        batch_root = os.path.join(runner.cfg.out_dir, batch_name)
        ensure_dir(batch_root)

        extract_root = os.path.join(batch_root, "_uploaded_archives")
        ensure_dir(extract_root)

        subject_paths = collect_batch_image_paths(
            multiple_value=batch_subject_files,
            directory_value=batch_subject_directory,
            zip_value=batch_subject_zip,
            extract_root=extract_root,
            label="subjects",
        )
        background_paths = collect_batch_image_paths(
            multiple_value=batch_background_files,
            directory_value=batch_background_directory,
            zip_value=batch_background_zip,
            extract_root=extract_root,
            label="backgrounds",
        )

        total_pairs = len(subject_paths) * len(background_paths)
        base_seed = int(ui_base["seed"])
        pair_index = 0

        gallery_items = []
        global_grid_items: List[Dict[str, Any]] = []
        records = []
        failures = []
        max_global_grid_items = 120

        print("\\n" + "=" * 76)
        print(
            f"[BATCH] {len(subject_paths)} subjects x "
            f"{len(background_paths)} backgrounds = {total_pairs} pairs"
        )
        print("=" * 76)

        for subject_index, subject_path in enumerate(subject_paths, start=1):
            subject_stem = os.path.splitext(os.path.basename(subject_path))[0]
            subject_slug = slugify(subject_stem)
            subject_rel_dir = os.path.join(
                batch_name,
                f"SUBJECT_{subject_index:03d}_{subject_slug}",
            )
            subject_abs_dir = os.path.join(runner.cfg.out_dir, subject_rel_dir)
            ensure_dir(subject_abs_dir)
            per_subject_grid_items: List[Dict[str, Any]] = []

            for background_index, background_path in enumerate(background_paths, start=1):
                pair_index += 1
                progress(
                    pair_index / max(total_pairs, 1),
                    desc=(
                        f"{pair_index}/{total_pairs} | "
                        f"{os.path.basename(subject_path)} × "
                        f"{os.path.basename(background_path)}"
                    ),
                )

                background_stem = os.path.splitext(os.path.basename(background_path))[0]
                background_slug = slugify(background_stem)

                ui_pair = dict(ui_base)
                ui_pair["path_animal"] = subject_path
                ui_pair["path_bg"] = background_path

                if bool(use_filename_as_animal):
                    ui_pair["animal_name"] = subject_stem.replace("_", " ").replace("-", " ")

                mode = str(batch_seed_mode or "same").strip().lower()
                if bool(ui_pair.get("randomize_seed_each_generate", False)) or mode == "random":
                    pair_seed = fresh_seed()
                elif mode == "increment":
                    pair_seed = base_seed + pair_index - 1
                else:
                    pair_seed = base_seed
                ui_pair["seed"] = int(pair_seed)

                apply_ui_params_to_runner(runner, ui_pair)
                job = build_job_from_ui(ui_pair)
                job.name = os.path.join(
                    subject_rel_dir,
                    f"BG_{background_index:03d}_{background_slug}_seed{pair_seed}",
                )

                print(
                    f"[BATCH {pair_index:03d}/{total_pairs}] "
                    f"subject={os.path.basename(subject_path)} | "
                    f"background={os.path.basename(background_path)} | "
                    f"seed={pair_seed}"
                )

                try:
                    imgs, paths = runner.run_one(job, log_every=0, show_preview=False)
                    output_path = paths[0]
                    output_image = imgs[0].copy()
                    caption = f"{subject_stem} | {background_stem} | seed {pair_seed}"

                    gallery_items.append((output_path, caption))
                    per_subject_grid_items.append(
                        {"image": output_image.copy(), "label": background_stem}
                    )
                    if len(global_grid_items) < max_global_grid_items:
                        global_grid_items.append(
                            {"image": output_image.copy(), "label": caption}
                        )

                    records.append(
                        {
                            "pair_index": pair_index,
                            "subject_index": subject_index,
                            "background_index": background_index,
                            "subject": subject_path,
                            "background": background_path,
                            "animal_name": job.animals,
                            "seed": int(pair_seed),
                            "output": output_path,
                        }
                    )

                except Exception as error:
                    failed_dir = os.path.join(
                        subject_abs_dir,
                        f"BG_{background_index:03d}_{background_slug}_FAILED",
                    )
                    ensure_dir(failed_dir)
                    error_path = os.path.join(failed_dir, "error.txt")
                    with open(error_path, "w", encoding="utf-8") as f:
                        f.write(traceback.format_exc())

                    failures.append(
                        {
                            "pair_index": pair_index,
                            "subject": subject_path,
                            "background": background_path,
                            "error_type": type(error).__name__,
                            "error": str(error),
                            "error_file": error_path,
                        }
                    )
                    print(f"[BATCH ERROR] {type(error).__name__}: {error}")

                finally:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            if per_subject_grid_items:
                per_subject_grid_path = os.path.join(subject_abs_dir, "grid_all_backgrounds.png")
                create_batch_grid(
                    per_subject_grid_items,
                    per_subject_grid_path,
                    columns=int(batch_grid_columns),
                )

        if not records:
            raise gr.Error(
                "No subject-background pair completed successfully. "
                f"Check error files in: {batch_root}"
            )

        global_grid_path = os.path.join(batch_root, "global_grid.png")
        create_batch_grid(
            global_grid_items,
            global_grid_path,
            columns=int(batch_grid_columns),
        )

        metadata = {
            "batch_name": batch_name,
            "created_at": datetime.now().isoformat(),
            "subject_count": len(subject_paths),
            "background_count": len(background_paths),
            "requested_pairs": total_pairs,
            "successful_pairs": len(records),
            "failed_pairs": len(failures),
            "global_grid_items": len(global_grid_items),
            "global_grid_limit": max_global_grid_items,
            "settings": ui_base,
            "batch_options": {
                "use_filename_as_animal": bool(use_filename_as_animal),
                "seed_mode": str(batch_seed_mode),
                "grid_columns": int(batch_grid_columns),
            },
            "results": records,
            "failures": failures,
        }
        save_json(os.path.join(batch_root, "batch_metadata.json"), metadata)

        zip_path = shutil.make_archive(batch_root, "zip", root_dir=batch_root)

        status = (
            "### Batch completed\\n\\n"
            f"- **Subjects:** `{len(subject_paths)}`\\n"
            f"- **Backgrounds:** `{len(background_paths)}`\\n"
            f"- **Requested pairs:** `{total_pairs}`\\n"
            f"- **Successful:** `{len(records)}`\\n"
            f"- **Failed:** `{len(failures)}`\\n"
            f"- **Output folder:** `{batch_root}`\\n"
            f"- **ZIP:** `{zip_path}`"
        )

        return gallery_items, global_grid_path, zip_path, status


    with gr.Blocks(theme=gr.themes.Base()) as demo:
        gr.Markdown("## Camouflage Master Tuning UI — Direct Layout on Canny / SoftEdge")

        with gr.Row():
            with gr.Column(scale=5):
                with gr.Tab("Inputs & Prompts"):
                    gr.Markdown(
                        "Upload images directly if you want to provide custom inputs. "
                        "If no image is uploaded, the code will use the two path fields as in the previous version."
                    )
                    subject_upload = gr.Image(label="Upload Subject Image", type="pil")
                    bg_upload = gr.Image(label="Upload Background Image", type="pil")

                    path_animal = gr.Textbox(value=UI_DEFAULTS["path_animal"], label="Animal Image Path / Subject Folder")
                    path_bg = gr.Textbox(value=UI_DEFAULTS["path_bg"], label="Background Image Path / Background Folder")
                    animal_name = gr.Textbox(value=UI_DEFAULTS["animal_name"], label="Animal Name")
                    prompt_bg = gr.Textbox(value=UI_DEFAULTS["prompt_bg"], label="Prompt Background", lines=2)
                    prompt_sub = gr.Textbox(value=UI_DEFAULTS["prompt_sub"], label="Prompt Subject Template", lines=2)
                    negative_prompt = gr.Textbox(value=UI_DEFAULTS["negative_prompt"], label="Negative Prompt", lines=2)

                with gr.Tab("Core Generation"):
                    steps = gr.Slider(1, 100, value=UI_DEFAULTS["steps"], step=1, label="Steps")
                    with gr.Row():
                        seed = gr.Number(value=UI_DEFAULTS["seed"], label="Seed", precision=0)
                        randomize_seed_each_generate = gr.Checkbox(value=UI_DEFAULTS["randomize_seed_each_generate"], label="Random Seed Each Generate")
                    bg_strength = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["bg_strength"], step=0.01, label="BG Strength")
                    guidance_bg = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_bg"], step=0.1, label="Guidance BG")
                    guidance_sub = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_sub"], step=0.1, label="Guidance Sub")
                    sub_init_noise = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["sub_init_noise"], step=0.01, label="Sub Init Noise")
                    couple_mode = gr.Dropdown(
                        choices=["none", "bg_only", "both"],
                        value=UI_DEFAULTS["couple_mode"],
                        label="Couple Mode",
                    )

                with gr.Tab("🎭 Edges & Mask"):
                    with gr.Row():
                        canny_low = gr.Slider(0, 255, value=UI_DEFAULTS["canny_low"], step=1, label="Canny Low")
                        canny_high = gr.Slider(0, 255, value=UI_DEFAULTS["canny_high"], step=1, label="Canny High")
                    canny_blur_ks = gr.Slider(1, 15, value=UI_DEFAULTS["canny_blur_ks"], step=2, label="Canny Blur KS")
                    use_broken_edges = gr.Checkbox(value=UI_DEFAULTS["use_broken_edges"], label="Use Broken Edges")
                    broken_drop_prob = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["broken_drop_prob"], step=0.01, label="Broken Drop Prob")

                    gr.Markdown("---")
                    mask_mode = gr.Dropdown(
                        choices=["silhouette", "outline", "composite", "softedge"],
                        value=UI_DEFAULTS["mask_mode"],
                        label="Mask Mode",
                    )
                    mask_dilate = gr.Slider(1, 31, value=UI_DEFAULTS["mask_dilate"], step=2, label="Mask Dilate")
                    mask_close_ks = gr.Slider(1, 31, value=UI_DEFAULTS["mask_close_ks"], step=2, label="Mask Close KS")
                    mask_blur = gr.Slider(1, 51, value=UI_DEFAULTS["mask_blur"], step=2, label="Mask Blur")
                    mask_ring_k = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_k"], step=2, label="Mask Ring K")
                    mask_ring_blur = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_blur"], step=2, label="Mask Ring Blur")
                    mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["mask_gamma"], step=0.1, label="Mask Gamma (old modes)")
                    mask_auto_invert = gr.Checkbox(value=UI_DEFAULTS["mask_auto_invert"], label="Mask Auto Invert")

                    gr.Markdown("#### Composite Mask Params (old modes)")
                    outer_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["outer_weight"], step=0.05, label="Outer Weight")
                    feature_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["feature_weight"], step=0.05, label="Feature Weight")
                    fill_weight = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["fill_weight"], step=0.05, label="Fill Weight")
                    feature_thresh = gr.Slider(0, 255, value=UI_DEFAULTS["feature_thresh"], step=1, label="Feature Thresh")
                    feature_open_ks = gr.Slider(1, 15, value=UI_DEFAULTS["feature_open_ks"], step=2, label="Feature Open KS")
                    feature_blur = gr.Slider(1, 21, value=UI_DEFAULTS["feature_blur"], step=2, label="Feature Blur")
                    fill_blur = gr.Slider(1, 31, value=UI_DEFAULTS["fill_blur"], step=2, label="Fill Blur")

                    gr.Markdown("#### SoftEdge → Kernel Gradient (for subject layout + final mask)")
                    use_soft_grad_mask = gr.Checkbox(value=UI_DEFAULTS["use_soft_grad_mask"], label="Use Kernel Gradient for Layout/Mask")
                    soft_grad_dilation_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_dilation_k"], step=2, label="Soft Grad Dilation K")
                    soft_grad_blur_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_blur_k"], step=2, label="Soft Grad Blur K")
                    soft_grad_kernel_shape = gr.Dropdown(
                        choices=["ellipse", "rect", "cross"],
                        value=UI_DEFAULTS["soft_grad_kernel_shape"],
                        label="Soft Grad Kernel Shape",
                    )
                    soft_grad_iterations = gr.Slider(1, 5, value=UI_DEFAULTS["soft_grad_iterations"], step=1, label="Soft Grad Iterations")
                    soft_grad_use_skeleton = gr.Checkbox(value=UI_DEFAULTS["soft_grad_use_skeleton"], label="Soft Grad Use Skeleton")
                    soft_grad_skeleton_stage = gr.Dropdown(
                        choices=["pre", "post"],
                        value=UI_DEFAULTS["soft_grad_skeleton_stage"],
                        label="Soft Grad Skeleton Stage",
                    )
                    soft_grad_post_skel_dilate_k = gr.Slider(0, 11, value=UI_DEFAULTS["soft_grad_post_skel_dilate_k"], step=1, label="Post Skeleton Dilate K")

                    gr.Markdown("#### Final Soft Mask Remap")
                    soft_mask_blur = gr.Slider(1, 31, value=UI_DEFAULTS["soft_mask_blur"], step=2, label="Soft Mask Blur")
                    soft_mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["soft_mask_gamma"], step=0.1, label="Soft Mask Gamma")
                    soft_mask_invert = gr.Checkbox(value=UI_DEFAULTS["soft_mask_invert"], label="Soft Mask Invert")
                    soft_mask_floor = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_floor"], step=0.01, label="Soft Mask Floor")
                    soft_mask_ceiling = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_ceiling"], step=0.01, label="Soft Mask Ceiling")

                    gr.Markdown("#### Auto Attention Compensation")
                    gr.Markdown("Enable this option to automatically compensate the soft-mask strength according to the effective silhouette size after dragging or resizing. When the object is smaller than the reference scale, the system reduces gamma/ceiling and slightly reduces blur so the subject remains visible.")
                    auto_attention_compensation = gr.Checkbox(value=UI_DEFAULTS["auto_attention_compensation"], label="Auto Attention Compensation")
                    with gr.Row():
                        attention_scale_ref = gr.Slider(0.05, 0.95, value=UI_DEFAULTS["attention_scale_ref"], step=0.01, label="Reference Scale")
                        btn_capture_scale_ref = gr.Button("Use Current Layout as Ref Scale")
                    with gr.Row():
                        attention_gamma_ref = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["attention_gamma_ref"], step=0.05, label="Reference Gamma")
                        attention_ceiling_ref = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["attention_ceiling_ref"], step=0.01, label="Reference Ceiling")
                    attention_blur_ref = gr.Slider(1, 31, value=UI_DEFAULTS["attention_blur_ref"], step=2, label="Reference Blur")

                with gr.Tab("⏱️ Control & Blending"):
                    canny_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["canny_scale"], step=0.01, label="Control Scale: Canny")
                    soft_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["soft_scale"], step=0.01, label="Control Scale: Soft Edge")
                    start_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["start_frac"], step=0.01, label="Start Frac Control")
                    end_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["end_frac"], step=0.01, label="End Frac Control")
                    gate_kind = gr.Dropdown(choices=["cosine", "linear"], value=UI_DEFAULTS["gate_kind"], label="Control Gate Kind")

                    gr.Markdown("---")
                    blend_profile = gr.Dropdown(choices=["decay", "ramp"], value=UI_DEFAULTS["blend_profile"], label="Blend Profile")
                    blend_start = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_start"], step=0.01, label="Blend Start Frac")
                    blend_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_end"], step=0.01, label="Blend End Frac")
                    alpha_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["alpha_end"], step=0.01, label="Blend Alpha End")

                    gr.Markdown("#### Subject Layout Placement (applied from Canny / SoftEdge)")
                    gr.Markdown("Use the drag-and-drop canvas on the right as the main layout control. These fields control the subject position, scale, and rotation directly from the Canny and SoftEdge stage, and are kept for config load/save.")
                    blend_mask_scale = gr.Slider(0.10, 2.50, value=UI_DEFAULTS["blend_mask_scale"], step=0.01, label="Layout Scale", elem_id="blend-mask-scale")
                    blend_mask_offset_x = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_x"], step=0.01, label="Layout Offset X (+ = right)", elem_id="blend-mask-offset-x")
                    blend_mask_offset_y = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_y"], step=0.01, label="Layout Offset Y (+ = down)", elem_id="blend-mask-offset-y")
                    blend_mask_rotation_deg = gr.Slider(-180.0, 180.0, value=UI_DEFAULTS["blend_mask_rotation_deg"], step=1.0, label="Layout Rotation (deg)", elem_id="blend-mask-rotation-deg")

                with gr.Tab("Config JSON"):
                    config_path = gr.Textbox(label="Config JSON Path", placeholder="/kaggle/working/outputs_camouflage_softgradmask/ui_config_20260314_150633.json")
                    btn_load = gr.Button("Load Config JSON")

                with gr.Row():
                    btn_generate = gr.Button("Generate Image", variant="primary", scale=2)
                    btn_save = gr.Button("Save Config", scale=1)

            with gr.Column(scale=4):
                out_img = gr.Image(label="Output Image", type="pil")
                out_score = gr.Textbox(label="Hidden Score", interactive=False)
                status_box = gr.Textbox(label="System Status", interactive=False)

                gr.Markdown("### Calculated Output Metrics")
                out_metrics_report = gr.HTML()
                out_metrics_json = gr.Code(label="Calculated Metrics (JSON)", language="json", interactive=False)

                gr.Markdown("### Interactive Subject Layout")
                btn_refresh_layout = gr.Button("Refresh Interactive Canvas")
                layout_editor = gr.HTML(value=safe_initial_layout_html())

                gr.Markdown("### Debug Views")
                with gr.Row():
                    dbg_bg_original = gr.Image(label="Original BG", type="pil")
                    dbg_input = gr.Image(label="Input Image", type="pil")
                with gr.Row():
                    dbg_canny = gr.Image(label="Canny Hint", type="pil")
                    dbg_soft = gr.Image(label="Soft Hint (Raw)", type="pil")
                with gr.Row():
                    dbg_sil = gr.Image(label="Silhouette Mask", type="pil")
                    dbg_feat = gr.Image(label="Feature Mask", type="pil")
                with gr.Row():
                    dbg_comp = gr.Image(label="Composite HW Mask", type="pil")
                    dbg_soft_for_mask = gr.Image(label="Soft For Mask (After Kernel Gradient)", type="pil")
                with gr.Row():
                    dbg_soft_hw = gr.Image(label="SoftEdge HW Mask", type="pil")
                    dbg_latent_mask = gr.Image(label="Final Latent Mask (after preprocess layout)", type="pil")

                gr.Markdown("### Stage 2 Breakdown Views")
                with gr.Row():
                    dbg_contour_agg = gr.Image(label="Contour Aggregation", type="pil")
                    dbg_region_filling = gr.Image(label="Region Filling", type="pil")
                with gr.Row():
                    dbg_canny_smoothing = gr.Image(label="Smoothing (Canny Path)", type="pil")
                    dbg_aux_support = gr.Image(label="Auxiliary Support S", type="pil")
                with gr.Row():
                    dbg_soft_gray = gr.Image(label="Soft Gray", type="pil")
                    dbg_soft_smoothing = gr.Image(label="Smoothing (Soft Path)", type="pil")
                with gr.Row():
                    dbg_soft_remap = gr.Image(label="Nonlinear Remapping", type="pil")
                    dbg_soft_clip = gr.Image(label="Clip to [0,1]", type="pil")

                gr.Markdown("### Stage 2 Values / Diagnostics")
                dbg_stage2_report = gr.HTML()
                dbg_stage2_json = gr.Code(label="Stage 2 Values (JSON)", language="json", interactive=False)


        gr.Markdown("---")
        gr.Markdown("## Batch: Subject Folder × Background Folder")
        gr.Markdown(
            "Chọn nguyên thư mục để upload toàn bộ ảnh. Bạn cũng có thể chọn nhiều "
            "file riêng lẻ hoặc upload một file ZIP. Tất cả subject sẽ chạy với tất cả background."
        )

        with gr.Row():
            with gr.Column(scale=1):
                with gr.Accordion("Subject inputs", open=True):
                    batch_subject_directory = gr.File(
                        label="Upload one SUBJECT folder",
                        file_count="directory",
                        file_types=["image"],
                        type="filepath",
                    )
                    batch_subject_files = gr.File(
                        label="Or select multiple SUBJECT images",
                        file_count="multiple",
                        file_types=["image"],
                        type="filepath",
                    )
                    batch_subject_zip = gr.File(
                        label="Or upload one SUBJECT ZIP",
                        file_count="single",
                        file_types=[".zip"],
                        type="filepath",
                    )

                with gr.Accordion("Background inputs", open=True):
                    batch_background_directory = gr.File(
                        label="Upload one BACKGROUND folder",
                        file_count="directory",
                        file_types=["image"],
                        type="filepath",
                    )
                    batch_background_files = gr.File(
                        label="Or select multiple BACKGROUND images",
                        file_count="multiple",
                        file_types=["image"],
                        type="filepath",
                    )
                    batch_background_zip = gr.File(
                        label="Or upload one BACKGROUND ZIP",
                        file_count="single",
                        file_types=[".zip"],
                        type="filepath",
                    )

                batch_use_filename_as_animal = gr.Checkbox(
                    value=True,
                    label="Use each subject filename as Animal Name",
                )
                batch_seed_mode = gr.Dropdown(
                    choices=[
                        ("Same seed for every pair", "same"),
                        ("Increment seed for each pair", "increment"),
                        ("Random seed for each pair", "random"),
                    ],
                    value="same",
                    label="Batch Seed Mode",
                )
                batch_grid_columns = gr.Dropdown(
                    choices=[2, 3, 4, 5],
                    value=3,
                    label="Grid Columns",
                )
                btn_batch_generate = gr.Button(
                    "Generate All Subjects × All Backgrounds",
                    variant="primary",
                )

            with gr.Column(scale=2):
                batch_gallery = gr.Gallery(
                    label="Batch Results",
                    columns=3,
                    rows=2,
                    height="auto",
                    object_fit="contain",
                    allow_preview=True,
                )
                batch_global_grid = gr.Image(
                    label="Global Grid",
                    type="filepath",
                    height=680,
                )
                batch_zip = gr.File(label="Download Full Batch ZIP")
                batch_status = gr.Markdown()

        all_inputs = [
            path_animal,
            path_bg,
            animal_name,
            prompt_bg,
            prompt_sub,
            negative_prompt,
            steps,
            seed,
            randomize_seed_each_generate,
            bg_strength,
            guidance_bg,
            guidance_sub,
            sub_init_noise,
            couple_mode,
            canny_low,
            canny_high,
            canny_blur_ks,
            use_broken_edges,
            broken_drop_prob,
            mask_mode,
            mask_dilate,
            mask_close_ks,
            mask_blur,
            mask_ring_k,
            mask_ring_blur,
            mask_gamma,
            mask_auto_invert,
            outer_weight,
            feature_weight,
            fill_weight,
            feature_thresh,
            feature_open_ks,
            feature_blur,
            fill_blur,
            use_soft_grad_mask,
            soft_grad_dilation_k,
            soft_grad_blur_k,
            soft_grad_kernel_shape,
            soft_grad_iterations,
            soft_grad_use_skeleton,
            soft_grad_skeleton_stage,
            soft_grad_post_skel_dilate_k,
            soft_mask_blur,
            soft_mask_gamma,
            soft_mask_invert,
            soft_mask_floor,
            soft_mask_ceiling,
            canny_scale,
            soft_scale,
            start_frac,
            end_frac,
            gate_kind,
            blend_profile,
            blend_start,
            blend_end,
            alpha_end,
            auto_attention_compensation,
            attention_scale_ref,
            attention_gamma_ref,
            attention_ceiling_ref,
            attention_blur_ref,
            blend_mask_scale,
            blend_mask_offset_x,
            blend_mask_offset_y,
            blend_mask_rotation_deg,
        ]

        btn_generate.click(
            fn=generate_preview,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[
                seed,
                out_img,
                dbg_bg_original,
                dbg_input,
                dbg_canny,
                dbg_soft,
                dbg_sil,
                dbg_feat,
                dbg_comp,
                dbg_soft_for_mask,
                dbg_soft_hw,
                dbg_latent_mask,
                dbg_contour_agg,
                dbg_region_filling,
                dbg_canny_smoothing,
                dbg_aux_support,
                dbg_soft_gray,
                dbg_soft_smoothing,
                dbg_soft_remap,
                dbg_soft_clip,
                out_score,
                status_box,
                out_metrics_report,
                out_metrics_json,
                dbg_stage2_report,
                dbg_stage2_json,
            ],
        )

        btn_capture_scale_ref.click(
            fn=capture_current_scale_reference,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[attention_scale_ref, status_box],
        )

        btn_save.click(
            fn=save_current_config,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=status_box,
        )

        btn_refresh_layout.click(
            fn=refresh_layout_editor,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[layout_editor, status_box],
        )

        btn_load.click(
            fn=load_config_to_ui,
            inputs=[config_path],
            outputs=all_inputs + [status_box, layout_editor],
        )


        btn_batch_generate.click(
            fn=generate_batch_all_pairs,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[
                batch_subject_files,
                batch_subject_directory,
                batch_subject_zip,
                batch_background_files,
                batch_background_directory,
                batch_background_zip,
                batch_use_filename_as_animal,
                batch_seed_mode,
                batch_grid_columns,
            ] + all_inputs,
            outputs=[
                batch_gallery,
                batch_global_grid,
                batch_zip,
                batch_status,
            ],
        )

    demo.launch(share=True if os.path.exists("/kaggle/working") else True, debug=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3a949867c7e7fd3051.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
# ============================================================
# Kaggle dependency setup
# If you run this in a Kaggle notebook, you can also run this once
# in a separate cell:
# !pip -q install -U diffusers transformers accelerate safetensors opencv-python controlnet-aux gradio scikit-image
# ============================================================

#NEW ENGLISH
# ============================================================
# Camouflage Latent Blending + Dual ControlNet + Gradio UI
# Clean version with JSON config load/save support
# Compatible with legacy ui_config_*.json (params as positional list)
# ============================================================
import os
import re
import math
import json
import time
import base64
import io
import html as html_lib
import contextlib
import shutil
from dataclasses import dataclass
from datetime import datetime
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)


# ============================================================
# 0) Small utils
# ============================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

# You can paste image from clipboard 
KAGGLE_ANIMALS_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/animals" 
KAGGLE_BG_DIR = "/kaggle/input/datasets/dangkhoi3107/thesis/bg"
DEFAULT_OUT_DIR = "/kaggle/working/outputs_camouflage_softgradmask" if os.path.exists("/kaggle/working") else "./outputs_camouflage_softgradmask"

def list_image_files(path: str) -> List[str]:
    path = str(path or "").strip()
    if not path or not os.path.exists(path):
        return []
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    if os.path.isfile(path):
        return [path] if os.path.splitext(path)[1].lower() in exts else []
    files = []
    for name in sorted(os.listdir(path)):
        full = os.path.join(path, name)
        if os.path.isfile(full) and os.path.splitext(name)[1].lower() in exts:
            files.append(full)
    return files


def resolve_image_path(path_or_dir: Optional[str], fallback_dir: Optional[str] = None) -> Optional[str]:
    candidate = str(path_or_dir or "").strip()
    if candidate:
        if os.path.isfile(candidate):
            return candidate
        if os.path.isdir(candidate):
            files = list_image_files(candidate)
            if files:
                return files[0]
    fallback = str(fallback_dir or "").strip()
    if fallback:
        if os.path.isfile(fallback):
            return fallback
        if os.path.isdir(fallback):
            files = list_image_files(fallback)
            if files:
                return files[0]
    return None


def _odd(k: int) -> int:
    k = int(k)
    return k if k % 2 == 1 else k + 1


def _ensure_odd(k: int, minimum: int = 1) -> int:
    k = max(int(k), int(minimum))
    return k if k % 2 == 1 else k + 1


def _get_kernel(shape: str, k: int) -> np.ndarray:
    k = _ensure_odd(k, 1)
    shape = str(shape).lower().strip()

    if shape == "rect":
        return cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    if shape == "cross":
        return cv2.getStructuringElement(cv2.MORPH_CROSS, (k, k))
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def seed_everything(seed: int) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def slugify(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"[^a-z0-9\-_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "item"


def format_prompt(template: str, animal: str) -> str:
    return (template or "").format(animals=animal, animal=animal)


def save_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def fresh_seed() -> int:
    seed = int.from_bytes(os.urandom(8), "big") % 2147483647
    return seed if seed > 0 else 1


def normalize_0_255(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-6:
        return np.zeros_like(x, dtype=np.uint8)
    x = (x - mn) / (mx - mn)
    return (x * 255.0).clip(0, 255).astype(np.uint8)


def gray01_to_u8(x: np.ndarray) -> np.ndarray:
    return (np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)


def pil_to_data_uri(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    encoded = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/{fmt.lower()};base64,{encoded}"


def make_checkerboard(size: Tuple[int, int], tile: int = 32) -> Image.Image:
    w, h = size
    tile = max(int(tile), 4)
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    c0 = np.array([38, 38, 42], dtype=np.uint8)
    c1 = np.array([58, 58, 64], dtype=np.uint8)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            use_c0 = ((x // tile) + (y // tile)) % 2 == 0
            arr[y : y + tile, x : x + tile] = c0 if use_c0 else c1
    return Image.fromarray(arr, mode="RGB")


def load_rgb_preview_image(path: Optional[str], size: Tuple[int, int]) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved and os.path.exists(resolved):
        return Image.open(resolved).convert("RGB").resize(size, Image.LANCZOS)
    return make_checkerboard(size)


def mask_preview_to_rgba(mask_pil: Image.Image, alpha_scale: float = 0.82) -> Image.Image:
    gray = np.array(mask_pil.convert("L"), dtype=np.uint8)
    edges = cv2.Canny(gray, 32, 128)

    rgba = np.zeros((gray.shape[0], gray.shape[1], 4), dtype=np.uint8)
    rgba[..., 0] = 255
    rgba[..., 1] = 255
    rgba[..., 2] = 255
    rgba[..., 3] = np.clip(gray.astype(np.float32) * float(alpha_scale), 0, 255).astype(np.uint8)

    outline = edges > 0
    rgba[outline, 0] = 255
    rgba[outline, 1] = 96
    rgba[outline, 2] = 96
    rgba[outline, 3] = 255
    return Image.fromarray(rgba, mode="RGBA")


def build_interactive_layout_html(
    bg_img: Image.Image,
    overlay_img: Image.Image,
    width: int,
    height: int,
    init_scale: float,
    init_offset_x: float,
    init_offset_y: float,
    init_rotation_deg: float,
) -> str:
    bg_uri = pil_to_data_uri(bg_img.convert("RGB"), fmt="PNG")
    overlay_uri = pil_to_data_uri(overlay_img.convert("RGBA"), fmt="PNG")
    canvas_w = int(width)
    canvas_h = int(height)
    display_w = min(max(canvas_w, 320), 640)
    display_h = max(int(display_w * canvas_h / max(canvas_w, 1)), 220)

    iframe_id = f"camo-placement-frame-{int(time.time() * 1000)}"

    inner_html = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <style>
    html, body { margin: 0; padding: 0; background: transparent; font-family: ui-sans-serif, system-ui, sans-serif; color: #e5e7eb; overflow: hidden; }
    .wrap { border: 1px solid rgba(255,255,255,0.08); border-radius: 14px; padding: 12px; background: #111827; color: #e5e7eb; box-sizing: border-box; width: 100%; min-height: 100vh; }
    .hint { font-size: 13px; line-height: 1.5; color: #cbd5e1; margin-bottom: 10px; }
    .canvas-wrap { position: relative; width: __DISPLAY_W__px; max-width: 100%; }
    canvas { width: 100%; height: auto; display: block; border-radius: 12px; cursor: grab; background: #0f172a; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.08); }
    canvas.dragging, canvas.resizing, canvas.rotating { cursor: grabbing; }
    .toolbar { margin-top: 10px; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }
    button { border: 0; border-radius: 10px; padding: 8px 12px; background: #1f2937; color: #f9fafb; cursor: pointer; }
    button:hover { background: #374151; }
    .stats { font-size: 12px; color: #cbd5e1; }
    .badge { display: inline-flex; align-items: center; gap: 6px; padding: 6px 10px; border-radius: 999px; background: rgba(255,255,255,0.06); }
    .status { font-size: 12px; color: #93c5fd; margin-top: 8px; }
  </style>
</head>
<body>
  <div class="wrap">
    <div class="hint">Drag the box to move the subject. Drag the blue handle at the bottom-right corner to resize, drag the yellow handle above the box to rotate, and use the mouse wheel to zoom. The Layout sliders on the left panel are synchronized <b>in real time</b>.</div>
    <div class="canvas-wrap">
      <canvas id="canvas" width="__CANVAS_W__" height="__CANVAS_H__"></canvas>
    </div>
    <div class="toolbar">
      <button type="button" data-action="center">Center</button>
      <button type="button" data-action="fit">Fit 0.7</button>
      <button type="button" data-action="reset">Reset</button>
      <span class="badge stats">scale=<span data-key="scale"></span></span>
      <span class="badge stats">offsetX=<span data-key="offsetX"></span></span>
      <span class="badge stats">offsetY=<span data-key="offsetY"></span></span>
      <span class="badge stats">rotation=<span data-key="rotationDeg"></span>°</span>
    </div>
    <div class="status" data-role="status">Canvas state is ready.</div>
  </div>
  <script>
    (() => {
      const canvas = document.getElementById('canvas');
      const ctx = canvas.getContext('2d');
      const statusEl = document.querySelector('[data-role="status"]');
      const bg = new Image();
      const overlay = new Image();
      bg.src = '__BG_URI__';
      overlay.src = '__OVERLAY_URI__';

      const clamp = (v, lo, hi) => Math.min(Math.max(v, lo), hi);
      const wrapDeg = (deg) => {
        let out = Number(deg) || 0;
        while (out > 180) out -= 360;
        while (out <= -180) out += 360;
        return out;
      };
      const wrapRad = (rad) => {
        let out = Number(rad) || 0;
        while (out > Math.PI) out -= Math.PI * 2;
        while (out <= -Math.PI) out += Math.PI * 2;
        return out;
      };
      const nearlyEqual = (a, b, eps=1e-4) => Math.abs((Number(a)||0) - (Number(b)||0)) <= eps;

      const state = {
        scale: clamp(Number('__INIT_SCALE__') || 1, 0.10, 2.50),
        offsetX: clamp(Number('__INIT_OFFSET_X__') || 0, -1.00, 1.00),
        offsetY: clamp(Number('__INIT_OFFSET_Y__') || 0, -1.00, 1.00),
        rotationDeg: wrapDeg(Number('__INIT_ROTATION_DEG__') || 0),
      };

      let parentDoc = null;
      try { parentDoc = window.parent && window.parent.document ? window.parent.document : null; } catch (_) { parentDoc = null; }

      function setStatus(msg) {
        if (statusEl) statusEl.textContent = msg;
      }

      function syncOneSlider(elemId, value) {
        if (!parentDoc) return false;
        const host = parentDoc.getElementById(elemId);
        if (!host) return false;
        const inputs = host.querySelectorAll('input');
        if (!inputs.length) return false;
        const fixed = elemId === 'blend-mask-rotation-deg' ? String(Number(value).toFixed(1)) : String(Number(value).toFixed(4));
        inputs.forEach((input) => {
          if (input.value !== fixed) {
            input.value = fixed;
            input.dispatchEvent(new Event('input', { bubbles: true }));
            input.dispatchEvent(new Event('change', { bubbles: true }));
          }
        });
        return true;
      }

      function readOneSlider(elemId, fallback) {
        if (!parentDoc) return fallback;
        const host = parentDoc.getElementById(elemId);
        if (!host) return fallback;
        const input = host.querySelector('input');
        if (!input) return fallback;
        const val = Number(input.value);
        return Number.isFinite(val) ? val : fallback;
      }

      function pushStateToParent() {
        if (window.parent) {
          window.parent.camoPlacementState = { ...state };
        }
        syncOneSlider('blend-mask-scale', state.scale);
        syncOneSlider('blend-mask-offset-x', state.offsetX);
        syncOneSlider('blend-mask-offset-y', state.offsetY);
        syncOneSlider('blend-mask-rotation-deg', state.rotationDeg);
      }

      function pullStateFromParent(force=false) {
        const nextScale = clamp(readOneSlider('blend-mask-scale', state.scale), 0.10, 2.50);
        const nextOffsetX = clamp(readOneSlider('blend-mask-offset-x', state.offsetX), -1.00, 1.00);
        const nextOffsetY = clamp(readOneSlider('blend-mask-offset-y', state.offsetY), -1.00, 1.00);
        const nextRotation = wrapDeg(readOneSlider('blend-mask-rotation-deg', state.rotationDeg));
        const changed = force || !nearlyEqual(nextScale, state.scale) || !nearlyEqual(nextOffsetX, state.offsetX) || !nearlyEqual(nextOffsetY, state.offsetY) || !nearlyEqual(nextRotation, state.rotationDeg, 0.11);
        if (changed) {
          state.scale = nextScale;
          state.offsetX = nextOffsetX;
          state.offsetY = nextOffsetY;
          state.rotationDeg = nextRotation;
          syncBadges();
          draw();
        }
      }

      let mode = null;
      let dragStart = null;
      let rotateStart = null;
      const handleR = 11;
      const rotateHandleGap = 42;

      function syncBadges() {
        document.querySelector('[data-key="scale"]').textContent = state.scale.toFixed(3);
        document.querySelector('[data-key="offsetX"]').textContent = (state.offsetX).toFixed(3);
        document.querySelector('[data-key="offsetY"]').textContent = state.offsetY.toFixed(3);
        document.querySelector('[data-key="rotationDeg"]').textContent = state.rotationDeg.toFixed(1);
      }

      function getGeometry() {
        const cw = canvas.width;
        const ch = canvas.height;
        const dw = cw * state.scale;
        const dh = ch * state.scale;
        const cx = (cw / 2) + state.offsetX * cw;
        const cy = (ch / 2) + state.offsetY * ch;
        const rad = state.rotationDeg * Math.PI / 180.0;
        const cos = Math.cos(rad);
        const sin = Math.sin(rad);

        const localToWorld = (lx, ly) => ({
          x: cx + lx * cos - ly * sin,
          y: cy + lx * sin + ly * cos,
        });
        const worldToLocal = (px, py) => {
          const dx = px - cx;
          const dy = py - cy;
          return {
            x: dx * cos + dy * sin,
            y: -dx * sin + dy * cos,
          };
        };

        const halfW = dw / 2;
        const halfH = dh / 2;
        const corners = [
          localToWorld(-halfW, -halfH),
          localToWorld( halfW, -halfH),
          localToWorld( halfW,  halfH),
          localToWorld(-halfW,  halfH),
        ];
        const resizeHandle = localToWorld(halfW, halfH);
        const rotateHandle = localToWorld(0, -halfH - rotateHandleGap);

        return {
          cw, ch, dw, dh, cx, cy, rad, halfW, halfH,
          corners, resizeHandle, rotateHandle, localToWorld, worldToLocal,
        };
      }

      function drawGuides(geom) {
        ctx.save();
        ctx.strokeStyle = 'rgba(255,255,255,0.15)';
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(geom.cw / 2, 0);
        ctx.lineTo(geom.cw / 2, geom.ch);
        ctx.moveTo(0, geom.ch / 2);
        ctx.lineTo(geom.cw, geom.ch / 2);
        ctx.stroke();

        ctx.setLineDash([8, 8]);
        ctx.strokeStyle = 'rgba(255,255,255,0.70)';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(geom.corners[0].x, geom.corners[0].y);
        for (let i = 1; i < geom.corners.length; i++) ctx.lineTo(geom.corners[i].x, geom.corners[i].y);
        ctx.closePath();
        ctx.stroke();
        ctx.setLineDash([]);

        const topMid = geom.localToWorld(0, -geom.halfH);
        ctx.beginPath();
        ctx.moveTo(topMid.x, topMid.y);
        ctx.lineTo(geom.rotateHandle.x, geom.rotateHandle.y);
        ctx.strokeStyle = 'rgba(251,191,36,0.9)';
        ctx.lineWidth = 2;
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#60a5fa';
        ctx.arc(geom.resizeHandle.x, geom.resizeHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();

        ctx.beginPath();
        ctx.fillStyle = '#fbbf24';
        ctx.arc(geom.rotateHandle.x, geom.rotateHandle.y, handleR, 0, Math.PI * 2);
        ctx.fill();
        ctx.lineWidth = 2;
        ctx.strokeStyle = 'rgba(255,255,255,0.95)';
        ctx.stroke();
        ctx.restore();
      }

      function draw() {
        const geom = getGeometry();
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        if (bg.complete) ctx.drawImage(bg, 0, 0, canvas.width, canvas.height);
        if (overlay.complete) {
          ctx.save();
          ctx.translate(geom.cx, geom.cy);
          ctx.rotate(geom.rad);
          ctx.drawImage(overlay, -geom.dw / 2, -geom.dh / 2, geom.dw, geom.dh);
          ctx.restore();
        }
        drawGuides(geom);
      }

      function localPoint(evt) {
        const rect = canvas.getBoundingClientRect();
        return {
          x: (evt.clientX - rect.left) * (canvas.width / rect.width),
          y: (evt.clientY - rect.top) * (canvas.height / rect.height),
        };
      }

      function startAction(evt) {
        evt.preventDefault();
        const p = localPoint(evt);
        const geom = getGeometry();
        const distResize = Math.hypot(p.x - geom.resizeHandle.x, p.y - geom.resizeHandle.y);
        const distRotate = Math.hypot(p.x - geom.rotateHandle.x, p.y - geom.rotateHandle.y);
        const local = geom.worldToLocal(p.x, p.y);
        const inside = Math.abs(local.x) <= geom.halfW && Math.abs(local.y) <= geom.halfH;

        if (distRotate <= handleR * 1.6) {
          mode = 'rotate';
          rotateStart = {
            baseRotation: state.rotationDeg,
            baseAngle: Math.atan2(p.y - geom.cy, p.x - geom.cx),
          };
          canvas.classList.add('rotating');
        } else if (distResize <= handleR * 1.6) {
          mode = 'resize';
          canvas.classList.add('resizing');
        } else if (inside) {
          mode = 'drag';
          dragStart = {
            startX: p.x,
            startY: p.y,
            baseOffsetX: state.offsetX,
            baseOffsetY: state.offsetY,
          };
          canvas.classList.add('dragging');
        }
      }

      function afterCanvasEdit(message) {
        syncBadges();
        pushStateToParent();
        draw();
        setStatus(message || 'Layout has been synchronized with the left panel.');
      }

      function moveAction(evt) {
        if (!mode) return;
        const p = localPoint(evt);
        const geom = getGeometry();

        if (mode === 'drag') {
          const dx = p.x - dragStart.startX;
          const dy = p.y - dragStart.startY;
          state.offsetX = clamp(dragStart.baseOffsetX + dx / canvas.width, -1.00, 1.00);
          state.offsetY = clamp(dragStart.baseOffsetY + dy / canvas.height, -1.00, 1.00);
        } else if (mode === 'resize') {
          const local = geom.worldToLocal(p.x, p.y);
          const sx = Math.abs(local.x) / (canvas.width / 2);
          const sy = Math.abs(local.y) / (canvas.height / 2);
          state.scale = clamp(Math.max(sx, sy, 0.10), 0.10, 2.50);
        } else if (mode === 'rotate') {
          const angle = Math.atan2(p.y - geom.cy, p.x - geom.cx);
          const delta = wrapRad(angle - rotateStart.baseAngle);
          state.rotationDeg = wrapDeg(rotateStart.baseRotation - delta * 180.0 / Math.PI);
        }

        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }

      function endAction() {
        mode = null;
        dragStart = null;
        rotateStart = null;
        canvas.classList.remove('dragging');
        canvas.classList.remove('resizing');
        canvas.classList.remove('rotating');
      }

      canvas.addEventListener('mousedown', startAction);
      window.addEventListener('mousemove', moveAction);
      window.addEventListener('mouseup', endAction);
      canvas.addEventListener('mouseleave', endAction);
      canvas.addEventListener('wheel', (evt) => {
        evt.preventDefault();
        const factor = evt.deltaY < 0 ? 1.05 : 0.95;
        state.scale = clamp(state.scale * factor, 0.10, 2.50);
        afterCanvasEdit('Synchronizing layout sliders in real time...');
      }, { passive: false });

      document.querySelector('[data-action="center"]').addEventListener('click', () => {
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        afterCanvasEdit('Centered and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="fit"]').addEventListener('click', () => {
        state.scale = 0.70;
        afterCanvasEdit('Set scale to 0.7 and synchronized with the layout sliders.');
      });
      document.querySelector('[data-action="reset"]').addEventListener('click', () => {
        state.scale = 1.0;
        state.offsetX = 0.0;
        state.offsetY = 0.0;
        state.rotationDeg = 0.0;
        afterCanvasEdit('Reset and synchronized with the layout sliders.');
      });

      bg.onload = draw;
      overlay.onload = draw;
      syncBadges();
      pushStateToParent();
      draw();
      setStatus(parentDoc ? 'Canvas is ready and synchronized in real time with the left panel.' : 'Canvas is ready. The parent panel is not accessible, so only internal synchronization is available.');
      setInterval(() => {
        if (!mode) pullStateFromParent(false);
      }, 120);
    })();
  </script>
</body>
</html>
"""

    inner_html = (
        inner_html.replace("__DISPLAY_W__", str(display_w))
        .replace("__CANVAS_W__", str(canvas_w))
        .replace("__CANVAS_H__", str(canvas_h))
        .replace("__BG_URI__", bg_uri)
        .replace("__OVERLAY_URI__", overlay_uri)
        .replace("__INIT_SCALE__", f"{float(init_scale):.6f}")
        .replace("__INIT_OFFSET_X__", f"{float(init_offset_x):.6f}")
        .replace("__INIT_OFFSET_Y__", f"{float(init_offset_y):.6f}")
        .replace("__INIT_ROTATION_DEG__", f"{float(init_rotation_deg):.6f}")
    )
    srcdoc = html_lib.escape(inner_html, quote=True)
    return f"""<div style='width:100%'>
  <iframe id="{iframe_id}" srcdoc="{srcdoc}" style="width:100%; height:{display_h + 150}px; border:0; background:transparent; border-radius:14px;"></iframe>
</div>"""


# ============================================================
# 1) Core config
# ============================================================
CFG: Dict[str, Any] = dict(
    out_dir=DEFAULT_OUT_DIR,
    base_model_id="runwayml/stable-diffusion-v1-5",
    canny_cn_id="lllyasviel/sd-controlnet-canny",
    soft_cn_id="lllyasviel/control_v11p_sd15_softedge",
    hed_annotator_id="lllyasviel/Annotators",
    height=512,
    width=512,
    steps=57,
    batch=1,
    seed=10,
    guidance_scale=2.0,
    guidance_bg=1.2,
    guidance_sub=2.3,
    guess_mode=True,
    canny_low=60,
    canny_high=160,
    canny_blur_ks=5,
    use_broken_edges=True,
    broken_drop_prob=0.30,
    mask_dilate=1,
    mask_close_ks=1,
    mask_close_iter=1,
    mask_blur=1,
    mask_mode="softedge",  # silhouette / outline / composite / softedge
    mask_ring_k=1,
    mask_ring_blur=1,
    mask_gamma=0.1,
    mask_auto_invert=True,
    outer_weight=1.00,
    feature_weight=0.45,
    fill_weight=0.10,
    feature_thresh=32,
    feature_open_ks=3,
    feature_blur=5,
    fill_blur=15,
    use_soft_grad_mask=False,
    soft_grad_dilation_k=3,
    soft_grad_blur_k=3,
    soft_grad_kernel_shape="ellipse",
    soft_grad_iterations=1,
    soft_grad_use_skeleton=False,
    soft_grad_skeleton_stage="post",
    soft_grad_skeleton_thresh="otsu",
    soft_grad_post_skel_dilate_k=0,
    soft_mask_blur=11,
    soft_mask_gamma=0.5,
    soft_mask_invert=False,
    soft_mask_floor=0.05,
    soft_mask_ceiling=0.90,
    debug_mask=False,
    save_mask_images=True,
    control_scales=(0.02, 1.5),
    use_weights="A",
    start_frac_control=0.0,
    end_frac_control=0.84,
    control_gate_kind="cosine",
    blend_profile="decay",
    blend_start_frac=0.05,
    blend_end_frac=0.56,
    blend_alpha_end=0.54,
    auto_attention_compensation=False,
    attention_scale_ref=0.32,
    attention_gamma_ref=0.5,
    attention_ceiling_ref=0.90,
    attention_blur_ref=11,
    blend_mask_scale=1.0,
    blend_mask_offset_x=0.0,
    blend_mask_offset_y=0.0,
    blend_mask_rotation_deg=0.0,
    randomize_seed_each_generate=False,
    bg_image_path=KAGGLE_BG_DIR,
    bg_strength=0.52,
    sub_init_noise=0.0,
    couple_mode="bg_only",
    prompt_bg=(
        ""
    ),
    prompt_subject_template=(
        "a hidden {animal} silhouette seamlessly integrated into the background, subtle facial cues, optical illusion, camouflage"
    ),
    negative=(
        "sticker, pasted object, sharp outline, cartoon, text, watermark, logo, "
        "extra limbs, deformed, low quality, high contrast foreground object"
    ),
    seed_stride=1000,
    log_every=10,
    show_preview=True,
)


def build_cfg(raw_cfg: Dict[str, Any]) -> SimpleNamespace:
    cfg = SimpleNamespace(**raw_cfg)

    if not getattr(cfg, "device", None):
        cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    if not getattr(cfg, "dtype", None):
        cfg.dtype = "float16" if cfg.device.startswith("cuda") and torch.cuda.is_available() else "float32"

    cfg.torch_dtype = torch.float16 if cfg.dtype == "float16" else torch.float32
    ensure_dir(cfg.out_dir)

    odd_fields = [
        "canny_blur_ks",
        "mask_dilate",
        "mask_close_ks",
        "mask_blur",
        "mask_ring_k",
        "mask_ring_blur",
        "feature_open_ks",
        "feature_blur",
        "fill_blur",
        "soft_mask_blur",
        "attention_blur_ref",
        "soft_grad_dilation_k",
        "soft_grad_blur_k",
        "soft_grad_post_skel_dilate_k",
    ]
    for name in odd_fields:
        value = getattr(cfg, name, 0)
        setattr(cfg, name, _odd(value) if value and value > 1 else value)

    cfg.control_scales = tuple(getattr(cfg, "control_scales", (1.0, 1.0)))
    cfg.seed_stride = int(getattr(cfg, "seed_stride", 1000))
    cfg.log_every = int(getattr(cfg, "log_every", 10))
    cfg.show_preview = bool(getattr(cfg, "show_preview", False))
    cfg.debug_mask = bool(getattr(cfg, "debug_mask", False))
    cfg.save_mask_images = bool(getattr(cfg, "save_mask_images", True))
    cfg.use_broken_edges = bool(getattr(cfg, "use_broken_edges", False))
    cfg.broken_drop_prob = float(getattr(cfg, "broken_drop_prob", 0.2))

    cfg.mask_mode = str(getattr(cfg, "mask_mode", "softedge")).lower().strip()
    cfg.mask_gamma = float(getattr(cfg, "mask_gamma", 2.0))
    cfg.mask_auto_invert = bool(getattr(cfg, "mask_auto_invert", True))

    cfg.feature_weight = float(getattr(cfg, "feature_weight", 0.45))
    cfg.fill_weight = float(getattr(cfg, "fill_weight", 0.10))
    cfg.outer_weight = float(getattr(cfg, "outer_weight", 1.00))
    cfg.feature_thresh = int(getattr(cfg, "feature_thresh", 32))
    cfg.feature_open_ks = int(getattr(cfg, "feature_open_ks", 3))
    cfg.feature_blur = int(getattr(cfg, "feature_blur", 5))
    cfg.fill_blur = int(getattr(cfg, "fill_blur", 15))

    cfg.soft_mask_blur = int(getattr(cfg, "soft_mask_blur", 5))
    cfg.soft_mask_gamma = float(getattr(cfg, "soft_mask_gamma", 1.4))
    cfg.soft_mask_invert = bool(getattr(cfg, "soft_mask_invert", False))
    cfg.soft_mask_floor = float(getattr(cfg, "soft_mask_floor", 0.0))
    cfg.soft_mask_ceiling = float(getattr(cfg, "soft_mask_ceiling", 1.0))

    cfg.auto_attention_compensation = bool(getattr(cfg, "auto_attention_compensation", False))
    cfg.attention_scale_ref = float(getattr(cfg, "attention_scale_ref", 0.32))
    cfg.attention_gamma_ref = float(getattr(cfg, "attention_gamma_ref", cfg.soft_mask_gamma))
    cfg.attention_ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", cfg.soft_mask_ceiling))
    cfg.attention_blur_ref = int(getattr(cfg, "attention_blur_ref", cfg.soft_mask_blur))

    cfg.use_soft_grad_mask = bool(getattr(cfg, "use_soft_grad_mask", True))
    cfg.soft_grad_dilation_k = int(getattr(cfg, "soft_grad_dilation_k", 5))
    cfg.soft_grad_blur_k = int(getattr(cfg, "soft_grad_blur_k", 5))
    cfg.soft_grad_kernel_shape = str(getattr(cfg, "soft_grad_kernel_shape", "ellipse")).lower().strip()
    cfg.soft_grad_iterations = int(getattr(cfg, "soft_grad_iterations", 1))
    cfg.soft_grad_use_skeleton = bool(getattr(cfg, "soft_grad_use_skeleton", False))
    cfg.soft_grad_skeleton_stage = str(getattr(cfg, "soft_grad_skeleton_stage", "post")).lower().strip()
    cfg.soft_grad_skeleton_thresh = getattr(cfg, "soft_grad_skeleton_thresh", "otsu")
    cfg.soft_grad_post_skel_dilate_k = int(getattr(cfg, "soft_grad_post_skel_dilate_k", 0))

    cfg.start_frac_control = float(getattr(cfg, "start_frac_control", 0.0))
    cfg.end_frac_control = float(getattr(cfg, "end_frac_control", 1.0))
    cfg.control_gate_kind = str(getattr(cfg, "control_gate_kind", "cosine")).lower().strip()
    cfg.blend_profile = str(getattr(cfg, "blend_profile", "decay")).lower().strip()
    cfg.blend_start_frac = float(getattr(cfg, "blend_start_frac", 0.05))
    cfg.blend_end_frac = float(getattr(cfg, "blend_end_frac", 0.70))
    cfg.blend_alpha_end = float(getattr(cfg, "blend_alpha_end", 0.45))
    cfg.blend_mask_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
    cfg.blend_mask_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
    cfg.blend_mask_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
    cfg.blend_mask_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
    cfg.randomize_seed_each_generate = bool(getattr(cfg, "randomize_seed_each_generate", False))

    cfg.bg_image_path = getattr(cfg, "bg_image_path", None)
    cfg.bg_strength = float(getattr(cfg, "bg_strength", 0.85))
    cfg.sub_init_noise = float(getattr(cfg, "sub_init_noise", 0.0))
    cfg.couple_mode = str(getattr(cfg, "couple_mode", "none")).lower().strip()
    cfg.use_weights = str(getattr(cfg, "use_weights", "A")).upper().strip()

    cfg.guidance_scale = float(getattr(cfg, "guidance_scale", 1.0))
    cfg.guidance_bg = float(getattr(cfg, "guidance_bg", cfg.guidance_scale))
    cfg.guidance_sub = float(getattr(cfg, "guidance_sub", cfg.guidance_scale))
    cfg.guess_mode = bool(getattr(cfg, "guess_mode", True))
    return cfg


# ============================================================
# 2) JSON UI config compatibility layer
# ============================================================
UI_FIELD_NAMES: List[str] = [
    "path_animal",
    "path_bg",
    "animal_name",
    "prompt_bg",
    "prompt_sub",
    "negative_prompt",
    "steps",
    "seed",
    "randomize_seed_each_generate",
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "couple_mode",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "use_broken_edges",
    "broken_drop_prob",
    "mask_mode",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "mask_gamma",
    "mask_auto_invert",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "use_soft_grad_mask",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_kernel_shape",
    "soft_grad_iterations",
    "soft_grad_use_skeleton",
    "soft_grad_skeleton_stage",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "soft_mask_gamma",
    "soft_mask_invert",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "gate_kind",
    "blend_profile",
    "blend_start",
    "blend_end",
    "alpha_end",
    "auto_attention_compensation",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "attention_blur_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
]


UI_DEFAULTS: Dict[str, Any] = {
    "path_animal": KAGGLE_ANIMALS_DIR,
    "path_bg": KAGGLE_BG_DIR,
    "animal_name": "cat",
    "prompt_bg": CFG["prompt_bg"],
    "prompt_sub": CFG["prompt_subject_template"],
    "negative_prompt": CFG["negative"],
    "steps": 57,
    "seed": 10,
    "randomize_seed_each_generate": False,
    "bg_strength": 0.52,
    "guidance_bg": 1.2,
    "guidance_sub": 2.3,
    "sub_init_noise": 0.0,
    "couple_mode": "bg_only",
    "canny_low": 60,
    "canny_high": 160,
    "canny_blur_ks": 5,
    "use_broken_edges": True,
    "broken_drop_prob": 0.3,
    "mask_mode": "softedge",
    "mask_dilate": 1,
    "mask_close_ks": 1,
    "mask_blur": 1,
    "mask_ring_k": 1,
    "mask_ring_blur": 1,
    "mask_gamma": 0.1,
    "mask_auto_invert": True,
    "outer_weight": 1.0,
    "feature_weight": 0.45,
    "fill_weight": 0.10,
    "feature_thresh": 32,
    "feature_open_ks": 3,
    "feature_blur": 5,
    "fill_blur": 15,
    "use_soft_grad_mask": False,
    "soft_grad_dilation_k": 3,
    "soft_grad_blur_k": 3,
    "soft_grad_kernel_shape": "ellipse",
    "soft_grad_iterations": 1,
    "soft_grad_use_skeleton": False,
    "soft_grad_skeleton_stage": "post",
    "soft_grad_post_skel_dilate_k": 0,
    "soft_mask_blur": 11,
    "soft_mask_gamma": 0.5,
    "soft_mask_invert": False,
    "soft_mask_floor": 0.05,
    "soft_mask_ceiling": 0.90,
    "canny_scale": 0.02,
    "soft_scale": 1.5,
    "start_frac": 0.0,
    "end_frac": 0.84,
    "gate_kind": "cosine",
    "blend_profile": "decay",
    "blend_start": 0.05,
    "blend_end": 0.56,
    "alpha_end": 0.54,
    "auto_attention_compensation": False,
    "attention_scale_ref": 0.32,
    "attention_gamma_ref": 0.5,
    "attention_ceiling_ref": 0.90,
    "attention_blur_ref": 11,
    "blend_mask_scale": 1.0,
    "blend_mask_offset_x": 0.0,
    "blend_mask_offset_y": 0.0,
    "blend_mask_rotation_deg": 0.0,
}


LAYOUT_ALIAS_TO_UI_FIELD = {
    "layout_scale": "blend_mask_scale",
    "layout_offset_x": "blend_mask_offset_x",
    "layout_offset_y": "blend_mask_offset_y",
    "layout_rotation_deg": "blend_mask_rotation_deg",
}

NUMERIC_INT_FIELDS = {
    "steps",
    "seed",
    "canny_low",
    "canny_high",
    "canny_blur_ks",
    "mask_dilate",
    "mask_close_ks",
    "mask_blur",
    "mask_ring_k",
    "mask_ring_blur",
    "feature_thresh",
    "feature_open_ks",
    "feature_blur",
    "fill_blur",
    "soft_grad_dilation_k",
    "soft_grad_blur_k",
    "soft_grad_iterations",
    "soft_grad_post_skel_dilate_k",
    "soft_mask_blur",
    "attention_blur_ref",
}

NUMERIC_FLOAT_FIELDS = {
    "bg_strength",
    "guidance_bg",
    "guidance_sub",
    "sub_init_noise",
    "broken_drop_prob",
    "mask_gamma",
    "outer_weight",
    "feature_weight",
    "fill_weight",
    "soft_mask_gamma",
    "soft_mask_floor",
    "soft_mask_ceiling",
    "canny_scale",
    "soft_scale",
    "start_frac",
    "end_frac",
    "blend_start",
    "blend_end",
    "alpha_end",
    "attention_scale_ref",
    "attention_gamma_ref",
    "attention_ceiling_ref",
    "blend_mask_scale",
    "blend_mask_offset_x",
    "blend_mask_offset_y",
    "blend_mask_rotation_deg",
}

BOOL_FIELDS = {
    "use_broken_edges",
    "mask_auto_invert",
    "use_soft_grad_mask",
    "soft_grad_use_skeleton",
    "soft_mask_invert",
    "auto_attention_compensation",
    "randomize_seed_each_generate",
}


def coerce_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y", "on"}
    return bool(value)


def coerce_ui_value(field: str, value: Any) -> Any:
    if field in BOOL_FIELDS:
        return coerce_bool(value)
    if field in NUMERIC_INT_FIELDS:
        return int(value)
    if field in NUMERIC_FLOAT_FIELDS:
        return float(value)
    return value


def normalize_ui_params(raw_params: Any) -> Dict[str, Any]:
    if isinstance(raw_params, list):
        params = dict(zip(UI_FIELD_NAMES, raw_params))
    elif isinstance(raw_params, dict):
        params = dict(raw_params)
        for alias_name, ui_name in LAYOUT_ALIAS_TO_UI_FIELD.items():
            if alias_name in params and ui_name not in params:
                params[ui_name] = params[alias_name]
    else:
        raise ValueError("Unsupported config format: 'params' must be a list or dict.")

    merged = dict(UI_DEFAULTS)
    merged.update(params)
    return {key: coerce_ui_value(key, merged[key]) for key in UI_FIELD_NAMES}


def load_ui_params_from_json(config_path: str) -> Dict[str, Any]:
    with open(config_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict) and "params" in payload:
        return normalize_ui_params(payload["params"])

    if isinstance(payload, dict):
        return normalize_ui_params(payload)

    raise ValueError("JSON config must be a dict or contain a 'params' field.")


def build_ui_dict_from_args(args: Sequence[Any]) -> Dict[str, Any]:
    if len(args) != len(UI_FIELD_NAMES):
        raise ValueError(f"Expected {len(UI_FIELD_NAMES)} UI values, got {len(args)}")
    raw = dict(zip(UI_FIELD_NAMES, args))
    return {key: coerce_ui_value(key, value) for key, value in raw.items()}


# ============================================================
# 3) Job definition
# ============================================================
@dataclass
class AnimalJob:
    image_path: str
    animals: str
    name: Optional[str] = None
    seed: Optional[int] = None
    subject_template: Optional[str] = None
    prompt_bg: Optional[str] = None
    negative: Optional[str] = None
    bg_image_path: Optional[str] = None
    bg_strength: Optional[float] = None


# ============================================================
# 4) Skeleton + kernel-gradient hint
# ============================================================
def skeletonize_u8(img_u8: np.ndarray, thresh: str | int = "otsu") -> np.ndarray:
    if img_u8.dtype != np.uint8:
        img_u8 = np.clip(img_u8, 0, 255).astype(np.uint8)

    if thresh == "otsu":
        _, bin_u8 = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, bin_u8 = cv2.threshold(img_u8, int(thresh), 255, cv2.THRESH_BINARY)

    mask = bin_u8 > 0

    try:
        from skimage.morphology import skeletonize

        return skeletonize(mask).astype(np.uint8) * 255
    except Exception:
        try:
            return cv2.ximgproc.thinning(mask.astype(np.uint8) * 255).astype(np.uint8)
        except Exception as e:
            raise RuntimeError(
                "Skeletonization failed: scikit-image and/or cv2.ximgproc is missing. "
                "Install: pip install scikit-image or opencv-contrib-python."
            ) from e


def kernel_gradient_hint(
    edges_2d: np.ndarray,
    dilation_k: int = 3,
    blur_k: int = 3,
    kernel_shape: str = "ellipse",
    iterations: int = 1,
    use_skeleton: bool = True,
    skeleton_stage: str = "pre",
    skeleton_thresh: str | int = "otsu",
    post_skel_dilate_k: int = 0,
) -> Image.Image:
    if edges_2d.dtype != np.uint8:
        edges_2d = np.clip(edges_2d, 0, 255).astype(np.uint8)

    x = edges_2d
    if use_skeleton and skeleton_stage.lower() == "pre":
        x = skeletonize_u8(x, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            x = cv2.dilate(x, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad = cv2.morphologyEx(
        x,
        cv2.MORPH_GRADIENT,
        _get_kernel(kernel_shape, dilation_k),
        iterations=iterations,
    )

    if use_skeleton and skeleton_stage.lower() == "post":
        grad = skeletonize_u8(grad, thresh=skeleton_thresh)
        if post_skel_dilate_k > 0:
            grad = cv2.dilate(grad, _get_kernel(kernel_shape, post_skel_dilate_k), iterations=1)

    grad_blur = cv2.GaussianBlur(grad, (_ensure_odd(blur_k, 3), _ensure_odd(blur_k, 3)), 0)
    return Image.fromarray(grad_blur).convert("RGB")


# ============================================================
# 5) Image + edge + mask helpers
# ============================================================
def load_image(path: str, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    resolved = resolve_image_path(path)
    if resolved is None:
        raise FileNotFoundError(f"No valid image file or image folder found: {path}")
    img = Image.open(resolved).convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    return img


def pil_to_cv_bgr(img_pil: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def gray_to_pil(gray: np.ndarray, size: Optional[Tuple[int, int]] = None) -> Image.Image:
    img = Image.fromarray(normalize_0_255(gray))
    if size is not None:
        img = img.resize(size, Image.NEAREST)
    return img


def rgb_uint8_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8))


def canny_clean_norm(img_bgr: np.ndarray, low: int = 80, high: int = 160, blur_ks: int = 3) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if blur_ks and blur_ks > 1:
        gray = cv2.GaussianBlur(gray, (blur_ks, blur_ks), 0)
    edges = cv2.Canny(gray, low, high)
    hint = np.stack([edges, edges, edges], axis=-1)
    return normalize_0_255(hint)


def canny_broken(edges_3ch: np.ndarray, drop_prob: float = 0.5, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    edges = edges_3ch[..., 0] > 0
    keep = rng.random(edges.shape) > drop_prob
    out = (edges & keep).astype(np.uint8) * 255
    return np.stack([out, out, out], axis=-1)


def edges_to_silhouette_mask(
    edges_3ch: np.ndarray,
    dilate: int = 17,
    close_ks: int = 17,
    close_iter: int = 1,
    blur: int = 41,
) -> np.ndarray:
    edges = (edges_3ch[..., 0] > 0).astype(np.uint8) * 255

    if dilate and dilate > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(dilate), _odd(dilate)))
        edges = cv2.dilate(edges, kernel, iterations=1)

    if close_ks and close_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(close_ks), _odd(close_ks)))
        edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=max(1, int(close_iter)))

    inv = cv2.bitwise_not(edges)
    flood = inv.copy()
    h, w = edges.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)

    fg = (flood > 0).astype(np.uint8) * 255
    if blur and blur > 1:
        fg = cv2.GaussianBlur(fg, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(fg).astype(np.float32) / 255.0


def extract_internal_feature_mask(
    edges_3ch: np.ndarray,
    sil_hw: np.ndarray,
    thresh: int = 32,
    open_ks: int = 3,
    blur: int = 5,
) -> np.ndarray:
    edges = edges_3ch[..., 0].astype(np.uint8)
    sil_bin = (sil_hw > 0.35).astype(np.uint8) * 255

    if sil_bin.max() == 0:
        return np.zeros_like(sil_hw, dtype=np.float32)

    feat = np.where(sil_bin > 0, edges, 0).astype(np.uint8)
    inner = sil_bin.copy()

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        inner = cv2.erode(inner, kernel, iterations=2)

    feat = np.where(inner > 0, feat, 0).astype(np.uint8)
    feat = (feat > thresh).astype(np.uint8) * 255

    if open_ks and open_ks > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(open_ks), _odd(open_ks)))
        feat = cv2.morphologyEx(feat, cv2.MORPH_OPEN, kernel, iterations=1)

    if blur and blur > 1:
        feat = cv2.GaussianBlur(feat, (_odd(blur), _odd(blur)), 0)

    return normalize_0_255(feat).astype(np.float32) / 255.0


def build_composite_mask_hw(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    outer_ring_k: int = 11,
    outer_ring_blur: int = 7,
    outer_weight: float = 1.0,
    feature_weight: float = 0.45,
    fill_weight: float = 0.10,
    fill_blur: int = 15,
) -> np.ndarray:
    sil_u8 = gray01_to_u8(sil_hw)

    if outer_ring_k and outer_ring_k > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(outer_ring_k), _odd(outer_ring_k)))
        dil = cv2.dilate(sil_u8, kernel, iterations=1)
        ero = cv2.erode(sil_u8, kernel, iterations=1)
        ring = cv2.subtract(dil, ero)
    else:
        ring = sil_u8.copy()

    if outer_ring_blur and outer_ring_blur > 1:
        ring = cv2.GaussianBlur(ring, (_odd(outer_ring_blur), _odd(outer_ring_blur)), 0)

    ring = normalize_0_255(ring).astype(np.float32) / 255.0

    fill = sil_u8.copy()
    if fill_blur and fill_blur > 1:
        fill = cv2.GaussianBlur(fill, (_odd(fill_blur), _odd(fill_blur)), 0)
    fill = normalize_0_255(fill).astype(np.float32) / 255.0

    feat = np.clip(feature_hw.astype(np.float32), 0.0, 1.0)
    out = outer_weight * ring + feature_weight * feat + fill_weight * fill
    return np.clip(out, 0.0, 1.0)


def softedge_to_mask_hw(
    soft_hint: np.ndarray,
    blur_ks: int = 0,
    gamma: float = 1.0,
    invert: bool = False,
    floor: float = 0.0,
    ceiling: float = 1.0,
) -> np.ndarray:
    gray = (
        cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
        if soft_hint.ndim == 3
        else soft_hint.astype(np.uint8)
    )
    mask = gray.astype(np.float32) / 255.0

    if blur_ks and blur_ks > 1:
        mask = cv2.GaussianBlur(mask, (_odd(blur_ks), _odd(blur_ks)), 0)

    if invert:
        mask = 1.0 - mask

    mask = np.clip(mask, 0.0, 1.0)
    if gamma != 1.0:
        mask = mask ** gamma

    floor = float(np.clip(floor, 0.0, 1.0))
    ceiling = float(np.clip(ceiling, 0.0, 1.0))
    if ceiling > floor:
        mask = np.clip((mask - floor) / (ceiling - floor), 0.0, 1.0)

    return np.clip(mask, 0.0, 1.0).astype(np.float32)


# ============================================================
# 6) Torch helpers
# ============================================================
def to_torch_image_hint(hint_uint8: np.ndarray, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = torch.from_numpy(hint_uint8).to(device=device).float() / 255.0
    return x.permute(2, 0, 1).unsqueeze(0).contiguous().to(dtype=dtype)


def make_latent_mask(mask_hw: np.ndarray, latent_h: int, latent_w: int, device: str, dtype: torch.dtype) -> torch.Tensor:
    mask = torch.from_numpy(mask_hw).float().unsqueeze(0).unsqueeze(0)
    mask = F.interpolate(mask, size=(latent_h, latent_w), mode="bilinear", align_corners=False)
    return mask.to(device=device, dtype=dtype)


def _dilate_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.max_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def _erode_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return -F.max_pool2d(-x, kernel_size=k, stride=1, padding=k // 2)


def _blur_t(x: torch.Tensor, k: int) -> torch.Tensor:
    k = _odd(k)
    return F.avg_pool2d(x, kernel_size=k, stride=1, padding=k // 2)


def save_mask_img(mask_1x1hw: torch.Tensor, out_path: str, width: int, height: int) -> None:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(arr).resize((width, height), Image.NEAREST).save(out_path)


def mask_tensor_to_pil(mask_1x1hw: torch.Tensor, width: int, height: int) -> Image.Image:
    arr = (mask_1x1hw[0, 0].detach().float().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr).resize((width, height), Image.NEAREST)


def has_layout_transform(cfg: SimpleNamespace) -> bool:
    return (
        abs(float(getattr(cfg, "blend_mask_scale", 1.0)) - 1.0) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_x", 0.0))) > 1e-6
        or abs(float(getattr(cfg, "blend_mask_offset_y", 0.0))) > 1e-6
    )


def transform_frame(
    arr: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
    border_value: float | Tuple[float, ...] = 0.0,
) -> np.ndarray:
    src = np.asarray(arr)
    h, w = src.shape[:2]
    scale = max(float(scale), 1e-3)
    # UI/canvas convention: +offset_x => move right, +offset_y => move down.
    # Because cv2.warpAffine samples via an inverse map, the translation terms here
    # need the opposite sign so the rendered output moves in the same direction as
    # the interactive canvas preview.
    tx = -float(offset_x_frac) * float(w)
    ty = -float(offset_y_frac) * float(h)
    theta = math.radians(float(rotation_deg))
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    cx = (float(w) - 1.0) * 0.5
    cy = (float(h) - 1.0) * 0.5

    t_neg = np.array([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [0.0, 0.0, 1.0]], dtype=np.float32)
    s_mat = np.array([[scale, 0.0, 0.0], [0.0, scale, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    r_mat = np.array([[cos_t, -sin_t, 0.0], [sin_t, cos_t, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    t_pos = np.array([[1.0, 0.0, cx + tx], [0.0, 1.0, cy + ty], [0.0, 0.0, 1.0]], dtype=np.float32)

    forward = t_pos @ r_mat @ s_mat @ t_neg
    m = np.linalg.inv(forward)[:2].astype(np.float32)

    out = cv2.warpAffine(
        src,
        m,
        (w, h),
        flags=interp,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )
    return out


def transform_mask_hw(

    mask_hw: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(mask_hw, dtype=np.float32),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0.0,
    )
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def transform_hint_uint8(
    hint: np.ndarray,
    scale: float = 1.0,
    offset_x_frac: float = 0.0,
    offset_y_frac: float = 0.0,
    rotation_deg: float = 0.0,
    interp: int = cv2.INTER_LINEAR,
) -> np.ndarray:
    out = transform_frame(
        np.asarray(hint, dtype=np.uint8),
        scale=scale,
        offset_x_frac=offset_x_frac,
        offset_y_frac=offset_y_frac,
        rotation_deg=rotation_deg,
        interp=interp,
        border_value=0,
    )
    return np.clip(out, 0, 255).astype(np.uint8)


def compute_layout_scale_eff(mask_hw: np.ndarray, threshold: float = 0.35) -> float:
    mask = np.asarray(mask_hw, dtype=np.float32)
    if mask.ndim != 2 or mask.size == 0:
        return 0.0

    total = float(mask.shape[0] * mask.shape[1])
    area = float((mask > float(threshold)).sum())
    if area <= 0.0:
        area = float(np.clip(mask, 0.0, 1.0).sum())
    if total <= 0.0 or area <= 0.0:
        return 0.0
    return float(math.sqrt(area / total))


def resolve_attention_compensation(cfg: SimpleNamespace, sil_hw: np.ndarray) -> Dict[str, Any]:
    scale_eff = compute_layout_scale_eff(sil_hw, threshold=0.35)
    scale_ref = max(float(getattr(cfg, "attention_scale_ref", 0.32)), 1e-4)
    ratio = max(scale_eff / scale_ref, 1e-4) if scale_eff > 0.0 else 1.0

    gamma_ref = float(getattr(cfg, "attention_gamma_ref", getattr(cfg, "soft_mask_gamma", 1.0)))
    ceiling_ref = float(getattr(cfg, "attention_ceiling_ref", getattr(cfg, "soft_mask_ceiling", 1.0)))
    blur_ref = int(getattr(cfg, "attention_blur_ref", getattr(cfg, "soft_mask_blur", 5)))

    effective_gamma = float(getattr(cfg, "soft_mask_gamma", gamma_ref))
    effective_ceiling = float(getattr(cfg, "soft_mask_ceiling", ceiling_ref))
    effective_blur = int(getattr(cfg, "soft_mask_blur", blur_ref))

    enabled = bool(getattr(cfg, "auto_attention_compensation", False))
    applies = enabled and str(getattr(cfg, "mask_mode", "softedge")).lower() == "softedge"

    if applies:
        effective_gamma = float(np.clip(gamma_ref * (ratio ** 0.70), 0.25, 0.95))
        effective_ceiling = float(np.clip(ceiling_ref * (ratio ** 0.25), 0.70, 0.97))
        blur_val = int(round(blur_ref * (ratio ** 0.50)))
        effective_blur = int(np.clip(_ensure_odd(blur_val, minimum=1), 1, 31))

    return {
        "enabled": enabled,
        "applies": applies,
        "scale_eff": float(scale_eff),
        "scale_ref": float(scale_ref),
        "ratio": float(ratio),
        "gamma_ref": float(gamma_ref),
        "ceiling_ref": float(ceiling_ref),
        "blur_ref": int(blur_ref),
        "effective_gamma": float(effective_gamma),
        "effective_ceiling": float(effective_ceiling),
        "effective_blur": int(effective_blur),
    }


def format_attention_status(info: Dict[str, Any], mask_mode: str) -> str:
    base = (
        f"scale_eff={float(info.get('scale_eff', 0.0)):.3f} | "
        f"scale_ref={float(info.get('scale_ref', 0.0)):.3f} | "
        f"ratio={float(info.get('ratio', 1.0)):.3f}"
    )
    if not bool(info.get("enabled", False)):
        return base + " | Auto attention compensation: OFF"
    if not bool(info.get("applies", False)):
        return base + f" | Auto attention compensation: ON, but mask_mode='{mask_mode}' so it is not applied to the soft mask."
    return (
        base
        + " | effective soft mask: "
        + f"gamma={float(info.get('effective_gamma', 0.0)):.3f}, "
        + f"ceiling={float(info.get('effective_ceiling', 0.0)):.3f}, "
        + f"blur={int(info.get('effective_blur', 0))}"
    )


def build_mask_latent(
    sil_hw: np.ndarray,
    feature_hw: np.ndarray,
    latent_h: int,
    latent_w: int,
    cfg: SimpleNamespace,
    job_out: str,
    soft_hint: Optional[np.ndarray] = None,
    soft_for_mask: Optional[np.ndarray] = None,
    soft_mask_hw: Optional[np.ndarray] = None,
) -> torch.Tensor:
    sil_mask = make_latent_mask(sil_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)

    if cfg.mask_auto_invert and float(sil_mask.mean()) > 0.60:
        sil_mask = 1.0 - sil_mask

    if cfg.save_mask_images:
        save_mask_img(sil_mask, os.path.join(job_out, "mask_silhouette.png"), cfg.width, cfg.height)

    mode = str(cfg.mask_mode).lower()

    if mode == "silhouette":
        mask = sil_mask
    elif mode == "outline":
        ring = (_dilate_t(sil_mask, int(cfg.mask_ring_k)) - _erode_t(sil_mask, int(cfg.mask_ring_k))).clamp(0, 1)
        if int(cfg.mask_ring_blur) > 1:
            ring = _blur_t(ring, int(cfg.mask_ring_blur)).clamp(0, 1)
        mask = ring
    elif mode == "composite":
        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )
        mask = make_latent_mask(comp_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    elif mode == "softedge":
        if soft_mask_hw is None:
            src_for_mask = soft_for_mask if soft_for_mask is not None else soft_hint
            if src_for_mask is None:
                raise ValueError("mask_mode='softedge' but soft source is None")
            soft_mask_hw = softedge_to_mask_hw(
                soft_hint=src_for_mask,
                blur_ks=int(cfg.soft_mask_blur),
                gamma=float(cfg.soft_mask_gamma),
                invert=bool(cfg.soft_mask_invert),
                floor=float(cfg.soft_mask_floor),
                ceiling=float(cfg.soft_mask_ceiling),
            )
        mask = make_latent_mask(soft_mask_hw, latent_h, latent_w, cfg.device, cfg.torch_dtype).clamp(0, 1)
    else:
        raise ValueError(f"Unknown mask_mode: {cfg.mask_mode}")

    if mode != "softedge":
        gamma = float(cfg.mask_gamma)
        if gamma != 1.0:
            mask = (mask.clamp(0, 1) ** gamma).clamp(0, 1)

    mask = mask.clamp(0, 1)

    if cfg.save_mask_images:
        save_mask_img(mask, os.path.join(job_out, "mask_latent.png"), cfg.width, cfg.height)
    return mask


def encode_prompt(pipe, prompt: str, negative_prompt: str, batch_size: int, do_cfg: bool, device: str) -> torch.Tensor:
    cond, uncond = pipe.encode_prompt(
        prompt=[prompt] * batch_size,
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=do_cfg,
        negative_prompt=[negative_prompt] * batch_size,
    )
    return torch.cat([uncond, cond], dim=0) if do_cfg else cond


def prepare_noise_latents(
    pipe,
    batch_size: int,
    height: int,
    width: int,
    generator: torch.Generator,
    device: str,
    dtype: torch.dtype,
) -> torch.Tensor:
    latent_h, latent_w = height // 8, width // 8
    shape = (batch_size, pipe.unet.config.in_channels, latent_h, latent_w)
    latents = torch.randn(shape, generator=generator, device=device, dtype=dtype)
    return latents * pipe.scheduler.init_noise_sigma


def pil_to_vae_tensor(img_pil: Image.Image, device: str, dtype: torch.dtype) -> torch.Tensor:
    arr = np.array(img_pil).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = x * 2.0 - 1.0
    return x.to(device=device, dtype=dtype)


@torch.no_grad()
def encode_image_latents(pipe, img_pil: Image.Image, generator: torch.Generator, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = pil_to_vae_tensor(img_pil, device=device, dtype=dtype)
    lat = pipe.vae.encode(x).latent_dist.sample(generator=generator)
    return lat * pipe.vae.config.scaling_factor


@torch.no_grad()
def decode_latents(pipe, latents: torch.Tensor) -> List[Image.Image]:
    latents = (1.0 / pipe.vae.config.scaling_factor) * latents
    img = pipe.vae.decode(latents).sample
    img = (img / 2 + 0.5).clamp(0, 1)
    img = img.cpu().permute(0, 2, 3, 1).numpy()
    imgs = (img * 255).round().astype(np.uint8)
    return [Image.fromarray(im) for im in imgs]


# ============================================================
# 7) Control weights + schedules
# ============================================================
def preset_weights_A(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.array([0, 0, 0, 0, 0.1, 0.1, 0.2, 0.2, 0.3, 0.6, 0.6, 0.6], dtype=np.float32)
    mid = 0.9
    if len(weights) != n_down:
        weights = np.resize(weights, n_down)
    return weights, mid


def preset_weights_B(n_down: int) -> Tuple[np.ndarray, float]:
    weights = np.linspace(0.25, 0.65, n_down).astype(np.float32)
    if n_down >= 12:
        for i in range(4, 9):
            weights[i] = min(0.70, weights[i] + 0.08)
    return weights, 0.55


def control_gate(step_idx: int, num_steps: int, start_frac: float, end_frac: float, kind: str = "cosine") -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    if step_idx < s0 or step_idx >= s1:
        return 0.0
    active = max(1, s1 - s0)
    x = (step_idx - s0) / active
    if kind == "linear":
        return float(1.0 - x)
    return float(math.cos(0.5 * math.pi * x))


def blend_alpha(
    step_idx: int,
    num_steps: int,
    start_frac: float,
    end_frac: float,
    alpha_end: float,
    profile: str = "decay",
) -> float:
    s0 = int(num_steps * start_frac)
    s1 = int(num_steps * end_frac)
    s0 = max(0, min(s0, num_steps - 1))
    s1 = max(0, min(s1, num_steps))

    if profile == "ramp":
        if step_idx <= s0:
            return 0.0
        if step_idx >= s1:
            return float(alpha_end)
        return float(((step_idx - s0) / max(1, s1 - s0)) * alpha_end)

    if step_idx <= s0:
        return 1.0
    if step_idx >= s1:
        return float(alpha_end)
    return float(1.0 - ((step_idx - s0) / max(1, s1 - s0)) * (1.0 - alpha_end))


@torch.no_grad()
def unet_with_optional_control(
    pipe,
    scheduler,
    latents,
    t,
    encoder_hidden_states,
    do_cfg,
    guidance_scale,
    controlnets=None,
    control_images=None,
    control_scales=None,
    layer_weights=None,
    mid_weight=None,
    step_gate=1.0,
    guess_mode=False,
    device="cuda",
    dtype=torch.float16,
) -> torch.Tensor:
    unet = pipe.unet
    latents_in = torch.cat([latents] * 2, dim=0) if do_cfg else latents
    latents_in = scheduler.scale_model_input(latents_in, t)
    down_res, mid_res = None, None

    if controlnets is not None and control_images is not None:
        if control_scales is None:
            control_scales = [1.0] * len(controlnets)

        control_cond_mask = None
        if do_cfg and guess_mode:
            control_cond_mask = torch.zeros((latents_in.shape[0], 1, 1, 1), device=device, dtype=dtype)
            control_cond_mask[latents.shape[0]:] = 1.0

        sum_down, sum_mid = None, None
        for cn, img, scale in zip(controlnets, control_images, control_scales):
            img_in = torch.cat([img] * 2, dim=0) if do_cfg else img
            cn_out = cn(
                latents_in,
                t,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=img_in,
                return_dict=True,
            )

            down = list(cn_out.down_block_res_samples)
            mid = cn_out.mid_block_res_sample

            if layer_weights is not None:
                for i in range(min(len(down), len(layer_weights))):
                    down[i] = down[i] * float(layer_weights[i])

            if mid_weight is not None:
                mid = mid * float(mid_weight)

            if step_gate != 1.0:
                down = [d * float(step_gate) for d in down]
                mid = mid * float(step_gate)

            if control_cond_mask is not None:
                down = [d * control_cond_mask for d in down]
                mid = mid * control_cond_mask

            if scale != 1.0:
                down = [d * float(scale) for d in down]
                mid = mid * float(scale)

            if sum_down is None:
                sum_down, sum_mid = down, mid
            else:
                for i in range(len(sum_down)):
                    sum_down[i] = sum_down[i] + down[i]
                sum_mid = sum_mid + mid

        down_res, mid_res = sum_down, sum_mid

    noise_pred = unet(
        latents_in,
        t,
        encoder_hidden_states=encoder_hidden_states,
        down_block_additional_residuals=down_res,
        mid_block_additional_residual=mid_res,
        return_dict=False,
    )[0]

    if do_cfg:
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + float(guidance_scale) * (noise_cond - noise_uncond)

    return noise_pred


# ============================================================
# 8) Runner
# ============================================================
class CamoRunner:
    def __init__(self, cfg: SimpleNamespace):
        self.cfg = cfg
        self.pipe = None
        self.control_nets = None
        self.hed = None

    def build_pipe(self):
        cfg = self.cfg
        seed_everything(int(cfg.seed))

        cn_canny = ControlNetModel.from_pretrained(cfg.canny_cn_id, torch_dtype=cfg.torch_dtype)
        cn_soft = ControlNetModel.from_pretrained(cfg.soft_cn_id, torch_dtype=cfg.torch_dtype)

        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            cfg.base_model_id,
            controlnet=[cn_canny, cn_soft],
            torch_dtype=cfg.torch_dtype,
            safety_checker=None,
        )
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to(cfg.device)
        pipe.set_progress_bar_config(disable=True)

        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
        pipe.enable_attention_slicing()

        self.pipe = pipe
        self.control_nets = pipe.controlnet.nets if hasattr(pipe.controlnet, "nets") else [pipe.controlnet]
        return self

    def _get_hed(self):
        if self.hed is not None:
            return self.hed
        try:
            from controlnet_aux import HEDdetector

            self.hed = HEDdetector.from_pretrained(self.cfg.hed_annotator_id)
            return self.hed
        except Exception:
            return None

    def preprocess(
        self,
        image_path: str,
        seed: int,
        layout_override: Optional[Tuple[float, float, float]] = None,
    ) -> Dict[str, Any]:
        cfg = self.cfg
        src = load_image(image_path, size=(cfg.width, cfg.height))
        src_cv = pil_to_cv_bgr(src)

        if layout_override is None:
            layout_scale = float(getattr(cfg, "blend_mask_scale", 1.0))
            layout_offset_x = float(getattr(cfg, "blend_mask_offset_x", 0.0))
            layout_offset_y = float(getattr(cfg, "blend_mask_offset_y", 0.0))
            layout_rotation_deg = float(getattr(cfg, "blend_mask_rotation_deg", 0.0))
        else:
            values = [float(v) for v in layout_override]
            if len(values) >= 4:
                layout_scale, layout_offset_x, layout_offset_y, layout_rotation_deg = values[:4]
            else:
                layout_scale, layout_offset_x, layout_offset_y = values[:3]
                layout_rotation_deg = 0.0

        canny_hint = canny_clean_norm(
            src_cv,
            low=cfg.canny_low,
            high=cfg.canny_high,
            blur_ks=cfg.canny_blur_ks,
        )

        soft_hint = None
        hed = self._get_hed()
        if hed is not None:
            try:
                hed_map = hed(src)
                hed_np = normalize_0_255(np.array(hed_map.convert("L")))
                soft_hint = np.stack([hed_np, hed_np, hed_np], axis=-1)
            except Exception:
                pass

        if soft_hint is None:
            soft_hint = canny_hint.copy()

        if layout_scale != 1.0 or layout_offset_x != 0.0 or layout_offset_y != 0.0 or layout_rotation_deg != 0.0:
            canny_hint = transform_hint_uint8(
                canny_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_NEAREST,
            )
            soft_hint = transform_hint_uint8(
                soft_hint,
                scale=layout_scale,
                offset_x_frac=layout_offset_x,
                offset_y_frac=layout_offset_y,
                rotation_deg=layout_rotation_deg,
                interp=cv2.INTER_LINEAR,
            )

        if cfg.use_broken_edges:
            canny_hint = canny_broken(canny_hint, drop_prob=cfg.broken_drop_prob, seed=seed)

        sil_hw = edges_to_silhouette_mask(
            canny_hint,
            dilate=int(cfg.mask_dilate),
            close_ks=int(cfg.mask_close_ks),
            close_iter=int(getattr(cfg, "mask_close_iter", 1)),
            blur=int(cfg.mask_blur),
        )

        feature_hw = extract_internal_feature_mask(
            canny_hint,
            sil_hw,
            thresh=int(cfg.feature_thresh),
            open_ks=int(cfg.feature_open_ks),
            blur=int(cfg.feature_blur),
        )

        comp_hw = build_composite_mask_hw(
            sil_hw=sil_hw,
            feature_hw=feature_hw,
            outer_ring_k=int(cfg.mask_ring_k),
            outer_ring_blur=int(cfg.mask_ring_blur),
            outer_weight=float(cfg.outer_weight),
            feature_weight=float(cfg.feature_weight),
            fill_weight=float(cfg.fill_weight),
            fill_blur=int(cfg.fill_blur),
        )

        soft_for_mask = soft_hint.copy()
        if bool(cfg.use_soft_grad_mask):
            soft_gray = cv2.cvtColor(soft_hint.astype(np.uint8), cv2.COLOR_RGB2GRAY)
            soft_grad_pil = kernel_gradient_hint(
                edges_2d=soft_gray,
                dilation_k=int(cfg.soft_grad_dilation_k),
                blur_k=int(cfg.soft_grad_blur_k),
                kernel_shape=str(cfg.soft_grad_kernel_shape),
                iterations=int(cfg.soft_grad_iterations),
                use_skeleton=bool(cfg.soft_grad_use_skeleton),
                skeleton_stage=str(cfg.soft_grad_skeleton_stage),
                skeleton_thresh=cfg.soft_grad_skeleton_thresh,
                post_skel_dilate_k=int(cfg.soft_grad_post_skel_dilate_k),
            )
            soft_for_mask = np.array(soft_grad_pil.convert("RGB"))

        attention_info = resolve_attention_compensation(cfg, sil_hw)
        soft_mask_hw = softedge_to_mask_hw(
            soft_hint=soft_for_mask,
            blur_ks=int(attention_info["effective_blur"]),
            gamma=float(attention_info["effective_gamma"]),
            invert=bool(cfg.soft_mask_invert),
            floor=float(cfg.soft_mask_floor),
            ceiling=float(attention_info["effective_ceiling"]),
        )

        return {
            "src": src,
            "canny_hint": canny_hint,
            "soft_hint": soft_hint,
            "sil_hw": sil_hw,
            "feature_hw": feature_hw,
            "comp_hw": comp_hw,
            "soft_for_mask": soft_for_mask,
            "soft_mask_hw": soft_mask_hw,
            "attention_info": attention_info,
        }

    def _job_out_dir(self, job: AnimalJob) -> str:
        base = os.path.splitext(os.path.basename(job.image_path))[0]
        job_name = job.name or f"{slugify(job.animals)}_{slugify(base)}"
        out_dir = os.path.join(self.cfg.out_dir, job_name)
        ensure_dir(out_dir)
        return out_dir

    def _save_run_config(self, job_out: str, job: AnimalJob, extra: dict) -> None:
        cfg_dict = {k: v for k, v in self.cfg.__dict__.items() if k != "torch_dtype"}
        cfg_dict["job"] = {
            "animals": job.animals,
            "image_path": job.image_path,
            "name": job.name,
            "seed": job.seed,
            **extra,
        }
        save_json(os.path.join(job_out, "run_config.json"), cfg_dict)

    def _probe_weights(self, lat_sub, t0, emb_sub, img_canny, do_cfg) -> Tuple[np.ndarray, float]:
        lat_in = torch.cat([lat_sub] * 2, 0) if do_cfg else lat_sub
        img_in = torch.cat([img_canny] * 2, 0) if do_cfg else img_canny
        probe = self.control_nets[0](
            self.pipe.scheduler.scale_model_input(lat_in, t0),
            t0,
            encoder_hidden_states=emb_sub,
            controlnet_cond=img_in,
            return_dict=True,
        )
        n_down = len(probe.down_block_res_samples)
        return preset_weights_B(n_down) if self.cfg.use_weights == "B" else preset_weights_A(n_down)

    def _init_latents_and_timesteps(self, job_out: str, job: AnimalJob, generator: torch.Generator):
        cfg, pipe = self.cfg, self.pipe
        pipe.scheduler.set_timesteps(int(cfg.steps), device=cfg.device)
        timesteps = pipe.scheduler.timesteps

        scheduler_bg = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_sub = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        scheduler_bg.set_timesteps(int(cfg.steps), device=cfg.device)
        scheduler_sub.set_timesteps(int(cfg.steps), device=cfg.device)

        bg_path = job.bg_image_path if job.bg_image_path is not None else cfg.bg_image_path
        bg_strength = float(job.bg_strength) if job.bg_strength is not None else float(cfg.bg_strength)

        if bg_path:
            bg_img = load_image(bg_path, size=(cfg.width, cfg.height))
            bg_img.save(os.path.join(job_out, "bg_input.png"))

            lat_img = encode_image_latents(pipe, bg_img, generator, cfg.device, cfg.torch_dtype)
            lat_img = lat_img.repeat(int(cfg.batch), 1, 1, 1)

            num_steps = int(cfg.steps)
            init_timestep = max(1, min(num_steps, int(num_steps * bg_strength)))
            t_start = min(max(num_steps - init_timestep, 0), num_steps - 1)
            t_start_1 = timesteps[t_start : t_start + 1].to(cfg.device).repeat(int(cfg.batch))

            noise = torch.randn(lat_img.shape, generator=generator, device=lat_img.device, dtype=lat_img.dtype)
            lat_bg = pipe.scheduler.add_noise(lat_img, noise, t_start_1)
            lat_sub = lat_bg.clone()

            if float(cfg.sub_init_noise) > 0:
                sub_noise = torch.randn(lat_sub.shape, generator=generator, device=lat_sub.device, dtype=lat_sub.dtype)
                lat_sub = lat_sub + float(cfg.sub_init_noise) * sub_noise

            return lat_bg, lat_sub, timesteps[t_start:], scheduler_bg, scheduler_sub

        lat_bg = prepare_noise_latents(
            pipe,
            int(cfg.batch),
            int(cfg.height),
            int(cfg.width),
            generator,
            cfg.device,
            cfg.torch_dtype,
        )
        return lat_bg, lat_bg.clone(), timesteps, scheduler_bg, scheduler_sub

    def _apply_coupling(self, lat_out, lat_bg_next, lat_sub_next):
        mode = str(self.cfg.couple_mode).lower()
        if mode == "both":
            return lat_out, lat_out
        if mode == "bg_only":
            return lat_out, lat_sub_next
        return lat_bg_next, lat_sub_next

    @torch.no_grad()
    def run_one(self, job: AnimalJob, log_every: Optional[int] = None, show_preview: Optional[bool] = None):
        cfg, pipe = self.cfg, self.pipe
        job_out = self._job_out_dir(job)
        seed = int(job.seed) if job.seed is not None else int(cfg.seed)
        seed_everything(seed)

        gs_bg = float(getattr(cfg, "guidance_bg", getattr(cfg, "guidance_scale", 1.0)))
        gs_sub = float(getattr(cfg, "guidance_sub", getattr(cfg, "guidance_scale", 1.0)))
        do_cfg_bg, do_cfg_sub = gs_bg > 1.0, gs_sub > 1.0
        log_every = int(cfg.log_every if log_every is None else log_every)

        prep = self.preprocess(job.image_path, seed=seed)
        img_canny = to_torch_image_hint(prep["canny_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)
        img_soft = to_torch_image_hint(prep["soft_hint"], cfg.device, cfg.torch_dtype).repeat(int(cfg.batch), 1, 1, 1)

        prompt_bg = job.prompt_bg or cfg.prompt_bg
        prompt_sub = format_prompt(job.subject_template or cfg.prompt_subject_template, job.animals)
        negative = job.negative or cfg.negative

        emb_bg = encode_prompt(pipe, prompt_bg, negative, int(cfg.batch), do_cfg_bg, cfg.device)
        emb_sub = encode_prompt(pipe, prompt_sub, negative, int(cfg.batch), do_cfg_sub, cfg.device)

        generator = torch.Generator(device=cfg.device).manual_seed(seed)
        lat_bg, lat_sub, timesteps_run, scheduler_bg, scheduler_sub = self._init_latents_and_timesteps(job_out, job, generator)

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            int(cfg.height) // 8,
            int(cfg.width) // 8,
            cfg,
            job_out=job_out,
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        w_down, w_mid = self._probe_weights(lat_sub, timesteps_run[0], emb_sub, img_canny[:1], do_cfg=do_cfg_sub)

        amp_ctx = (
            torch.autocast(device_type="cuda", dtype=cfg.torch_dtype)
            if cfg.device.startswith("cuda")
            else contextlib.nullcontext()
        )

        lat_out = lat_bg.clone()
        num_run = len(timesteps_run)

        with amp_ctx:
            for i, t in enumerate(timesteps_run):
                gate = control_gate(
                    i,
                    num_run,
                    float(cfg.start_frac_control),
                    float(cfg.end_frac_control),
                    str(cfg.control_gate_kind),
                )

                eps_bg = unet_with_optional_control(
                    pipe,
                    scheduler_bg,
                    lat_bg,
                    t,
                    emb_bg,
                    do_cfg_bg,
                    gs_bg,
                    None,
                    None,
                    device=cfg.device,
                    dtype=cfg.torch_dtype,
                )
                lat_bg_next = scheduler_bg.step(eps_bg, t, lat_bg).prev_sample

                eps_sub = unet_with_optional_control(
                    pipe,
                    scheduler_sub,
                    lat_sub,
                    t,
                    emb_sub,
                    do_cfg_sub,
                    gs_sub,
                    self.control_nets,
                    [img_canny, img_soft],
                    list(cfg.control_scales),
                    w_down,
                    w_mid,
                    gate,
                    bool(cfg.guess_mode),
                    cfg.device,
                    cfg.torch_dtype,
                )
                lat_sub_next = scheduler_sub.step(eps_sub, t, lat_sub).prev_sample

                alpha = blend_alpha(
                    i,
                    num_run,
                    float(cfg.blend_start_frac),
                    float(cfg.blend_end_frac),
                    float(cfg.blend_alpha_end),
                    str(cfg.blend_profile),
                )

                lat_out = lat_bg_next * (1.0 - alpha * mask) + lat_sub_next * (alpha * mask)
                lat_bg, lat_sub = self._apply_coupling(lat_out, lat_bg_next, lat_sub_next)

                if log_every and (i + 1) % log_every == 0:
                    print(f"[{job.animals}] step {i + 1:>3}/{num_run} | gate={gate:.3f} | alpha={alpha:.3f}")

        imgs = decode_latents(pipe, lat_out)
        paths = []
        for k, img in enumerate(imgs):
            out_path = os.path.join(job_out, f"out_{slugify(job.animals)}_{k:02d}_seed{seed}.png")
            img.save(out_path)
            paths.append(out_path)

        self._save_run_config(
            job_out,
            job,
            extra={
                "prompt_bg": prompt_bg,
                "prompt_sub": prompt_sub,
                "bg_strength": job.bg_strength or cfg.bg_strength,
            },
        )
        return imgs, paths


# ============================================================
# 9) Scoring / UI integration helpers
# ============================================================
def hidden_score(out_img_pil: Image.Image, outline_like_hw: np.ndarray) -> float:
    img = np.array(out_img_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160).astype(np.float32) / 255.0
    ring = (outline_like_hw > 0.25).astype(np.uint8)
    in_mean = float(edges[ring > 0].mean()) if (ring > 0).any() else 0.0
    out_mean = float(edges[ring == 0].mean())
    return float(-abs((in_mean - out_mean) - 0.01) - 0.5 * in_mean)


# ------------------------------------------------------------
# Demo metrics / quantitative proxy calculations
# ------------------------------------------------------------
def _to_gray01_from_pil(img_pil: Image.Image, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    img = img_pil.convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    arr = np.array(img).astype(np.uint8)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    return np.clip(gray, 0.0, 1.0)


def _to_gray01_from_any(x: Any, size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    if isinstance(x, Image.Image):
        return _to_gray01_from_pil(x, size=size)
    arr = np.asarray(x)
    if arr.ndim == 3:
        arr = cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    arr = arr.astype(np.float32)
    if arr.max() > 1.5:
        arr = arr / 255.0
    if size is not None:
        arr = cv2.resize(arr, size, interpolation=cv2.INTER_LINEAR)
    return np.clip(arr, 0.0, 1.0)


def _global_ssim(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    mux = float(x.mean())
    muy = float(y.mean())
    varx = float(((x - mux) ** 2).mean())
    vary = float(((y - muy) ** 2).mean())
    cov = float(((x - mux) * (y - muy)).mean())

    denom = (mux ** 2 + muy ** 2 + c1) * (varx + vary + c2)
    if abs(denom) < 1e-12:
        return 0.0
    return float(((2 * mux * muy + c1) * (2 * cov + c2)) / denom)


def compute_ssim_bg(out_img: Image.Image, bg_img: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_gray = _to_gray01_from_pil(out_img, size=size)
    bg_gray = _to_gray01_from_pil(bg_img, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)
    bg_region = 1.0 - np.clip(mask, 0.0, 1.0)

    x = out_gray * bg_region
    y = bg_gray * bg_region

    try:
        from skimage.metrics import structural_similarity as ssim
        return float(ssim(x, y, data_range=1.0))
    except Exception:
        return _global_ssim(x, y)


def compute_edge_similarity(out_img: Image.Image, subject_edge_hint: Image.Image, mask_img: Image.Image) -> float:
    size = out_img.size
    out_rgb = np.array(out_img.convert("RGB").resize(size, Image.LANCZOS)).astype(np.uint8)
    out_gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY)
    e_out = cv2.Canny(out_gray, 80, 160).astype(np.float32) / 255.0

    e_sub = _to_gray01_from_pil(subject_edge_hint, size=size)
    mask = _to_gray01_from_pil(mask_img, size=size)

    a = (e_sub * mask).reshape(-1).astype(np.float32)
    b = (e_out * mask).reshape(-1).astype(np.float32)

    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom < 1e-8:
        return 0.0
    return float(np.dot(a, b) / denom)


def _image_edge_density(img_pil: Image.Image, mask_img: Optional[Image.Image] = None) -> float:
    gray = _to_gray01_from_pil(img_pil)
    edges = cv2.Canny((gray * 255).astype(np.uint8), 80, 160).astype(np.float32) / 255.0
    if mask_img is None:
        return float(edges.mean())
    mask = _to_gray01_from_pil(mask_img, size=img_pil.size)
    active = mask > 0.25
    if not active.any():
        return 0.0
    return float(edges[active].mean())


def compute_demo_metrics(
    runner: CamoRunner,
    job: AnimalJob,
    out_img: Image.Image,
    previews: Dict[str, Any],
    hidden_score_value: float,
) -> Dict[str, Any]:
    cfg = runner.cfg
    bg_img = previews["preview_bg_original"]
    canny_hint_img = previews["preview_canny"]
    mask_img = previews["preview_mask"]

    mask_np = _to_gray01_from_pil(mask_img)
    active = mask_np > 0.25

    metrics = {
        "seed": int(job.seed),
        "subject_name": str(job.animals),
        "subject_path": str(job.image_path),
        "background_path": str(job.bg_image_path or cfg.bg_image_path),
        "hidden_score": float(hidden_score_value),
        "SSIM_bg": compute_ssim_bg(out_img, bg_img, mask_img),
        "S_edge": compute_edge_similarity(out_img, canny_hint_img, mask_img),
        "mask_mean": float(mask_np.mean()),
        "mask_max": float(mask_np.max()),
        "mask_active_ratio": float(active.mean()),
        "output_edge_density_global": _image_edge_density(out_img),
        "output_edge_density_in_mask": _image_edge_density(out_img, mask_img),
        "layout_effective_scale": float(previews["attention_info"].get("scale_eff", 0.0)),
        "layout_scale": float(cfg.blend_mask_scale),
        "layout_offset_x": float(cfg.blend_mask_offset_x),
        "layout_offset_y": float(cfg.blend_mask_offset_y),
        "layout_rotation_deg": float(cfg.blend_mask_rotation_deg),
        "steps": int(cfg.steps),
        "bg_strength": float(cfg.bg_strength),
        "guidance_bg": float(cfg.guidance_bg),
        "guidance_sub": float(cfg.guidance_sub),
        "canny_scale": float(cfg.control_scales[0]),
        "softedge_scale": float(cfg.control_scales[1]),
        "blend_profile": str(cfg.blend_profile),
        "blend_start_frac": float(cfg.blend_start_frac),
        "blend_end_frac": float(cfg.blend_end_frac),
        "blend_alpha_end": float(cfg.blend_alpha_end),
        "mask_mode": str(cfg.mask_mode),
        "soft_mask_blur": int(cfg.soft_mask_blur),
        "soft_mask_gamma": float(cfg.soft_mask_gamma),
        "soft_mask_floor": float(cfg.soft_mask_floor),
        "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
    }
    return metrics


def print_demo_metrics(metrics: Dict[str, Any]) -> None:
    print("\n" + "=" * 72)
    print("Calculated Demo Metrics")
    print("=" * 72)
    ordered_keys = [
        "seed",
        "subject_name",
        "SSIM_bg",
        "S_edge",
        "hidden_score",
        "mask_mean",
        "mask_max",
        "mask_active_ratio",
        "output_edge_density_global",
        "output_edge_density_in_mask",
        "layout_effective_scale",
        "steps",
        "bg_strength",
        "guidance_bg",
        "guidance_sub",
        "canny_scale",
        "softedge_scale",
        "blend_profile",
        "blend_start_frac",
        "blend_end_frac",
        "blend_alpha_end",
    ]
    for key in ordered_keys:
        value = metrics.get(key)
        if isinstance(value, float):
            print(f"{key:>28}: {value:.6f}")
        else:
            print(f"{key:>28}: {value}")
    print("=" * 72 + "\n")


def build_demo_metrics_html(metrics: Dict[str, Any]) -> str:
    core_rows = [
        ("SSIM_bg", metrics.get("SSIM_bg")),
        ("S_edge", metrics.get("S_edge")),
        ("Hidden Score", metrics.get("hidden_score")),
        ("Mask Mean", metrics.get("mask_mean")),
        ("Mask Active Ratio", metrics.get("mask_active_ratio")),
        ("Output Edge Density", metrics.get("output_edge_density_global")),
        ("Output Edge Density in Mask", metrics.get("output_edge_density_in_mask")),
        ("Layout Effective Scale", metrics.get("layout_effective_scale")),
    ]

    param_rows = [
        ("Seed", metrics.get("seed")),
        ("Steps", metrics.get("steps")),
        ("BG Strength", metrics.get("bg_strength")),
        ("Guidance BG", metrics.get("guidance_bg")),
        ("Guidance Subject", metrics.get("guidance_sub")),
        ("Canny Scale", metrics.get("canny_scale")),
        ("SoftEdge Scale", metrics.get("softedge_scale")),
        ("Blend Profile", metrics.get("blend_profile")),
        ("Blend Start", metrics.get("blend_start_frac")),
        ("Blend End", metrics.get("blend_end_frac")),
        ("Alpha End", metrics.get("blend_alpha_end")),
    ]

    def fmt(v: Any) -> str:
        if isinstance(v, float):
            return f"{v:.6f}"
        return str(v)

    def rows_html(rows: List[Tuple[str, Any]]) -> str:
        return "".join(
            f"<tr><td style='padding:5px 10px; color:#cbd5e1;'>{html_lib.escape(str(k))}</td>"
            f"<td style='padding:5px 10px; color:#f8fafc; font-weight:600;'>{html_lib.escape(fmt(v))}</td></tr>"
            for k, v in rows
        )

    return f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">Calculated Output Metrics</div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:14px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Proxy Metrics</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(core_rows)}</table>
            </div>
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Main Parameters</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">{rows_html(param_rows)}</table>
            </div>
        </div>
    </div>
    """


def apply_ui_params_to_runner(runner: CamoRunner, ui: Dict[str, Any]) -> None:
    cfg = runner.cfg
    cfg.steps = int(ui["steps"])
    cfg.bg_strength = float(ui["bg_strength"])
    cfg.guidance_bg = float(ui["guidance_bg"])
    cfg.guidance_sub = float(ui["guidance_sub"])
    cfg.sub_init_noise = float(ui["sub_init_noise"])
    cfg.couple_mode = str(ui["couple_mode"])

    cfg.canny_low = int(ui["canny_low"])
    cfg.canny_high = int(ui["canny_high"])
    cfg.canny_blur_ks = int(ui["canny_blur_ks"])
    cfg.use_broken_edges = bool(ui["use_broken_edges"])
    cfg.broken_drop_prob = float(ui["broken_drop_prob"])

    cfg.mask_mode = str(ui["mask_mode"])
    cfg.mask_dilate = int(ui["mask_dilate"])
    cfg.mask_close_ks = int(ui["mask_close_ks"])
    cfg.mask_blur = int(ui["mask_blur"])
    cfg.mask_ring_k = int(ui["mask_ring_k"])
    cfg.mask_ring_blur = int(ui["mask_ring_blur"])
    cfg.mask_gamma = float(ui["mask_gamma"])
    cfg.mask_auto_invert = bool(ui["mask_auto_invert"])

    cfg.outer_weight = float(ui["outer_weight"])
    cfg.feature_weight = float(ui["feature_weight"])
    cfg.fill_weight = float(ui["fill_weight"])
    cfg.feature_thresh = int(ui["feature_thresh"])
    cfg.feature_open_ks = int(ui["feature_open_ks"])
    cfg.feature_blur = int(ui["feature_blur"])
    cfg.fill_blur = int(ui["fill_blur"])

    cfg.use_soft_grad_mask = bool(ui["use_soft_grad_mask"])
    cfg.soft_grad_dilation_k = int(ui["soft_grad_dilation_k"])
    cfg.soft_grad_blur_k = int(ui["soft_grad_blur_k"])
    cfg.soft_grad_kernel_shape = str(ui["soft_grad_kernel_shape"])
    cfg.soft_grad_iterations = int(ui["soft_grad_iterations"])
    cfg.soft_grad_use_skeleton = bool(ui["soft_grad_use_skeleton"])
    cfg.soft_grad_skeleton_stage = str(ui["soft_grad_skeleton_stage"])
    cfg.soft_grad_post_skel_dilate_k = int(ui["soft_grad_post_skel_dilate_k"])

    cfg.soft_mask_blur = int(ui["soft_mask_blur"])
    cfg.soft_mask_gamma = float(ui["soft_mask_gamma"])
    cfg.soft_mask_invert = bool(ui["soft_mask_invert"])
    cfg.soft_mask_floor = float(ui["soft_mask_floor"])
    cfg.soft_mask_ceiling = float(ui["soft_mask_ceiling"])

    cfg.control_scales = (float(ui["canny_scale"]), float(ui["soft_scale"]))
    cfg.start_frac_control = float(ui["start_frac"])
    cfg.end_frac_control = float(ui["end_frac"])
    cfg.control_gate_kind = str(ui["gate_kind"])

    cfg.blend_profile = str(ui["blend_profile"])
    cfg.blend_start_frac = float(ui["blend_start"])
    cfg.blend_end_frac = float(ui["blend_end"])
    cfg.blend_alpha_end = float(ui["alpha_end"])
    cfg.auto_attention_compensation = bool(ui["auto_attention_compensation"])
    cfg.attention_scale_ref = float(ui["attention_scale_ref"])
    cfg.attention_gamma_ref = float(ui["attention_gamma_ref"])
    cfg.attention_ceiling_ref = float(ui["attention_ceiling_ref"])
    cfg.attention_blur_ref = int(ui["attention_blur_ref"])
    cfg.blend_mask_scale = float(ui["blend_mask_scale"])
    cfg.blend_mask_offset_x = float(ui["blend_mask_offset_x"])
    cfg.blend_mask_offset_y = float(ui["blend_mask_offset_y"])
    cfg.blend_mask_rotation_deg = float(ui["blend_mask_rotation_deg"])
    cfg.randomize_seed_each_generate = bool(ui["randomize_seed_each_generate"])


def build_job_from_ui(ui: Dict[str, Any]) -> AnimalJob:
    animal_path_raw = str(ui["path_animal"]).strip()
    bg_path_raw = str(ui["path_bg"]).strip()

    resolved_animal = resolve_image_path(animal_path_raw, fallback_dir=KAGGLE_ANIMALS_DIR)
    if resolved_animal is None:
        raise gr.Error(f"No valid subject image found from: {animal_path_raw}")

    resolved_bg = resolve_image_path(bg_path_raw, fallback_dir=KAGGLE_BG_DIR) if bg_path_raw or KAGGLE_BG_DIR else None

    return AnimalJob(
        image_path=resolved_animal,
        bg_image_path=resolved_bg,
        animals=str(ui["animal_name"]),
        seed=int(ui["seed"]),
        prompt_bg=str(ui["prompt_bg"]),
        subject_template=str(ui["prompt_sub"]),
        negative=str(ui["negative_prompt"]),
        name=f"UI_SoftGradMask_{int(time.time())}",
        bg_strength=float(ui["bg_strength"]),
    )



def stage2_canny_breakdown(canny_hint_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    edges = canny_hint_u8[..., 0].astype(np.uint8)

    contour_agg = edges.copy()

    if int(cfg.mask_dilate) > 1:
        k = _odd(int(cfg.mask_dilate))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.dilate(contour_agg, kernel, iterations=1)

    if int(cfg.mask_close_ks) > 1:
        k = _odd(int(cfg.mask_close_ks))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        contour_agg = cv2.morphologyEx(
            contour_agg,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=max(1, int(getattr(cfg, "mask_close_iter", 1))),
        )

    inv = cv2.bitwise_not(contour_agg)
    flood = inv.copy()
    h, w = contour_agg.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 0)
    region_filled = (flood > 0).astype(np.uint8) * 255

    smoothing = region_filled.copy()
    if int(cfg.mask_blur) > 1:
        k = _odd(int(cfg.mask_blur))
        smoothing = cv2.GaussianBlur(smoothing, (k, k), 0)

    return {
        "canny_edges": edges,
        "contour_aggregation": contour_agg,
        "region_filling": region_filled,
        "canny_smoothing": smoothing,
    }


def stage2_softedge_breakdown(soft_input_u8: np.ndarray, cfg: SimpleNamespace) -> Dict[str, np.ndarray]:
    if soft_input_u8.ndim == 3:
        gray = cv2.cvtColor(soft_input_u8.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = soft_input_u8.astype(np.uint8)

    smoothed = gray.astype(np.float32) / 255.0
    if int(cfg.soft_mask_blur) > 1:
        k = _odd(int(cfg.soft_mask_blur))
        smoothed = cv2.GaussianBlur(smoothed, (k, k), 0)

    remap_input = smoothed.copy()
    if bool(cfg.soft_mask_invert):
        remap_input = 1.0 - remap_input
    remap_input = np.clip(remap_input, 0.0, 1.0)

    gamma_remap = remap_input.copy()
    gamma = float(cfg.soft_mask_gamma)
    if gamma != 1.0:
        gamma_remap = gamma_remap ** gamma

    floor = float(np.clip(cfg.soft_mask_floor, 0.0, 1.0))
    ceiling = float(np.clip(cfg.soft_mask_ceiling, 0.0, 1.0))
    clipped = gamma_remap.copy()
    if ceiling > floor:
        clipped = np.clip((clipped - floor) / (ceiling - floor), 0.0, 1.0)

    final_soft_mask = np.clip(clipped, 0.0, 1.0)

    return {
        "soft_gray": gray01_to_u8(gray.astype(np.float32) / 255.0),
        "soft_smoothing": gray01_to_u8(smoothed),
        "soft_after_gamma_remap": gray01_to_u8(gamma_remap),
        "soft_after_clip": gray01_to_u8(final_soft_mask),
    }


def _stats_from_array(name: str, arr: np.ndarray) -> Dict[str, Any]:
    x = np.asarray(arr).astype(np.float32)
    if x.ndim == 3:
        x = x.mean(axis=2)

    nonzero = float((x > 0).sum())
    total = float(x.size)
    ratio = nonzero / total if total > 0 else 0.0

    return {
        "name": name,
        "shape": tuple(x.shape),
        "min": float(x.min()) if x.size else 0.0,
        "max": float(x.max()) if x.size else 0.0,
        "mean": float(x.mean()) if x.size else 0.0,
        "std": float(x.std()) if x.size else 0.0,
        "nonzero_ratio": float(ratio),
    }


def build_stage2_value_dict(
    runner: CamoRunner,
    job: AnimalJob,
    prep: Dict[str, Any],
    mask: torch.Tensor,
    canny_steps: Dict[str, np.ndarray],
    soft_steps: Dict[str, np.ndarray],
) -> Dict[str, Any]:
    cfg = runner.cfg

    mask_pil = mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height)
    mask_np = np.array(mask_pil.convert("L"))

    info = {
        "paths": {
            "animal_image": job.image_path,
            "background_image": job.bg_image_path or cfg.bg_image_path,
            "out_dir": runner._job_out_dir(job),
        },
        "layout": {
            "blend_mask_scale": float(cfg.blend_mask_scale),
            "blend_mask_offset_x": float(cfg.blend_mask_offset_x),
            "blend_mask_offset_y": float(cfg.blend_mask_offset_y),
            "blend_mask_rotation_deg": float(cfg.blend_mask_rotation_deg),
        },
        "canny_params": {
            "canny_low": int(cfg.canny_low),
            "canny_high": int(cfg.canny_high),
            "canny_blur_ks": int(cfg.canny_blur_ks),
            "use_broken_edges": bool(cfg.use_broken_edges),
            "broken_drop_prob": float(cfg.broken_drop_prob),
        },
        "support_params": {
            "mask_dilate": int(cfg.mask_dilate),
            "mask_close_ks": int(cfg.mask_close_ks),
            "mask_close_iter": int(getattr(cfg, "mask_close_iter", 1)),
            "mask_blur": int(cfg.mask_blur),
            "mask_mode": str(cfg.mask_mode),
        },
        "soft_mask_params": {
            "use_soft_grad_mask": bool(cfg.use_soft_grad_mask),
            "soft_grad_dilation_k": int(cfg.soft_grad_dilation_k),
            "soft_grad_blur_k": int(cfg.soft_grad_blur_k),
            "soft_grad_kernel_shape": str(cfg.soft_grad_kernel_shape),
            "soft_grad_iterations": int(cfg.soft_grad_iterations),
            "soft_grad_use_skeleton": bool(cfg.soft_grad_use_skeleton),
            "soft_grad_skeleton_stage": str(cfg.soft_grad_skeleton_stage),
            "soft_grad_post_skel_dilate_k": int(cfg.soft_grad_post_skel_dilate_k),
            "soft_mask_blur": int(cfg.soft_mask_blur),
            "soft_mask_gamma": float(cfg.soft_mask_gamma),
            "soft_mask_invert": bool(cfg.soft_mask_invert),
            "soft_mask_floor": float(cfg.soft_mask_floor),
            "soft_mask_ceiling": float(cfg.soft_mask_ceiling),
        },
        "attention_info": prep["attention_info"],
        "arrays": {
            "canny_hint": _stats_from_array("canny_hint", prep["canny_hint"]),
            "contour_aggregation": _stats_from_array("contour_aggregation", canny_steps["contour_aggregation"]),
            "region_filling": _stats_from_array("region_filling", canny_steps["region_filling"]),
            "canny_smoothing": _stats_from_array("canny_smoothing", canny_steps["canny_smoothing"]),
            "sil_hw": _stats_from_array("sil_hw", prep["sil_hw"]),
            "feature_hw": _stats_from_array("feature_hw", prep["feature_hw"]),
            "comp_hw": _stats_from_array("comp_hw", prep["comp_hw"]),
            "soft_hint": _stats_from_array("soft_hint", prep["soft_hint"]),
            "soft_for_mask": _stats_from_array("soft_for_mask", prep["soft_for_mask"]),
            "soft_gray": _stats_from_array("soft_gray", soft_steps["soft_gray"]),
            "soft_smoothing": _stats_from_array("soft_smoothing", soft_steps["soft_smoothing"]),
            "soft_after_gamma_remap": _stats_from_array("soft_after_gamma_remap", soft_steps["soft_after_gamma_remap"]),
            "soft_after_clip": _stats_from_array("soft_after_clip", soft_steps["soft_after_clip"]),
            "soft_mask_hw": _stats_from_array("soft_mask_hw", prep["soft_mask_hw"]),
            "latent_mask": _stats_from_array("latent_mask", mask_np),
        },
    }
    return info


def build_stage2_debug_html(stage2_info: Dict[str, Any]) -> str:
    def render_dict(d: Dict[str, Any]) -> str:
        rows = []
        for k, v in d.items():
            if isinstance(v, dict):
                rows.append(
                    f"<tr><td colspan='2' style='padding-top:10px; font-weight:700; color:#93c5fd'>{k}</td></tr>"
                )
                for kk, vv in v.items():
                    rows.append(
                        f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{kk}</td>"
                        f"<td style='padding:4px 10px; color:#f8fafc'>{vv}</td></tr>"
                    )
            else:
                rows.append(
                    f"<tr><td style='padding:4px 10px; color:#cbd5e1'>{k}</td>"
                    f"<td style='padding:4px 10px; color:#f8fafc'>{v}</td></tr>"
                )
        return "".join(rows)

    arrays_html = []
    for name, stats in stage2_info["arrays"].items():
        arrays_html.append(f"""
        <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
            <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">{name}</div>
            <table style="width:100%; font-size:13px; border-collapse:collapse;">
                <tr><td style="padding:3px 8px; color:#cbd5e1;">shape</td><td style="padding:3px 8px; color:#f8fafc;">{stats['shape']}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">min</td><td style="padding:3px 8px; color:#f8fafc;">{stats['min']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">max</td><td style="padding:3px 8px; color:#f8fafc;">{stats['max']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">mean</td><td style="padding:3px 8px; color:#f8fafc;">{stats['mean']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">std</td><td style="padding:3px 8px; color:#f8fafc;">{stats['std']:.4f}</td></tr>
                <tr><td style="padding:3px 8px; color:#cbd5e1;">nonzero_ratio</td><td style="padding:3px 8px; color:#f8fafc;">{stats['nonzero_ratio']:.4f}</td></tr>
            </table>
        </div>
        """)

    html = f"""
    <div style="margin-top:10px; padding:14px; border-radius:14px; background:#0f172a; border:1px solid rgba(255,255,255,.08);">
        <div style="font-size:16px; font-weight:700; color:#f8fafc; margin-bottom:10px;">
            Stage-2 Diagnostic Values
        </div>

        <div style="display:grid; grid-template-columns:1fr 1fr; gap:16px;">
            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Config Summary</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict({
                        "paths": stage2_info["paths"],
                        "layout": stage2_info["layout"],
                        "canny_params": stage2_info["canny_params"],
                        "support_params": stage2_info["support_params"],
                        "soft_mask_params": stage2_info["soft_mask_params"],
                    })}
                </table>
            </div>

            <div style="border:1px solid rgba(255,255,255,.08); border-radius:12px; padding:10px; background:#111827;">
                <div style="font-weight:700; color:#93c5fd; margin-bottom:6px;">Attention / Compensation</div>
                <table style="width:100%; font-size:13px; border-collapse:collapse;">
                    {render_dict(stage2_info["attention_info"])}
                </table>
            </div>
        </div>

        <div style="margin-top:14px; display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px;">
            {''.join(arrays_html)}
        </div>
    </div>
    """
    return html


def make_debug_previews(runner: CamoRunner, job: AnimalJob):
    prep = runner.preprocess(job.image_path, seed=int(job.seed))
    job_out = runner._job_out_dir(job)
    latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8

    mask = build_mask_latent(
        prep["sil_hw"],
        prep["feature_hw"],
        latent_h,
        latent_w,
        runner.cfg,
        job_out=job_out,
        soft_hint=prep["soft_hint"],
        soft_for_mask=prep["soft_for_mask"],
        soft_mask_hw=prep["soft_mask_hw"],
    )

    canny_steps = stage2_canny_breakdown(prep["canny_hint"], runner.cfg)
    soft_steps = stage2_softedge_breakdown(prep["soft_for_mask"], runner.cfg)

    stage2_info = build_stage2_value_dict(
        runner=runner,
        job=job,
        prep=prep,
        mask=mask,
        canny_steps=canny_steps,
        soft_steps=soft_steps,
    )

    score_base = prep["sil_hw"].copy()
    stage2_json = json.dumps(stage2_info, ensure_ascii=False, indent=2)
    stage2_html = build_stage2_debug_html(stage2_info)

    previews = {
        "preview_bg_original": load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height)),
        "preview_input": prep["src"],
        "preview_canny": rgb_uint8_to_pil(prep["canny_hint"]),
        "preview_soft": rgb_uint8_to_pil(prep["soft_hint"]),
        "preview_sil": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_feat": gray_to_pil((prep["feature_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_comp": gray_to_pil((prep["comp_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_for_mask": rgb_uint8_to_pil(prep["soft_for_mask"]),
        "preview_soft_hw": gray_to_pil((prep["soft_mask_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_mask": mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height),
        "preview_contour_agg": gray_to_pil(canny_steps["contour_aggregation"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_region_filling": gray_to_pil(canny_steps["region_filling"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_canny_smoothing": gray_to_pil(canny_steps["canny_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_aux_support": gray_to_pil((prep["sil_hw"] * 255.0).astype(np.uint8), size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_gray": gray_to_pil(soft_steps["soft_gray"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_smoothing": gray_to_pil(soft_steps["soft_smoothing"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_remap": gray_to_pil(soft_steps["soft_after_gamma_remap"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_soft_clip": gray_to_pil(soft_steps["soft_after_clip"], size=(runner.cfg.width, runner.cfg.height)),
        "preview_stage2_report_html": stage2_html,
        "preview_stage2_report_json": stage2_json,
        "score_base": score_base,
        "attention_info": prep["attention_info"],
    }
    return previews


# ============================================================
# 10) Main / Gradio UI
# ============================================================
if __name__ == "__main__":
    cfg = build_cfg(CFG)
    runner = CamoRunner(cfg).build_pipe()


    # ------------------------------------------------------------
    # Upload helpers: single preview + multi-file batch mode
    # ------------------------------------------------------------
    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

    def uploaded_item_to_path(item: Any) -> Optional[str]:
        """Normalize Gradio File values across old/new Gradio versions."""
        if item is None:
            return None

        if isinstance(item, (str, os.PathLike)):
            path = os.fspath(item)
        elif isinstance(item, dict):
            path = item.get("path") or item.get("name")
        else:
            path = getattr(item, "path", None) or getattr(item, "name", None)

        if not path:
            return None
        path = os.path.abspath(os.fspath(path))
        return path if os.path.isfile(path) else None


    def normalize_uploaded_paths(upload_value: Any) -> List[str]:
        """Return a de-duplicated list of valid uploaded image paths."""
        if upload_value is None:
            return []

        raw_items = upload_value if isinstance(upload_value, (list, tuple)) else [upload_value]
        paths: List[str] = []
        seen = set()

        for item in raw_items:
            path = uploaded_item_to_path(item)
            if path is None:
                continue

            # Some temporary Gradio files do not retain the original extension,
            # therefore PIL verification is the final source of truth.
            try:
                with Image.open(path) as probe:
                    probe.verify()
            except Exception:
                continue

            key = os.path.realpath(path)
            if key not in seen:
                seen.add(key)
                paths.append(path)
        return paths


    def save_uploaded_image(upload_value: Any, prefix: str) -> Optional[str]:
        """Save only the first uploaded image for preview/layout functions."""
        paths = normalize_uploaded_paths(upload_value)
        if not paths:
            return None

        upload_dir = os.path.join(runner.cfg.out_dir, "_ui_uploads")
        ensure_dir(upload_dir)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        path = os.path.join(upload_dir, f"{prefix}_{timestamp}.png")
        with Image.open(paths[0]) as img:
            img.convert("RGB").save(path)
        return path


    def apply_uploads_to_ui(
        ui: Dict[str, Any],
        subject_upload: Any = None,
        bg_upload: Any = None,
    ) -> Dict[str, Any]:
        """Use the first uploaded subject/BG for preview and layout editing."""
        ui = dict(ui)

        subject_path = save_uploaded_image(subject_upload, "subject")
        bg_path = save_uploaded_image(bg_upload, "background")

        if subject_path is not None:
            ui["path_animal"] = subject_path
        if bg_path is not None:
            ui["path_bg"] = bg_path
        return ui


    def resolve_batch_image_paths(
        upload_value: Any,
        path_or_dir: Optional[str],
        fallback_dir: Optional[str],
        label: str,
    ) -> List[str]:
        """Uploads have priority; otherwise enumerate every image in the path/folder."""
        uploaded = normalize_uploaded_paths(upload_value)
        if uploaded:
            return uploaded

        candidates = list_image_files(str(path_or_dir or "").strip())
        if not candidates:
            candidates = list_image_files(str(fallback_dir or "").strip())
        if not candidates:
            raise gr.Error(f"No valid {label} images were found in uploads or folder path.")
        return candidates


    def save_pair_artifacts(
        pair_dir: str,
        subject_path: str,
        bg_path: str,
        output_img: Image.Image,
        final_mask: Image.Image,
    ) -> Dict[str, str]:
        """Save the standardized files requested for every subject-background pair."""
        ensure_dir(pair_dir)

        subject_out = os.path.join(pair_dir, "subject_input.png")
        bg_out = os.path.join(pair_dir, "background_input.png")
        mask_out = os.path.join(pair_dir, "final_mask.png")
        result_out = os.path.join(pair_dir, "output.png")

        load_image(subject_path, size=(runner.cfg.width, runner.cfg.height)).save(subject_out)
        load_image(bg_path, size=(runner.cfg.width, runner.cfg.height)).save(bg_out)
        final_mask.convert("L").save(mask_out)
        output_img.convert("RGB").save(result_out)

        return {
            "subject_input": subject_out,
            "background_input": bg_out,
            "final_mask": mask_out,
            "output": result_out,
        }


    def render_layout_editor_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed), layout_override=(1.0, 0.0, 0.0, 0.0))

        latent_h, latent_w = runner.cfg.height // 8, runner.cfg.width // 8
        cfg_preview = SimpleNamespace(**vars(runner.cfg))
        cfg_preview.save_mask_images = False
        cfg_preview.blend_mask_scale = 1.0
        cfg_preview.blend_mask_offset_x = 0.0
        cfg_preview.blend_mask_offset_y = 0.0
        cfg_preview.blend_mask_rotation_deg = 0.0

        mask = build_mask_latent(
            prep["sil_hw"],
            prep["feature_hw"],
            latent_h,
            latent_w,
            cfg_preview,
            job_out=runner._job_out_dir(job),
            soft_hint=prep["soft_hint"],
            soft_for_mask=prep["soft_for_mask"],
            soft_mask_hw=prep["soft_mask_hw"],
        )

        bg_img = load_rgb_preview_image(job.bg_image_path or runner.cfg.bg_image_path, (runner.cfg.width, runner.cfg.height))
        overlay_img = mask_preview_to_rgba(mask_tensor_to_pil(mask, runner.cfg.width, runner.cfg.height))
        return build_interactive_layout_html(
            bg_img=bg_img,
            overlay_img=overlay_img,
            width=runner.cfg.width,
            height=runner.cfg.height,
            init_scale=float(ui["blend_mask_scale"]),
            init_offset_x=float(ui["blend_mask_offset_x"]),
            init_offset_y=float(ui["blend_mask_offset_y"]),
            init_rotation_deg=float(ui["blend_mask_rotation_deg"]),
        )

    def summarize_layout_attention_from_ui(ui: Dict[str, Any]) -> str:
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        return format_attention_status(prep["attention_info"], runner.cfg.mask_mode)

    def refresh_layout_editor(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        html = render_layout_editor_from_ui(ui)
        status = "Layout canvas has been refreshed from the Canny/SoftEdge support. " + summarize_layout_attention_from_ui(ui)
        return html, status

    def capture_current_scale_reference(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)
        prep = runner.preprocess(job.image_path, seed=int(job.seed))
        scale_eff = float(prep["attention_info"]["scale_eff"])
        status = f"Captured the current layout scale as the reference: {scale_eff:.3f}"
        return scale_eff, status

    def safe_initial_layout_html() -> str:
        try:
            return render_layout_editor_from_ui(normalize_ui_params(UI_DEFAULTS))
        except Exception as e:
            return (
                "<div style='padding:12px;border:1px solid rgba(255,255,255,.1);border-radius:12px'>"
                f"Could not create the interactive layout canvas. Please check the image paths and click <b>Refresh Interactive Canvas</b>.<br><small>{e}</small></div>"
            )

    JS_SYNC_CANVAS_TO_ARGS = r"""
    (...args) => {
      const state = window.camoPlacementState;
      const syncHost = (elemId, value) => {
        const host = document.getElementById(elemId);
        if (!host) return;
        host.querySelectorAll('input').forEach((input) => {
          input.value = String(Number(value).toFixed(4));
          input.dispatchEvent(new Event('input', { bubbles: true }));
          input.dispatchEvent(new Event('change', { bubbles: true }));
        });
      };
      if (state) {
        const n = args.length;
        if (n >= 4) {
          args[n - 4] = Number(state.scale);
          args[n - 3] = Number(state.offsetX);
          args[n - 2] = Number(state.offsetY);
          args[n - 1] = Number(state.rotationDeg);
        }
        syncHost('blend-mask-scale', state.scale);
        syncHost('blend-mask-offset-x', state.offsetX);
        syncHost('blend-mask-offset-y', state.offsetY);
        syncHost('blend-mask-rotation-deg', state.rotationDeg);
      }
      return args;
    }
    """

    def generate_preview(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        if bool(ui["randomize_seed_each_generate"]):
            ui["seed"] = fresh_seed()

        apply_ui_params_to_runner(runner, ui)
        job = build_job_from_ui(ui)

        print(f"\\n[UI] Generating {job.animals} - Seed {job.seed} - Steps {runner.cfg.steps}...")

        previews = make_debug_previews(runner, job)
        imgs, _ = runner.run_one(job, log_every=0, show_preview=False)
        score = hidden_score(imgs[0], previews["score_base"])
        demo_metrics = compute_demo_metrics(runner, job, imgs[0], previews, hidden_score_value=score)
        print_demo_metrics(demo_metrics)
        metrics_json = json.dumps(demo_metrics, ensure_ascii=False, indent=2)
        metrics_html = build_demo_metrics_html(demo_metrics)
        save_json(os.path.join(runner._job_out_dir(job), "calculated_demo_metrics.json"), demo_metrics)

        status = (
            f"Generation completed | seed={job.seed} | "
            f"SSIM_bg={demo_metrics['SSIM_bg']:.4f} | "
            f"S_edge={demo_metrics['S_edge']:.4f}. "
            + format_attention_status(previews["attention_info"], runner.cfg.mask_mode)
        )

        return (
            int(job.seed),
            imgs[0],
            previews["preview_bg_original"],
            previews["preview_input"],
            previews["preview_canny"],
            previews["preview_soft"],
            previews["preview_sil"],
            previews["preview_feat"],
            previews["preview_comp"],
            previews["preview_soft_for_mask"],
            previews["preview_soft_hw"],
            previews["preview_mask"],
            previews["preview_contour_agg"],
            previews["preview_region_filling"],
            previews["preview_canny_smoothing"],
            previews["preview_aux_support"],
            previews["preview_soft_gray"],
            previews["preview_soft_smoothing"],
            previews["preview_soft_remap"],
            previews["preview_soft_clip"],
            f"Hidden Score: {score:.4f}",
            status,
            metrics_html,
            metrics_json,
            previews["preview_stage2_report_html"],
            previews["preview_stage2_report_json"],
        )


    def generate_batch(subject_upload=None, bg_upload=None, *args):
        """
        Run the Cartesian product of subjects x backgrounds sequentially.

        Each pair folder contains:
        - subject_input.png
        - background_input.png
        - final_mask.png
        - output.png
        - run_config.json
        - calculated_demo_metrics.json
        """
        ui = build_ui_dict_from_args(args)

        subject_paths = resolve_batch_image_paths(
            subject_upload,
            ui["path_animal"],
            KAGGLE_ANIMALS_DIR,
            label="subject",
        )
        bg_paths = resolve_batch_image_paths(
            bg_upload,
            ui["path_bg"],
            KAGGLE_BG_DIR,
            label="background",
        )

        # One seed is shared by every pair so comparisons across backgrounds are fair.
        batch_seed = fresh_seed() if bool(ui["randomize_seed_each_generate"]) else int(ui["seed"])
        ui["seed"] = int(batch_seed)
        apply_ui_params_to_runner(runner, ui)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        batch_name = f"batch_{timestamp}_{len(subject_paths)}subjects_{len(bg_paths)}bgs"
        batch_root = os.path.join(runner.cfg.out_dir, batch_name)
        ensure_dir(batch_root)

        total_pairs = len(subject_paths) * len(bg_paths)
        gallery_items = []
        manifest_pairs: List[Dict[str, Any]] = []
        failures: List[Dict[str, Any]] = []
        pair_index = 0

        print("\n" + "=" * 80)
        print(f"[BATCH] {len(subject_paths)} subjects x {len(bg_paths)} backgrounds = {total_pairs} pairs")
        print(f"[BATCH] seed={batch_seed} | output={batch_root}")
        print("=" * 80)

        for subject_idx, subject_path in enumerate(subject_paths, start=1):
            subject_stem = slugify(os.path.splitext(os.path.basename(subject_path))[0])

            # For a true multi-subject batch, filenames become prompt labels.
            # For one subject, keep the manually entered Animal Name when provided.
            if len(subject_paths) == 1 and str(ui["animal_name"]).strip():
                subject_label = str(ui["animal_name"]).strip()
            else:
                subject_label = subject_stem.replace("_", " ")

            for bg_idx, bg_path in enumerate(bg_paths, start=1):
                pair_index += 1
                bg_stem = slugify(os.path.splitext(os.path.basename(bg_path))[0])
                pair_name = f"{pair_index:04d}_s{subject_idx:03d}_{subject_stem}__b{bg_idx:03d}_{bg_stem}"
                relative_pair_dir = os.path.join(batch_name, pair_name)

                job = AnimalJob(
                    image_path=subject_path,
                    bg_image_path=bg_path,
                    animals=subject_label,
                    seed=int(batch_seed),
                    prompt_bg=str(ui["prompt_bg"]),
                    subject_template=str(ui["prompt_sub"]),
                    negative=str(ui["negative_prompt"]),
                    name=relative_pair_dir,
                    bg_strength=float(ui["bg_strength"]),
                )

                try:
                    print(f"[BATCH {pair_index}/{total_pairs}] subject={subject_stem} | bg={bg_stem}")
                    previews = make_debug_previews(runner, job)
                    imgs, generated_paths = runner.run_one(job, log_every=0, show_preview=False)
                    output_img = imgs[0]
                    score = hidden_score(output_img, previews["score_base"])
                    metrics = compute_demo_metrics(
                        runner,
                        job,
                        output_img,
                        previews,
                        hidden_score_value=score,
                    )

                    pair_dir = runner._job_out_dir(job)
                    files = save_pair_artifacts(
                        pair_dir=pair_dir,
                        subject_path=subject_path,
                        bg_path=bg_path,
                        output_img=output_img,
                        final_mask=previews["preview_mask"],
                    )
                    metrics_path = os.path.join(pair_dir, "calculated_demo_metrics.json")
                    save_json(metrics_path, metrics)

                    caption = f"{subject_stem} × {bg_stem} | seed {batch_seed}"
                    gallery_items.append((files["output"], caption))
                    manifest_pairs.append({
                        "pair_index": pair_index,
                        "subject_index": subject_idx,
                        "background_index": bg_idx,
                        "subject_name": subject_label,
                        "subject_source": subject_path,
                        "background_source": bg_path,
                        "seed": int(batch_seed),
                        "pair_folder": pair_dir,
                        "files": files,
                        "native_generated_files": generated_paths,
                        "metrics": metrics,
                    })
                except Exception as exc:
                    error_record = {
                        "pair_index": pair_index,
                        "subject_source": subject_path,
                        "background_source": bg_path,
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                    failures.append(error_record)
                    print(f"[BATCH ERROR {pair_index}/{total_pairs}] {error_record['error']}")
                finally:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

        manifest = {
            "batch_name": batch_name,
            "created_at": timestamp,
            "subject_count": len(subject_paths),
            "background_count": len(bg_paths),
            "total_pairs": total_pairs,
            "successful_pairs": len(manifest_pairs),
            "failed_pairs": len(failures),
            "shared_seed": int(batch_seed),
            "subject_paths": subject_paths,
            "background_paths": bg_paths,
            "ui_params": ui,
            "pairs": manifest_pairs,
            "failures": failures,
        }
        manifest_path = os.path.join(batch_root, "batch_manifest.json")
        save_json(manifest_path, manifest)

        # Create ZIP next to the batch directory. The archive contains the whole batch folder.
        zip_path = shutil.make_archive(
            base_name=batch_root,
            format="zip",
            root_dir=runner.cfg.out_dir,
            base_dir=batch_name,
        )

        if not manifest_pairs:
            raise gr.Error(
                f"Batch finished but every pair failed. See logs and manifest: {manifest_path}"
            )

        status = (
            f"Batch completed: {len(manifest_pairs)}/{total_pairs} successful, "
            f"{len(failures)} failed | ZIP: {zip_path}"
        )
        manifest_json = json.dumps(manifest, ensure_ascii=False, indent=2)
        return int(batch_seed), gallery_items, zip_path, status, manifest_json


    def save_current_config(subject_upload=None, bg_upload=None, *args):
        ui = build_ui_dict_from_args(args)
        ui = apply_uploads_to_ui(ui, subject_upload, bg_upload)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filepath = os.path.join(runner.cfg.out_dir, f"ui_config_{timestamp}.json")
        payload = {
            "timestamp": timestamp,
            "version": 5,
            "layout_stage": "preprocess_hints",
            "params": ui,
            "layout_params": {
                "layout_scale": ui["blend_mask_scale"],
                "layout_offset_x": ui["blend_mask_offset_x"],
                "layout_offset_y": ui["blend_mask_offset_y"],
                "layout_rotation_deg": ui["blend_mask_rotation_deg"],
            },
            "attention_compensation": {
                "enabled": ui["auto_attention_compensation"],
                "reference_scale": ui["attention_scale_ref"],
                "reference_gamma": ui["attention_gamma_ref"],
                "reference_ceiling": ui["attention_ceiling_ref"],
                "reference_blur": ui["attention_blur_ref"],
            },
            "legacy_params": [ui[name] for name in UI_FIELD_NAMES],
        }
        save_json(filepath, payload)
        return f"Configuration saved at: {filepath}"

    def load_config_to_ui(config_path: str):
        config_path = str(config_path).strip()
        if not config_path:
            raise gr.Error("Please enter a JSON config path.")
        if not os.path.exists(config_path):
            raise gr.Error(f"File not found: {config_path}")

        with open(config_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        raw_params = payload.get("params", payload) if isinstance(payload, dict) else payload
        missing_keys = []
        if isinstance(raw_params, dict):
            missing_keys = [name for name in UI_FIELD_NAMES if name not in raw_params]
        elif isinstance(raw_params, list):
            missing_keys = UI_FIELD_NAMES[len(raw_params):]

        ui = normalize_ui_params(raw_params)
        auto_filled = [k for k in missing_keys if k in {"auto_attention_compensation", "attention_scale_ref", "attention_gamma_ref", "attention_ceiling_ref", "attention_blur_ref", "blend_mask_scale", "blend_mask_offset_x", "blend_mask_offset_y", "blend_mask_rotation_deg", "randomize_seed_each_generate"}]
        status = f"Loaded config: {config_path}"
        if auto_filled:
            status += " | Auto-filled defaults: " + ", ".join(auto_filled)
        layout_html = render_layout_editor_from_ui(ui)
        return [ui[name] for name in UI_FIELD_NAMES] + [status, layout_html]

    with gr.Blocks(theme=gr.themes.Base()) as demo:
        gr.Markdown("## Camouflage Master Tuning UI — Direct Layout on Canny / SoftEdge")

        with gr.Row():
            with gr.Column(scale=5):
                with gr.Tab("Inputs & Prompts"):
                    gr.Markdown(
                        "Upload multiple subjects and multiple backgrounds. Batch mode runs every subject through every background. "
                        "The first uploaded subject/background is used by Preview and the interactive layout canvas. "
                        "If no files are uploaded, all images from the two folder paths below are used."
                    )
                    subject_upload = gr.File(
                        label="Upload Subject Images (multiple)",
                        file_count="multiple",
                        file_types=["image"],
                        type="filepath",
                    )
                    bg_upload = gr.File(
                        label="Upload Background Images (multiple)",
                        file_count="multiple",
                        file_types=["image"],
                        type="filepath",
                    )

                    path_animal = gr.Textbox(value=UI_DEFAULTS["path_animal"], label="Animal Image Path / Subject Folder")
                    path_bg = gr.Textbox(value=UI_DEFAULTS["path_bg"], label="Background Image Path / Background Folder")
                    animal_name = gr.Textbox(value=UI_DEFAULTS["animal_name"], label="Animal Name")
                    prompt_bg = gr.Textbox(value=UI_DEFAULTS["prompt_bg"], label="Prompt Background", lines=2)
                    prompt_sub = gr.Textbox(value=UI_DEFAULTS["prompt_sub"], label="Prompt Subject Template", lines=2)
                    negative_prompt = gr.Textbox(value=UI_DEFAULTS["negative_prompt"], label="Negative Prompt", lines=2)

                with gr.Tab("Core Generation"):
                    steps = gr.Slider(1, 100, value=UI_DEFAULTS["steps"], step=1, label="Steps")
                    with gr.Row():
                        seed = gr.Number(value=UI_DEFAULTS["seed"], label="Seed", precision=0)
                        randomize_seed_each_generate = gr.Checkbox(value=UI_DEFAULTS["randomize_seed_each_generate"], label="Random Seed Each Generate")
                    bg_strength = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["bg_strength"], step=0.01, label="BG Strength")
                    guidance_bg = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_bg"], step=0.1, label="Guidance BG")
                    guidance_sub = gr.Slider(0.0, 10.0, value=UI_DEFAULTS["guidance_sub"], step=0.1, label="Guidance Sub")
                    sub_init_noise = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["sub_init_noise"], step=0.01, label="Sub Init Noise")
                    couple_mode = gr.Dropdown(
                        choices=["none", "bg_only", "both"],
                        value=UI_DEFAULTS["couple_mode"],
                        label="Couple Mode",
                    )

                with gr.Tab("🎭 Edges & Mask"):
                    with gr.Row():
                        canny_low = gr.Slider(0, 255, value=UI_DEFAULTS["canny_low"], step=1, label="Canny Low")
                        canny_high = gr.Slider(0, 255, value=UI_DEFAULTS["canny_high"], step=1, label="Canny High")
                    canny_blur_ks = gr.Slider(1, 15, value=UI_DEFAULTS["canny_blur_ks"], step=2, label="Canny Blur KS")
                    use_broken_edges = gr.Checkbox(value=UI_DEFAULTS["use_broken_edges"], label="Use Broken Edges")
                    broken_drop_prob = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["broken_drop_prob"], step=0.01, label="Broken Drop Prob")

                    gr.Markdown("---")
                    mask_mode = gr.Dropdown(
                        choices=["silhouette", "outline", "composite", "softedge"],
                        value=UI_DEFAULTS["mask_mode"],
                        label="Mask Mode",
                    )
                    mask_dilate = gr.Slider(1, 31, value=UI_DEFAULTS["mask_dilate"], step=2, label="Mask Dilate")
                    mask_close_ks = gr.Slider(1, 31, value=UI_DEFAULTS["mask_close_ks"], step=2, label="Mask Close KS")
                    mask_blur = gr.Slider(1, 51, value=UI_DEFAULTS["mask_blur"], step=2, label="Mask Blur")
                    mask_ring_k = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_k"], step=2, label="Mask Ring K")
                    mask_ring_blur = gr.Slider(1, 21, value=UI_DEFAULTS["mask_ring_blur"], step=2, label="Mask Ring Blur")
                    mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["mask_gamma"], step=0.1, label="Mask Gamma (old modes)")
                    mask_auto_invert = gr.Checkbox(value=UI_DEFAULTS["mask_auto_invert"], label="Mask Auto Invert")

                    gr.Markdown("#### Composite Mask Params (old modes)")
                    outer_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["outer_weight"], step=0.05, label="Outer Weight")
                    feature_weight = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["feature_weight"], step=0.05, label="Feature Weight")
                    fill_weight = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["fill_weight"], step=0.05, label="Fill Weight")
                    feature_thresh = gr.Slider(0, 255, value=UI_DEFAULTS["feature_thresh"], step=1, label="Feature Thresh")
                    feature_open_ks = gr.Slider(1, 15, value=UI_DEFAULTS["feature_open_ks"], step=2, label="Feature Open KS")
                    feature_blur = gr.Slider(1, 21, value=UI_DEFAULTS["feature_blur"], step=2, label="Feature Blur")
                    fill_blur = gr.Slider(1, 31, value=UI_DEFAULTS["fill_blur"], step=2, label="Fill Blur")

                    gr.Markdown("#### SoftEdge → Kernel Gradient (for subject layout + final mask)")
                    use_soft_grad_mask = gr.Checkbox(value=UI_DEFAULTS["use_soft_grad_mask"], label="Use Kernel Gradient for Layout/Mask")
                    soft_grad_dilation_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_dilation_k"], step=2, label="Soft Grad Dilation K")
                    soft_grad_blur_k = gr.Slider(1, 21, value=UI_DEFAULTS["soft_grad_blur_k"], step=2, label="Soft Grad Blur K")
                    soft_grad_kernel_shape = gr.Dropdown(
                        choices=["ellipse", "rect", "cross"],
                        value=UI_DEFAULTS["soft_grad_kernel_shape"],
                        label="Soft Grad Kernel Shape",
                    )
                    soft_grad_iterations = gr.Slider(1, 5, value=UI_DEFAULTS["soft_grad_iterations"], step=1, label="Soft Grad Iterations")
                    soft_grad_use_skeleton = gr.Checkbox(value=UI_DEFAULTS["soft_grad_use_skeleton"], label="Soft Grad Use Skeleton")
                    soft_grad_skeleton_stage = gr.Dropdown(
                        choices=["pre", "post"],
                        value=UI_DEFAULTS["soft_grad_skeleton_stage"],
                        label="Soft Grad Skeleton Stage",
                    )
                    soft_grad_post_skel_dilate_k = gr.Slider(0, 11, value=UI_DEFAULTS["soft_grad_post_skel_dilate_k"], step=1, label="Post Skeleton Dilate K")

                    gr.Markdown("#### Final Soft Mask Remap")
                    soft_mask_blur = gr.Slider(1, 31, value=UI_DEFAULTS["soft_mask_blur"], step=2, label="Soft Mask Blur")
                    soft_mask_gamma = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["soft_mask_gamma"], step=0.1, label="Soft Mask Gamma")
                    soft_mask_invert = gr.Checkbox(value=UI_DEFAULTS["soft_mask_invert"], label="Soft Mask Invert")
                    soft_mask_floor = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_floor"], step=0.01, label="Soft Mask Floor")
                    soft_mask_ceiling = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["soft_mask_ceiling"], step=0.01, label="Soft Mask Ceiling")

                    gr.Markdown("#### Auto Attention Compensation")
                    gr.Markdown("Enable this option to automatically compensate the soft-mask strength according to the effective silhouette size after dragging or resizing. When the object is smaller than the reference scale, the system reduces gamma/ceiling and slightly reduces blur so the subject remains visible.")
                    auto_attention_compensation = gr.Checkbox(value=UI_DEFAULTS["auto_attention_compensation"], label="Auto Attention Compensation")
                    with gr.Row():
                        attention_scale_ref = gr.Slider(0.05, 0.95, value=UI_DEFAULTS["attention_scale_ref"], step=0.01, label="Reference Scale")
                        btn_capture_scale_ref = gr.Button("Use Current Layout as Ref Scale")
                    with gr.Row():
                        attention_gamma_ref = gr.Slider(0.1, 5.0, value=UI_DEFAULTS["attention_gamma_ref"], step=0.05, label="Reference Gamma")
                        attention_ceiling_ref = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["attention_ceiling_ref"], step=0.01, label="Reference Ceiling")
                    attention_blur_ref = gr.Slider(1, 31, value=UI_DEFAULTS["attention_blur_ref"], step=2, label="Reference Blur")

                with gr.Tab("⏱️ Control & Blending"):
                    canny_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["canny_scale"], step=0.01, label="Control Scale: Canny")
                    soft_scale = gr.Slider(0.0, 2.0, value=UI_DEFAULTS["soft_scale"], step=0.01, label="Control Scale: Soft Edge")
                    start_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["start_frac"], step=0.01, label="Start Frac Control")
                    end_frac = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["end_frac"], step=0.01, label="End Frac Control")
                    gate_kind = gr.Dropdown(choices=["cosine", "linear"], value=UI_DEFAULTS["gate_kind"], label="Control Gate Kind")

                    gr.Markdown("---")
                    blend_profile = gr.Dropdown(choices=["decay", "ramp"], value=UI_DEFAULTS["blend_profile"], label="Blend Profile")
                    blend_start = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_start"], step=0.01, label="Blend Start Frac")
                    blend_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["blend_end"], step=0.01, label="Blend End Frac")
                    alpha_end = gr.Slider(0.0, 1.0, value=UI_DEFAULTS["alpha_end"], step=0.01, label="Blend Alpha End")

                    gr.Markdown("#### Subject Layout Placement (applied from Canny / SoftEdge)")
                    gr.Markdown("Use the drag-and-drop canvas on the right as the main layout control. These fields control the subject position, scale, and rotation directly from the Canny and SoftEdge stage, and are kept for config load/save.")
                    blend_mask_scale = gr.Slider(0.10, 2.50, value=UI_DEFAULTS["blend_mask_scale"], step=0.01, label="Layout Scale", elem_id="blend-mask-scale")
                    blend_mask_offset_x = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_x"], step=0.01, label="Layout Offset X (+ = right)", elem_id="blend-mask-offset-x")
                    blend_mask_offset_y = gr.Slider(-1.00, 1.00, value=UI_DEFAULTS["blend_mask_offset_y"], step=0.01, label="Layout Offset Y (+ = down)", elem_id="blend-mask-offset-y")
                    blend_mask_rotation_deg = gr.Slider(-180.0, 180.0, value=UI_DEFAULTS["blend_mask_rotation_deg"], step=1.0, label="Layout Rotation (deg)", elem_id="blend-mask-rotation-deg")

                with gr.Tab("Config JSON"):
                    config_path = gr.Textbox(label="Config JSON Path", placeholder="/kaggle/working/outputs_camouflage_softgradmask/ui_config_20260314_150633.json")
                    btn_load = gr.Button("Load Config JSON")

                with gr.Row():
                    btn_generate_batch = gr.Button("Run All Subject × BG + ZIP", variant="primary", scale=2)
                    btn_generate = gr.Button("Preview First Pair", scale=1)
                    btn_save = gr.Button("Save Config", scale=1)

            with gr.Column(scale=4):
                gr.Markdown("### Batch Outputs")
                batch_gallery = gr.Gallery(label="All Generated Pairs", columns=3, height=420)
                batch_zip = gr.File(label="Download Complete Batch ZIP")
                batch_manifest_json = gr.Code(label="Batch Manifest (JSON)", language="json", interactive=False)

                gr.Markdown("### First-Pair Preview / Debug")
                out_img = gr.Image(label="Output Image", type="pil")
                out_score = gr.Textbox(label="Hidden Score", interactive=False)
                status_box = gr.Textbox(label="System Status", interactive=False)

                gr.Markdown("### Calculated Output Metrics")
                out_metrics_report = gr.HTML()
                out_metrics_json = gr.Code(label="Calculated Metrics (JSON)", language="json", interactive=False)

                gr.Markdown("### Interactive Subject Layout")
                btn_refresh_layout = gr.Button("Refresh Interactive Canvas")
                layout_editor = gr.HTML(value=safe_initial_layout_html())

                gr.Markdown("### Debug Views")
                with gr.Row():
                    dbg_bg_original = gr.Image(label="Original BG", type="pil")
                    dbg_input = gr.Image(label="Input Image", type="pil")
                with gr.Row():
                    dbg_canny = gr.Image(label="Canny Hint", type="pil")
                    dbg_soft = gr.Image(label="Soft Hint (Raw)", type="pil")
                with gr.Row():
                    dbg_sil = gr.Image(label="Silhouette Mask", type="pil")
                    dbg_feat = gr.Image(label="Feature Mask", type="pil")
                with gr.Row():
                    dbg_comp = gr.Image(label="Composite HW Mask", type="pil")
                    dbg_soft_for_mask = gr.Image(label="Soft For Mask (After Kernel Gradient)", type="pil")
                with gr.Row():
                    dbg_soft_hw = gr.Image(label="SoftEdge HW Mask", type="pil")
                    dbg_latent_mask = gr.Image(label="Final Latent Mask (after preprocess layout)", type="pil")

                gr.Markdown("### Stage 2 Breakdown Views")
                with gr.Row():
                    dbg_contour_agg = gr.Image(label="Contour Aggregation", type="pil")
                    dbg_region_filling = gr.Image(label="Region Filling", type="pil")
                with gr.Row():
                    dbg_canny_smoothing = gr.Image(label="Smoothing (Canny Path)", type="pil")
                    dbg_aux_support = gr.Image(label="Auxiliary Support S", type="pil")
                with gr.Row():
                    dbg_soft_gray = gr.Image(label="Soft Gray", type="pil")
                    dbg_soft_smoothing = gr.Image(label="Smoothing (Soft Path)", type="pil")
                with gr.Row():
                    dbg_soft_remap = gr.Image(label="Nonlinear Remapping", type="pil")
                    dbg_soft_clip = gr.Image(label="Clip to [0,1]", type="pil")

                gr.Markdown("### Stage 2 Values / Diagnostics")
                dbg_stage2_report = gr.HTML()
                dbg_stage2_json = gr.Code(label="Stage 2 Values (JSON)", language="json", interactive=False)

        all_inputs = [
            path_animal,
            path_bg,
            animal_name,
            prompt_bg,
            prompt_sub,
            negative_prompt,
            steps,
            seed,
            randomize_seed_each_generate,
            bg_strength,
            guidance_bg,
            guidance_sub,
            sub_init_noise,
            couple_mode,
            canny_low,
            canny_high,
            canny_blur_ks,
            use_broken_edges,
            broken_drop_prob,
            mask_mode,
            mask_dilate,
            mask_close_ks,
            mask_blur,
            mask_ring_k,
            mask_ring_blur,
            mask_gamma,
            mask_auto_invert,
            outer_weight,
            feature_weight,
            fill_weight,
            feature_thresh,
            feature_open_ks,
            feature_blur,
            fill_blur,
            use_soft_grad_mask,
            soft_grad_dilation_k,
            soft_grad_blur_k,
            soft_grad_kernel_shape,
            soft_grad_iterations,
            soft_grad_use_skeleton,
            soft_grad_skeleton_stage,
            soft_grad_post_skel_dilate_k,
            soft_mask_blur,
            soft_mask_gamma,
            soft_mask_invert,
            soft_mask_floor,
            soft_mask_ceiling,
            canny_scale,
            soft_scale,
            start_frac,
            end_frac,
            gate_kind,
            blend_profile,
            blend_start,
            blend_end,
            alpha_end,
            auto_attention_compensation,
            attention_scale_ref,
            attention_gamma_ref,
            attention_ceiling_ref,
            attention_blur_ref,
            blend_mask_scale,
            blend_mask_offset_x,
            blend_mask_offset_y,
            blend_mask_rotation_deg,
        ]

        btn_generate_batch.click(
            fn=generate_batch,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[seed, batch_gallery, batch_zip, status_box, batch_manifest_json],
        )

        btn_generate.click(
            fn=generate_preview,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[
                seed,
                out_img,
                dbg_bg_original,
                dbg_input,
                dbg_canny,
                dbg_soft,
                dbg_sil,
                dbg_feat,
                dbg_comp,
                dbg_soft_for_mask,
                dbg_soft_hw,
                dbg_latent_mask,
                dbg_contour_agg,
                dbg_region_filling,
                dbg_canny_smoothing,
                dbg_aux_support,
                dbg_soft_gray,
                dbg_soft_smoothing,
                dbg_soft_remap,
                dbg_soft_clip,
                out_score,
                status_box,
                out_metrics_report,
                out_metrics_json,
                dbg_stage2_report,
                dbg_stage2_json,
            ],
        )

        btn_capture_scale_ref.click(
            fn=capture_current_scale_reference,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[attention_scale_ref, status_box],
        )

        btn_save.click(
            fn=save_current_config,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=status_box,
        )

        btn_refresh_layout.click(
            fn=refresh_layout_editor,
            js=JS_SYNC_CANVAS_TO_ARGS,
            inputs=[subject_upload, bg_upload] + all_inputs,
            outputs=[layout_editor, status_box],
        )

        btn_load.click(
            fn=load_config_to_ui,
            inputs=[config_path],
            outputs=all_inputs + [status_box, layout_editor],
        )

    demo.queue().launch(share=True if os.path.exists("/kaggle/working") else True, debug=True)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ec8897dbc14acf3580.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_58/1011399609.py:196: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(rgba, mode="RGBA")



[BATCH] 1 subjects x 1 backgrounds = 1 pairs
[BATCH] seed=1483999164 | output=/kaggle/working/outputs_camouflage_softgradmask/batch_20260724_082843_1subjects_1bgs
[BATCH 1/1] subject=lizard_000d3a9260 | bg=1784881533568_7645371572226054863_7645371572226054863_17ad067543bfa8e9b8308b64b0b8b86a

[BATCH] 1 subjects x 1 backgrounds = 1 pairs
[BATCH] seed=1107092567 | output=/kaggle/working/outputs_camouflage_softgradmask/batch_20260724_082915_1subjects_1bgs
[BATCH 1/1] subject=lizard_000d3a9260 | bg=1784881533568_7645371572226054863_7645371572226054863_17ad067543bfa8e9b8308b64b0b8b86a

[BATCH] 1 subjects x 1 backgrounds = 1 pairs
[BATCH] seed=793451715 | output=/kaggle/working/outputs_camouflage_softgradmask/batch_20260724_082958_1subjects_1bgs
[BATCH 1/1] subject=lizard_000d3a9260 | bg=1784881533568_7645371572226054863_7645371572226054863_17ad067543bfa8e9b8308b64b0b8b86a
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ec8897dbc14acf3580.gra